# 03 nco supervisado attention


## 1. Configuración previa y recursos


In [ ]:
import sys, os, json, shutil, subprocess, importlib.metadata
from pathlib import Path
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('Por instrucción del autor: ejecutar exclusivamente en Google Colab.') from exc
BANK_REFERENCE = {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}
BANK_ID = BANK_REFERENCE['bank_id']
SUPPORTED_BANK_PROTOCOLS = (BANK_REFERENCE['protocol_version'],)
RESUME_RUN_ID = ''
TRAIN_DEVICE = 'auto'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
WORK = Path('/content/TESIS_METHODS_03')
RUNTIME = WORK / 'runtime'
(RUNTIME / 'svrplab').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'tests').mkdir(exist_ok=True)
(RUNTIME / 'svrplab/__init__.py').write_text('"""Métodos y evaluador sellado del banco de tesis."""\n')
os.chdir(RUNTIME)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
needed = ['numpy', 'pandas', 'matplotlib', 'pytest', 'ortools']
if version('torch') is None:
    raise RuntimeError('Se requiere un runtime Colab con PyTorch instalado; no reemplazar CUDA automáticamente.')
needed.append('ortools')
def check_ortools_version():
    import re
    installed = version('ortools')
    if installed is None:
        return
    match = re.match(r'^(\d+)\.(\d+)(?:\.|$)', installed)
    if match is None or not (9, 11) <= tuple(map(int, match.groups())) < (10, 0):
        raise RuntimeError(f'OR-Tools {installed} fuera del rango >=9.11,<10; usar un runtime compatible sin reemplazar bibliotecas automáticamente.')
check_ortools_version()
missing = [name for name in needed if version(name) is None]
if missing:
    constraints = RUNTIME / 'constraints.txt'
    protected = ['numpy','pandas','matplotlib','scipy','protobuf','torch','pillow']
    constraints.write_text(''.join(f'{name}=={version(name)}\n' for name in protected if version(name)))
    requirements = ['gurobipy>=11,<14' if name == 'gurobipy' else ('ortools>=9.11,<10' if name == 'ortools' else name) for name in missing]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), *requirements], check=True)
check_ortools_version()
if version('ortools') is None:
    raise RuntimeError('OR-Tools no está disponible para el profesor de factibilidad.')
print('Python:', sys.version)
print('Dependencias:', {name: version(name) for name in needed})
print('GPU recomendada; la asignación se verificará al importar PyTorch.')


In [ ]:
CONFIG = {'schema_version': 'methods-v2', 'role': 'development_only', 'objective': 'conservative_upper_travel_minutes_v1', 'time_scale': 1000, 'seeds': [11, 29, 47], 'search_seconds': 120.0, 'scenario_split': 'evaluation', 'export_all_scenario_events': True, 'methods': {'aco': {'max_iterations': None, 'ants': 20, 'alpha': 1.0, 'beta': 2.0, 'evaporation': 0.1, 'q0': 0.2, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'tabu': {'max_iterations': None, 'neighborhood_samples': 80, 'tenure': 15, 'initial_restarts': 4, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'gurobi': {'threads': 1, 'mip_gap': 0.0}, 'am_supervised': {'n_samples': 16, 'permute_samples': False, 'completion_search_nodes': 2048}, 'pomo': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_facs': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'edd': {}, 'ortools_routing': {'capacity_scale': 1000}, 'pomo_untrained': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'random_masked': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'hbg_no_pheromone': {'ants': 16, 'iterations': 100, 'alpha': 0.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'hbg_untrained': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}}}
print(json.dumps(CONFIG, indent=2))


In [ ]:
SUPERVISED_CONFIG = {'schema_version': 'supervised-03-v1', 'role': 'development_only', 'replicate_id': 'scientific_repair_secondary_v3', 'split_strategy': 'leave_one_geometry_out_with_inner_validation', 'teacher': {'seeds': [101, 211, 307], 'seconds_per_method': 120.0, 'time_scale': 1000, 'aco': {'max_iterations': None, 'ants': 20, 'alpha': 1.0, 'beta': 2.0, 'evaporation': 0.1, 'q0': 0.2, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'tabu': {'max_iterations': None, 'neighborhood_samples': 80, 'tenure': 15, 'initial_restarts': 20, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'fallback': {'enabled': True, 'seeds': [401, 503, 607], 'seconds_per_attempt': 600.0, 'workers': 1}}, 'model': {'model_dim': 128, 'num_heads': 8, 'num_layers': 3, 'dropout': 0.1, 'tanh_clip': 10.0}, 'training': {'epochs': 30, 'steps_per_epoch': 16, 'gradient_accumulation': 4, 'learning_rate': 0.0001, 'weight_decay': 1e-06, 'gradient_clip_norm': 1.0, 'patience': 5, 'min_delta': 0.0001, 'seconds_per_model': 600.0, 'permute_nodes': True, 'deterministic_algorithms': True}}
print(json.dumps(SUPERVISED_CONFIG, indent=2))


## 2. Drive y aceptación del banco


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/TESIS-EHBG-FACS')
import hashlib
def verify_before_import(folder):
    folder = Path(folder)
    manifest = json.loads((folder / 'CHECKSUMS.json').read_text())
    actual = {str(f.relative_to(folder)) for f in folder.rglob('*')
              if f.is_file() and f.name != 'CHECKSUMS.json' and '__pycache__' not in f.relative_to(folder).parts}
    if actual != set(manifest):
        raise ValueError('Inventario del banco distinto del manifiesto.')
    for relative, expected in manifest.items():
        file = folder / relative
        if not file.resolve().is_relative_to(folder.resolve()):
            raise ValueError('Ruta de manifiesto fuera del banco.')
        if hashlib.sha256(file.read_bytes()).hexdigest() != expected:
            raise ValueError(f'Hash incorrecto: {relative}')
    return manifest
if BANK_ID != BANK_REFERENCE['bank_id']:
    raise ValueError('BANK_ID difiere de la referencia validada; actualizar la referencia común antes de cambiar banco.')
remote_bank = DRIVE_ROOT / 'banks' / BANK_ID
manifest = verify_before_import(remote_bank)
manifest_sha = hashlib.sha256(json.dumps(manifest, sort_keys=True, separators=(',', ':'),
                                        ensure_ascii=False, allow_nan=False).encode('utf-8')).hexdigest()
if manifest_sha != BANK_REFERENCE['manifest_sha256']:
    raise ValueError('El banco no coincide con el manifiesto aceptado en 00; no importar ni ejecutar.')
acceptance = json.loads((remote_bank / 'acceptance.json').read_text())
if (acceptance.get('status') != BANK_REFERENCE['acceptance_status']
        or acceptance.get('protocol_version') != BANK_REFERENCE['protocol_version']):
    raise ValueError('El acta no coincide con la aceptación esperada del banco.')
BANK = WORK / 'banks' / BANK_ID
if not BANK.exists():
    BANK.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(remote_bank, BANK)
if verify_before_import(BANK) != manifest:
    raise ValueError('La copia local no coincide con Drive.')
shutil.copy2(BANK / 'code/svrplab_pilot.py', RUNTIME / 'svrplab/pilot.py')
print('Banco validado en 00, copiado y hashes verificados:', BANK_ID, manifest_sha)


## 3. Implementación legible incluida


In [ ]:
%%writefile svrplab/search.py
"""Contrato de búsqueda determinista; evaluación estocástica en pilot, sin certificados.

El objetivo es el viaje con una cota uniforme de medias por arco. No representa
el costo óptimo del SVRP ni el costo completo de recurso todavía pendiente.
"""
from __future__ import annotations

import math
import numpy as np

from . import pilot as p

OBJECTIVE = 'conservative_upper_travel_minutes_v1'


class SearchProblem:
    def __init__(self, x, spec, time_scale=1000):
        self.x = x
        self.n = x.n
        self.vehicles = len(x.capacities)
        self.capacities = x.capacities
        self.demands = x.demands
        self.open = np.maximum(x.windows[:, 0], x.appear_times)
        self.close = x.windows[:, 1]
        self.horizon = float(x.windows[0, 1])
        self.distances = p.distances(x)
        self.travel = p.nominal_upper_matrix(x, spec, time_scale) / time_scale
        self.time_scale = time_scale

    def route_stats(self, route, vehicle):
        clock = travel = late = 0.0
        node = 0
        starts = []
        for customer in route:
            leg = float(self.travel[node, customer])
            travel += leg
            clock = max(clock + leg, float(self.open[customer]))
            late += max(0.0, clock - float(self.close[customer]))
            starts.append(clock)
            node = customer
        if route:
            leg = float(self.travel[node, 0])
            clock += leg
            travel += leg
        return {'travel_minutes': travel, 'lateness_minutes': late,
                'depot_overtime_minutes': max(0.0, clock - self.horizon),
                'capacity_excess': max(0.0, float(sum(self.demands[j] for j in route)
                                                - self.capacities[vehicle])),
                'service_starts': starts, 'return_minutes': clock}

    def score(self, routes):
        p.validate_routes(self.x, routes)
        stats = [self.route_stats(route, v) for v, route in enumerate(routes)]
        return (sum(r['lateness_minutes'] + r['depot_overtime_minutes'] for r in stats),
                sum(r['travel_minutes'] for r in stats))

    def result(self, routes, status, elapsed, history, **metadata):
        if not math.isfinite(elapsed) or elapsed < 0:
            raise ValueError('Duración de búsqueda inválida.')
        if routes is not None:
            routes = [[int(j) for j in route] for route in routes]
            violation, cost = self.score(routes)
            metadata.update(search_violation_minutes=violation,
                            search_objective_minutes=cost,
                            conservative_feasible=violation <= 1e-9)
        metadata['objective_scope'] = OBJECTIVE
        return {'routes': routes, 'status': status, 'solve_seconds': float(elapsed),
                'history': history, 'metadata': metadata}


In [ ]:
%%writefile svrplab/baselines.py
"""Independent conventional controls and paid construction/local search helpers.

EDD means earliest deadline first with feasible cheapest insertion. It is a
fast control, not a lower bound. OR-Tools Routing uses guided local search on
exactly the shared conservative travel grid. It is a heuristic reference, not
an optimal ceiling. No function reads certificates, scenarios or learned data.
All calls that construct/search routes are to be executed exclusively in Colab.

Primary API: https://developers.google.com/optimization/routing/vrptw and
https://developers.google.com/optimization/routing/routing_options .
"""
from __future__ import annotations

from collections.abc import Mapping
import importlib.metadata
import math
from numbers import Integral, Real
import time

import numpy as np

_TOL = 1e-9
_MAX_INT = 2**60
EDD_DEFAULTS = {'seconds': 120.}
ORTOOLS_DEFAULTS = {'seconds': 120., 'capacity_scale': 1000}


def _config(defaults, config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración debe ser un mapping.')
    unknown = set(config or {}) - set(defaults)
    if unknown:
        raise ValueError(f'Parámetros desconocidos: {sorted(unknown)}')
    result = defaults | dict(config or {})
    seconds = result['seconds']
    if isinstance(seconds, bool) or not isinstance(seconds, Real) or not math.isfinite(seconds) or seconds < 0:
        raise ValueError('seconds debe ser finito y no negativo.')
    result['seconds'] = float(seconds)
    if 'capacity_scale' in result:
        value = result['capacity_scale']
        if isinstance(value, bool) or not isinstance(value, Integral) or value <= 0:
            raise ValueError('capacity_scale debe ser un entero positivo.')
        result['capacity_scale'] = int(value)
    return result


def _seed(seed):
    if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    return int(seed)


def _route_values(problem, route, vehicle):
    stats = problem.route_stats(route, vehicle)
    return (stats['lateness_minutes'] + stats['depot_overtime_minutes'], stats['travel_minutes'])


def _profile(problem, route):
    """Earliest prefix and latest suffix starts for constant-time insertions."""
    nodes = [0] + list(route) + [0]
    earliest = [0.]
    for previous, customer in zip(nodes[:-2], nodes[1:-1]):
        earliest.append(max(earliest[-1] + float(problem.travel[previous, customer]),
                            float(problem.open[customer])))
    latest = [0.] * len(nodes)
    latest[-1] = problem.horizon
    for position in range(len(nodes) - 2, 0, -1):
        customer, following = nodes[position:position + 2]
        latest[position] = min(float(problem.close[customer]),
                               latest[position + 1] - float(problem.travel[customer, following]))
    feasible = all(earliest[i] <= latest[i] + _TOL for i in range(1, len(nodes) - 1))
    return nodes, earliest, latest, feasible


def construct_insertion(problem, rng, deadline, pheromone=None, alpha=0., beta=2., q0=1.):
    """Complete EDD/RCL construction with hard capacity and paid soft repair.

    The deterministic case processes scarce-capacity customers, then deadlines.
    Stochastic construction samples among the three earliest eligible customers
    and feasible insertion positions, guided by travel and (optional) pheromone.
    If no temporal insertion exists, choose minimum total temporal violation;
    do not drop customers, add vehicles, or declare the instance infeasible.
    """
    routes = [[] for _ in range(problem.vehicles)]
    loads = np.zeros(problem.vehicles)
    remaining = set(range(1, problem.n + 1))
    profiles = [_profile(problem, route) for route in routes]
    while remaining:
        if time.perf_counter() >= deadline:
            return None, 'time_limit'
        availability = {j: int(np.count_nonzero(loads + problem.demands[j] <= problem.capacities + _TOL))
                        for j in remaining}
        if min(availability.values()) == 0:
            return None, 'capacity_dead_end'
        scarce = min(availability.values())
        ordered = sorted((j for j in remaining if availability[j] == scarce),
                         key=lambda j: (float(problem.close[j]), float(problem.open[j]), j))
        deterministic = q0 >= 1. or rng.random() < q0
        customer = ordered[0] if deterministic else ordered[int(rng.integers(min(3, len(ordered))))]
        feasible = []
        for vehicle, route in enumerate(routes):
            if time.perf_counter() >= deadline:
                return None, 'time_limit'
            if loads[vehicle] + problem.demands[customer] > problem.capacities[vehicle] + _TOL:
                continue
            nodes, earliest, latest, valid = profiles[vehicle]
            if not valid:
                continue
            for position in range(len(route) + 1):
                previous, following = nodes[position:position + 2]
                start = max(earliest[position] + float(problem.travel[previous, customer]),
                            float(problem.open[customer]))
                if (start > problem.close[customer] + _TOL
                        or start + problem.travel[customer, following] > latest[position + 1] + _TOL):
                    continue
                delta = float(problem.travel[previous, customer] + problem.travel[customer, following]
                              - problem.travel[previous, following])
                trail = (1. if pheromone is None else
                         math.sqrt(max(1e-12, float(pheromone[previous, customer]))
                                   * max(1e-12, float(pheromone[customer, following]))))
                weight = alpha * math.log(trail) - beta * math.log1p(max(0., delta))
                feasible.append((vehicle, position, delta, weight))
        if feasible:
            if deterministic:
                # ACO exploitation uses its pheromone; alpha=0 is cheapest EDD.
                choice = max(feasible, key=lambda row: (row[3], -row[2], -row[0], -row[1]))
            else:
                log_weights = np.array([row[3] for row in feasible])
                weights = np.exp(log_weights - np.max(log_weights))
                choice = feasible[int(rng.choice(len(feasible), p=weights / weights.sum()))]
            vehicle, position = choice[:2]
        else:
            # Preserve a complete diagnostic solution even when construction
            # cannot respect all windows. Full affected-route scores are paid.
            best = None
            for vehicle, route in enumerate(routes):
                if loads[vehicle] + problem.demands[customer] > problem.capacities[vehicle] + _TOL:
                    continue
                old = _route_values(problem, route, vehicle)
                for position in range(len(route) + 1):
                    if time.perf_counter() >= deadline:
                        return None, 'time_limit'
                    candidate = route[:position] + [customer] + route[position:]
                    new = _route_values(problem, candidate, vehicle)
                    row = (new[0] - old[0], new[1] - old[1], vehicle, position)
                    if best is None or row < best:
                        best = row
            if best is None:
                return None, 'capacity_dead_end'
            vehicle, position = best[2:]
        routes[vehicle].insert(position, customer)
        loads[vehicle] += problem.demands[customer]
        profiles[vehicle] = _profile(problem, routes[vehicle])
        remaining.remove(customer)
    return routes, 'complete'


def improve_feasible_routes(problem, routes, rng, deadline, max_moves=100):
    """Paid first-improving relocate repair/descent; retains the input incumbent.

    Return ``(routes, diagnostics)``. Coverage, capacity and fleet remain hard;
    score is lexicographic (temporal violation, conservative travel). An already
    feasible incumbent therefore stays feasible. Infeasible input is permitted
    for repair, but success is not guaranteed. No external routes/solver used.
    """
    if isinstance(max_moves, bool) or not isinstance(max_moves, Integral) or max_moves < 0:
        raise ValueError('max_moves debe ser entero no negativo.')
    current = [list(route) for route in routes]
    incumbent_score = problem.score(current)  # Common coverage/capacity contract before repair.
    stats = [_route_values(problem, route, v) for v, route in enumerate(current)]
    loads = [float(sum(problem.demands[j] for j in route)) for route in current]
    evaluated = accepted = 0
    termination = 'move_limit'
    for _ in range(max_moves):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        positions = [(v, i) for v, route in enumerate(current) for i in range(len(route))]
        # Repair the worst routes first; randomize ties with the method's RNG.
        if rng is not None:
            rng.shuffle(positions)
        positions.sort(key=lambda pair: -stats[pair[0]][0])
        chosen = None
        for origin, offset in positions:
            customer = current[origin][offset]
            reduced = current[origin][:offset] + current[origin][offset + 1:]
            reduced_stats = _route_values(problem, reduced, origin)
            for target, route in enumerate(current):
                if target != origin and loads[target] + problem.demands[customer] > problem.capacities[target] + _TOL:
                    continue
                base = reduced if target == origin else route
                for position in range(len(base) + 1):
                    if time.perf_counter() >= deadline:
                        termination = 'time_limit'
                        break
                    if target == origin and position == offset:
                        continue
                    inserted = base[:position] + [customer] + base[position:]
                    new_stats = _route_values(problem, inserted, target)
                    old_pair = stats[origin]
                    new_pair = new_stats
                    if target != origin:
                        old_pair = tuple(stats[origin][k] + stats[target][k] for k in (0, 1))
                        new_pair = tuple(reduced_stats[k] + new_stats[k] for k in (0, 1))
                    evaluated += 1
                    better = (new_pair[0] < old_pair[0] - _TOL or
                              (abs(new_pair[0] - old_pair[0]) <= _TOL
                               and new_pair[1] < old_pair[1] - _TOL
                               and (old_pair[0] > _TOL or new_pair[0] <= _TOL)))
                    if better:
                        candidate = [list(existing) for existing in current]
                        candidate[origin] = reduced
                        candidate[target] = inserted
                        candidate_score = problem.score(candidate)
                        if (candidate_score >= incumbent_score or
                                (incumbent_score[0] <= _TOL and candidate_score[0] > _TOL)):
                            continue
                        incumbent_score = candidate_score
                        chosen = (origin, target, customer, reduced, reduced_stats, inserted, new_stats)
                        break
                if chosen is not None or termination == 'time_limit':
                    break
            if chosen is not None or termination == 'time_limit':
                break
        if chosen is None:
            if termination != 'time_limit':
                termination = 'relocate_local_optimum'
            break
        origin, target, customer, reduced, reduced_stats, inserted, new_stats = chosen
        if origin != target:
            current[origin], stats[origin] = reduced, reduced_stats
            loads[origin] -= problem.demands[customer]
            loads[target] += problem.demands[customer]
        current[target], stats[target] = inserted, new_stats
        accepted += 1
    return current, {'evaluated_neighbors': evaluated, 'accepted_moves': accepted,
                     'termination': termination, 'operator': 'first_improving_relocate',
                     'objective': 'lexicographic_violation_then_conservative_travel'}


def _finish(problem, routes, started, cfg, seed, method, termination, history=(), **extra):
    score = None if routes is None else problem.score(routes)
    status = 'no_solution' if routes is None else ('feasible' if score[0] <= _TOL else 'conservative_infeasible')
    elapsed = time.perf_counter() - started
    result = problem.result(routes, status, elapsed, list(history), method=method, seed=seed,
                            effective_config=cfg, termination=termination,
                            implementation='independent_conventional_baselines_v2',
                            construction='from_scratch_without_certificates',
                            budget_includes_construction=True, budget_includes_repair=True,
                            surrogate_optimal=False, stochastic_optimal=False,
                            reference_is_bound=False, **extra)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - cfg['seconds'])
    return result


def solve_edd(problem, seed, config=None):
    """Fast deterministic EDD insertion; ends after construction, without LS."""
    started = time.perf_counter()
    cfg, seed = _config(EDD_DEFAULTS, config), _seed(seed)
    routes, reason = construct_insertion(problem, np.random.default_rng(seed), started + cfg['seconds'])
    return _finish(problem, routes, started, cfg, seed, 'edd', reason,
                   seed_affects_search=False, construction_rule='EDD_feasible_cheapest_insertion',
                   budget_policy='upper_limit_fast_control_not_required_to_exhaust')


def _ortools():
    try:
        from ortools.constraint_solver import pywrapcp, routing_enums_pb2
    except ImportError as exc:
        raise RuntimeError('La referencia OR-Tools requiere instalar ortools en Colab.') from exc
    return pywrapcp, routing_enums_pb2


def _integer(value, scale):
    raw = float(value) * scale
    if not math.isfinite(raw) or raw < 0 or raw > _MAX_INT:
        raise ValueError('Recurso fuera del rango entero seguro.')
    rounded = round(raw)
    if not math.isclose(raw, rounded, rel_tol=0., abs_tol=1e-6):
        raise ValueError('El recurso no pertenece a la rejilla declarada; no se redondeará para favorecer al método.')
    return int(rounded)


def solve_ortools_routing(problem, seed, config=None):
    """Fixed-fleet Routing+GLS heuristic reference; no dropped customers."""
    started = time.perf_counter()
    cfg, seed = _config(ORTOOLS_DEFAULTS, config), _seed(seed)
    deadline = started + cfg['seconds']
    if time.perf_counter() >= deadline:
        return _finish(problem, None, started, cfg, seed, 'ortools_routing', 'time_limit')
    pywrapcp, routing_enums = _ortools()
    # The reference pays for the identical conventional feasible seed; its
    # absence never substitutes a certificate or changes the fixed fleet.
    routes, construction_reason = construct_insertion(problem, np.random.default_rng(seed), deadline)
    feasible_seed = routes is not None and problem.score(routes)[0] <= _TOL
    seed_score = None if routes is None else list(problem.score(routes))
    # An infeasible EDD diagnostic is not a Routing solution. Retain its score
    # for auditing, but never return its routes under the OR-Tools method name.
    routes = routes if feasible_seed else None
    origin = 'edd_paid_fallback' if feasible_seed else 'none'

    def finish(termination, **extra):
        return _finish(problem, routes, started, cfg, seed, 'ortools_routing', termination,
                       construction_reason=construction_reason,
                       edd_diagnostic_score=seed_score,
                       conventional_seed_used=bool(feasible_seed),
                       solution_origin=origin,
                       returned_paid_edd_fallback=(origin == 'edd_paid_fallback'),
                       seed_affects_search=False, **extra)

    if time.perf_counter() >= deadline:
        return finish('time_limit')
    scale = problem.time_scale
    if isinstance(scale, bool) or not isinstance(scale, Integral) or scale <= 0:
        raise ValueError('time_scale debe ser entero positivo.')
    matrix = [[_integer(value, scale) for value in row] for row in problem.travel]
    horizon = math.floor(float(problem.horizon) * scale)
    demands = [_integer(value, cfg['capacity_scale']) for value in problem.demands]
    capacities = [_integer(value, cfg['capacity_scale']) for value in problem.capacities]
    windows = [(math.ceil(float(problem.open[j]) * scale), min(horizon, math.floor(float(problem.close[j]) * scale)))
               for j in range(problem.n + 1)]
    if problem.vehicles < 1 or any(lo > hi for lo, hi in windows[1:]):
        return finish('incompatible_integer_model')
    manager = pywrapcp.RoutingIndexManager(problem.n + 1, problem.vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)
    transit = routing.RegisterTransitCallback(lambda a, b: matrix[manager.IndexToNode(a)][manager.IndexToNode(b)])
    routing.SetArcCostEvaluatorOfAllVehicles(transit)
    routing.AddDimension(transit, horizon, horizon, True, 'Time')
    times = routing.GetDimensionOrDie('Time')
    for customer in range(1, problem.n + 1):
        times.CumulVar(manager.NodeToIndex(customer)).SetRange(*windows[customer])
    for vehicle in range(problem.vehicles):
        times.CumulVar(routing.Start(vehicle)).SetRange(0, 0)
        times.CumulVar(routing.End(vehicle)).SetRange(0, horizon)
    demand = routing.RegisterUnaryTransitCallback(lambda a: demands[manager.IndexToNode(a)])
    routing.AddDimensionWithVehicleCapacity(demand, 0, capacities, True, 'Capacity')
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    params.local_search_metaheuristic = routing_enums.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.log_search = False
    # Import/build/seed/extraction are all part of the same end-to-end budget.
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        return finish('time_limit')
    params.time_limit.seconds = int(remaining)
    params.time_limit.nanos = int((remaining - int(remaining)) * 1e9)
    routing.CloseModelWithParameters(params)
    assignment = None
    if feasible_seed:
        assignment = routing.ReadAssignmentFromRoutes(routes, True)
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        return finish('time_limit')
    params.time_limit.seconds = int(remaining)
    params.time_limit.nanos = int((remaining - int(remaining)) * 1e9)
    assignment = (routing.SolveFromAssignmentWithParameters(assignment, params) if assignment is not None
                  else routing.SolveWithParameters(params))
    if assignment is not None:
        candidate = []
        for vehicle in range(problem.vehicles):
            index, route = routing.Start(vehicle), []
            while not routing.IsEnd(index):
                customer = manager.IndexToNode(index)
                if customer:
                    route.append(customer)
                index = assignment.Value(routing.NextVar(index))
            candidate.append(route)
        candidate_score = problem.score(candidate)
        if candidate_score[0] > _TOL:
            raise AssertionError('OR-Tools no satisface el contrato conservador compartido.')
        if routes is None or candidate_score <= problem.score(routes):
            routes = candidate
            origin = 'ortools_routing_assignment'
    termination = ('time_limit' if time.perf_counter() >= deadline else
                   ('routing_no_assignment' if assignment is None else 'routing_search_finished'))
    return finish(termination,
                   solver_status=int(routing.status()), ortools_version=importlib.metadata.version('ortools'),
                   routing_search='parallel_cheapest_insertion_plus_guided_local_search',
                   routing_assignment_found=assignment is not None,
                   capacity_scale=cfg['capacity_scale'], time_scale=int(scale),
                   bound_scope='no_optimality_bound_from_routing_search')


In [ ]:
%%writefile svrplab/metaheuristics.py
"""Transparent classical comparators for the conservative fixed-fleet problem.

Own ACO and Tabu variants, not official reproductions of a named paper. Both pay
for EDD feasible insertion, repair and local search within their declared wall
clock budget. ACO guides feasible insertions with shared directed pheromone and
improves constructed solutions by relocate descent. Tabu uses relocate, swap,
2-opt and inter-route 2-opt-star. Certificates and scenarios are never read.

max_iterations=None means time-limited search; an integer remains available for
controlled reproducibility tests. Every completed incumbent survives timeout;
unsuccessful constructions are recorded, not labelled proof of infeasibility.
Execute construction/search exclusively in Colab.
"""
from __future__ import annotations

import math
import itertools
import time
from collections.abc import Mapping

import numpy as np

from .search import SearchProblem
from .baselines import construct_insertion, improve_feasible_routes


ACO_DEFAULTS = {
    'seconds': 120., 'max_iterations': None, 'ants': 20,
    'alpha': 1., 'beta': 2., 'evaporation': .1, 'q0': .2,
    'local_search_seconds': 1., 'local_search_moves': 20,
}
TABU_DEFAULTS = {
    'seconds': 120., 'max_iterations': None, 'neighborhood_samples': 80,
    'tenure': 15, 'initial_restarts': 4,
    'local_search_seconds': 1., 'local_search_moves': 20,
}
_TOL = 1e-9


def _configuration(defaults, config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración debe ser un mapping.')
    unknown = set(config or {}) - set(defaults)
    if unknown:
        raise ValueError(f'Parámetros desconocidos: {sorted(unknown)}')
    result = defaults | dict(config or {})
    for name, value in result.items():
        if name == 'max_iterations' and value is None:
            continue
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.integer, np.floating)):
            raise ValueError(f'Parámetro numérico inválido: {name}')
        if not math.isfinite(float(value)):
            raise ValueError(f'Parámetro no finito: {name}')
    for name in ('max_iterations', 'ants', 'neighborhood_samples', 'tenure', 'initial_restarts', 'local_search_moves'):
        if name in result and result[name] is not None:
            if int(result[name]) != result[name] or result[name] < 1:
                raise ValueError(f'{name} debe ser entero positivo.')
            result[name] = int(result[name])
    if result['seconds'] < 0 or result['local_search_seconds'] < 0:
        raise ValueError('Los presupuestos deben ser no negativos.')
    for name in ('alpha', 'beta'):
        if name in result and result[name] < 0:
            raise ValueError(f'{name} debe ser no negativo.')
    if 'evaporation' in result and not 0 < result['evaporation'] <= 1:
        raise ValueError('evaporation debe pertenecer a (0, 1].')
    if 'q0' in result and not 0 <= result['q0'] <= 1:
        raise ValueError('q0 debe pertenecer a [0, 1].')
    return {k: (v if v is None or isinstance(v, int) else float(v)) for k, v in result.items()}


def _seed_value(seed):
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    return int(seed)


def _capacity_screen(problem):
    """Condiciones necesarias; pasarlas no demuestra que exista partición."""
    if problem.vehicles == 0:
        return problem.n == 0
    return (float(np.sum(problem.demands[1:])) <= float(np.sum(problem.capacities)) + _TOL
            and (problem.n == 0 or float(np.max(problem.demands[1:]))
                 <= float(np.max(problem.capacities)) + _TOL))


def _history_row(started, iteration, score, phase, **extra):
    return {'iteration': int(iteration), 'elapsed_seconds': time.perf_counter() - started,
            'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
            'phase': phase, **extra}


def _finish(problem, routes, score, started, config, seed, history, method,
            termination, iterations, **extra):
    elapsed = time.perf_counter() - started
    status = ('no_solution' if routes is None else
              'feasible' if score[0] <= _TOL else 'conservative_infeasible')
    result = problem.result(
        routes, status, elapsed, history, method=method, seed=seed,
        effective_config=config, termination=termination, iterations=int(iterations),
        implementation='project_own_insertion_local_search_v2',
        construction='from_scratch_without_certificates',
        search_objective='lexicographic_temporal_violation_then_conservative_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        budget_includes_construction=True, budget_includes_repair=True,
        initialization='paid_edd_feasible_cheapest_insertion',
        budget_overrun_seconds=max(0., elapsed - config['seconds']),
        time_limit_granularity='construction_step_or_neighborhood_candidate',
        seed_reproducibility='same_environment_and_iteration_cutoff; wall_time_cutoff_may_differ',
        **extra)
    # Incluir también la validación final del contrato común en el tiempo medido.
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - config['seconds'])
    return result


def _construct(problem, rng, deadline, pheromone, alpha, beta, q0):
    return construct_insertion(problem, rng, deadline, pheromone, alpha, beta, q0)


def _iterations(limit):
    return itertools.count(1) if limit is None else range(1, limit + 1)


def _improve(problem, routes, rng, deadline, config):
    """Per-construction local search remains inside the shared search deadline."""
    if time.perf_counter() >= deadline or config['local_search_seconds'] <= 0:
        return routes, {'evaluated_neighbors': 0, 'accepted_moves': 0}
    local_deadline = min(deadline, time.perf_counter() + config['local_search_seconds'])
    return improve_feasible_routes(problem, routes, rng, local_deadline,
                                   max_moves=config['local_search_moves'])


def _arcs(routes):
    """Atributos dirigidos con vehículo, sin arco artificial 0→0 para vacíos."""
    return {(v, int(a), int(b)) for v, route in enumerate(routes) if route
            for a, b in zip([0] + list(route), list(route) + [0])}


def _reinforce(pheromone, routes, weight):
    for route in routes:
        if route:
            for a, b in zip([0] + list(route), list(route) + [0]):
                pheromone[a, b] += weight


def solve_aco(problem: SearchProblem, seed, config=None):
    """EDD seed + pheromone-guided feasible insertion + paid relocate descent.

    Rank deposits: global best 1 and iteration best .5 after evaporation.
    The method remains a declared project variant, not a reproduction claim.
    """
    started = time.perf_counter()
    cfg = _configuration(ACO_DEFAULTS, config)
    seed = _seed_value(seed)
    deadline = started + cfg['seconds']
    rng = np.random.default_rng(seed)
    best, best_score, history = None, None, []
    completed_iterations = constructed = capacity_dead_ends = 0
    local_evaluated = local_moves = 0
    termination = 'iteration_limit'
    if not _capacity_screen(problem):
        return _finish(problem, best, best_score, started, cfg, seed, history, 'aco',
                       'capacity_screen_failed', 0, constructed_solutions=0)
    pheromone = np.ones((problem.n + 1, problem.n + 1))
    if time.perf_counter() < deadline:
        routes, reason = _construct(problem, rng, deadline, pheromone, 0., cfg['beta'], 1.)
        if routes is not None:
            routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
            local_evaluated += diagnostics['evaluated_neighbors']
            local_moves += diagnostics['accepted_moves']
            best, best_score = routes, problem.score(routes)
            constructed += 1
            history.append(_history_row(started, 0, best_score, 'paid_edd_initialization'))
        elif reason == 'capacity_dead_end':
            capacity_dead_ends += 1
    for iteration in _iterations(cfg['max_iterations']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        iteration_best, iteration_score = None, None
        for _ in range(cfg['ants']):
            if time.perf_counter() >= deadline:
                termination = 'time_limit'
                break
            routes, reason = _construct(problem, rng, deadline, pheromone,
                                        cfg['alpha'], cfg['beta'], cfg['q0'])
            if routes is None:
                if reason == 'time_limit':
                    termination = 'time_limit'
                    break
                capacity_dead_ends += 1
                continue
            routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
            local_evaluated += diagnostics['evaluated_neighbors']
            local_moves += diagnostics['accepted_moves']
            score = problem.score(routes)
            constructed += 1
            if iteration_score is None or score < iteration_score:
                iteration_best, iteration_score = routes, score
            if best_score is None or score < best_score:
                best, best_score = routes, score
                history.append(_history_row(started, iteration, best_score, 'incumbent',
                                            constructed_solutions=constructed))
        if termination == 'time_limit':
            break
        pheromone *= 1. - cfg['evaporation']
        if iteration_best is not None:
            _reinforce(pheromone, iteration_best, .5)
        if best is not None:
            _reinforce(pheromone, best, 1.)
        np.clip(pheromone, 1e-6, 1e6, out=pheromone)
        completed_iterations = iteration
        if best_score is not None:
            history.append(_history_row(started, iteration, best_score, 'iteration',
                                        constructed_solutions=constructed))
    if time.perf_counter() >= deadline:
        termination = 'time_limit'
    return _finish(problem, best, best_score, started, cfg, seed, history, 'aco',
                   termination, completed_iterations, constructed_solutions=constructed,
                   capacity_dead_ends=capacity_dead_ends,
                   local_search_evaluated_neighbors=local_evaluated, local_search_accepted_moves=local_moves,
                   construction_rule='EDD_RCL_feasible_insertion_with_pheromone',
                   pheromone_rule='global_best_1_plus_iteration_best_0.5_after_evaporation')


def _apply_move(routes, kind, first, second):
    """Transformación pura; posiciones de relocate se refieren tras extracción."""
    candidate = [list(route) for route in routes]
    a, i = first
    b, j = second
    if kind == 'relocate':
        customer = candidate[a].pop(i)
        candidate[b].insert(j, customer)
    elif kind == 'swap':
        candidate[a][i], candidate[b][j] = candidate[b][j], candidate[a][i]
    elif kind == '2opt_star':
        if a == b:
            raise ValueError('2-opt-star requiere rutas distintas.')
        candidate[a], candidate[b] = candidate[a][:i] + candidate[b][j:], candidate[b][:j] + candidate[a][i:]
    elif kind == '2opt':
        if a != b:
            raise ValueError('2-opt requiere una sola ruta.')
        lo, hi = sorted((i, j))
        candidate[a][lo:hi + 1] = reversed(candidate[a][lo:hi + 1])
    else:
        raise ValueError(f'Movimiento desconocido: {kind}')
    return candidate


def _random_neighbor(routes, rng):
    positions = [(v, i) for v, route in enumerate(routes) for i in range(len(route))]
    if not positions:
        return None
    kind = ('relocate', 'swap', '2opt', '2opt_star')[int(rng.integers(4))]
    if kind == 'relocate':
        first = positions[int(rng.integers(len(positions)))]
        target = int(rng.integers(len(routes)))
        target_length = len(routes[target]) - int(target == first[0])
        second = (target, int(rng.integers(target_length + 1)))
    elif kind == '2opt_star':
        if len(routes) < 2:
            return None
        a, b = map(int, rng.choice(len(routes), size=2, replace=False))
        first = (a, int(rng.integers(len(routes[a]) + 1)))
        second = (b, int(rng.integers(len(routes[b]) + 1)))
    elif kind == 'swap':
        if len(positions) < 2:
            return None
        left, right = rng.choice(len(positions), size=2, replace=False)
        first, second = positions[int(left)], positions[int(right)]
    else:
        eligible = [v for v, route in enumerate(routes) if len(route) >= 2]
        if not eligible:
            return None
        vehicle = int(rng.choice(eligible))
        left, right = rng.choice(len(routes[vehicle]), size=2, replace=False)
        first, second = (vehicle, int(left)), (vehicle, int(right))
    candidate = _apply_move(routes, kind, first, second)
    return None if candidate == routes else (candidate, kind)


def _tabu_admissible(added_arcs, tabu_until, iteration, score, best_score):
    # Aspiración estricta: únicamente superar al mejor global lexicográfico.
    return score < best_score or not any(iteration < tabu_until.get(arc, 0) for arc in added_arcs)


def _capacity_valid(problem, routes):
    return all(sum(float(problem.demands[j]) for j in route) <= problem.capacities[v] + _TOL
               for v, route in enumerate(routes))


def solve_tabu(problem: SearchProblem, seed, config=None):
    """Búsqueda tabú con multiaranque propio y vecinos de capacidad válida.

    Acepta deterioro y violaciones temporales para explorar. Un arco eliminado
    queda prohibido para reinserción durante las próximas ``tenure`` iteraciones;
    así la reversión de relocate, swap y 2-opt es tabú, salvo aspiración. Si el
    muestreo no encuentra movimiento admisible se avanza la iteración (expira la
    memoria) y se vuelve a muestrear, sin declarar óptimo local ni infactibilidad.
    """
    started = time.perf_counter()
    cfg = _configuration(TABU_DEFAULTS, config)
    seed = _seed_value(seed)
    deadline = started + cfg['seconds']
    rng = np.random.default_rng(seed)
    best, best_score, history = None, None, []
    restarts = capacity_dead_ends = iterations = accepted = evaluated = 0
    local_evaluated = local_moves = 0
    termination = 'iteration_limit'
    if not _capacity_screen(problem):
        return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                       'capacity_screen_failed', 0, initial_attempts=0)
    # Construcción determinista inicial, seguida de intentos estocásticos;
    # todos se cobran al mismo reloj y ninguno usa una solución de otro método.
    neutral_pheromone = np.ones((problem.n + 1, problem.n + 1))
    for restart in range(cfg['initial_restarts']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        restarts += 1
        routes, reason = _construct(problem, rng, deadline, neutral_pheromone,
                                    alpha=0., beta=2., q0=1. if restart == 0 else .5)
        if routes is None:
            if reason == 'time_limit':
                termination = 'time_limit'
                break
            capacity_dead_ends += 1
            continue
        routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
        local_evaluated += diagnostics['evaluated_neighbors']
        local_moves += diagnostics['accepted_moves']
        score = problem.score(routes)
        if best_score is None or score < best_score:
            best, best_score = routes, score
            history.append(_history_row(started, 0, best_score, 'initialization', restart=restart))
    if best is None:
        if termination != 'time_limit':
            termination = 'construction_failed'
        return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                       termination, 0, initial_attempts=restarts, capacity_dead_ends=capacity_dead_ends)
    current = [list(route) for route in best]
    current_score = best_score
    tabu_until = {}
    for iteration in _iterations(cfg['max_iterations']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        current_arcs = _arcs(current)
        chosen = chosen_score = chosen_removed = chosen_kind = None
        seen = set()
        for _ in range(cfg['neighborhood_samples']):
            if time.perf_counter() >= deadline:
                termination = 'time_limit'
                break
            proposed = _random_neighbor(current, rng)
            if proposed is None:
                continue
            candidate, kind = proposed
            key = tuple(tuple(route) for route in candidate)
            if key in seen:
                continue
            seen.add(key)
            if not _capacity_valid(problem, candidate):
                continue
            score = problem.score(candidate)
            evaluated += 1
            candidate_arcs = _arcs(candidate)
            added = candidate_arcs - current_arcs
            if not _tabu_admissible(added, tabu_until, iteration, score, best_score):
                continue
            if chosen_score is None or score < chosen_score:
                chosen, chosen_score = candidate, score
                chosen_removed, chosen_kind = current_arcs - candidate_arcs, kind
        if chosen is not None:
            current, current_score = chosen, chosen_score
            accepted += 1
            for arc in chosen_removed:
                tabu_until[arc] = iteration + cfg['tenure'] + 1
            if current_score < best_score:
                best, best_score = [list(route) for route in current], current_score
        tabu_until = {arc: expiry for arc, expiry in tabu_until.items() if expiry > iteration}
        iterations = iteration
        history.append(_history_row(started, iteration, best_score, 'iteration',
                                    current_violation_minutes=float(current_score[0]),
                                    current_travel_minutes=float(current_score[1]),
                                    accepted_move=chosen_kind, evaluated_neighbors=evaluated))
        if termination == 'time_limit':
            break
    if time.perf_counter() >= deadline:
        termination = 'time_limit'
    return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                   termination, iterations, initial_attempts=restarts,
                   capacity_dead_ends=capacity_dead_ends, evaluated_neighbors=evaluated,
                   accepted_moves=accepted, local_search_evaluated_neighbors=local_evaluated,
                   local_search_accepted_moves=local_moves, tabu_rule='removed_directed_vehicle_arcs_forbid_readdition',
                   aspiration='strict_global_best_lexicographic_improvement')


In [ ]:
%%writefile svrplab/experiments.py
"""Ejecución Colab, evaluación única y checkpoints inmutables para métodos 01–06.

No modifica el banco ni entrega certificados a los algoritmos. Cada resultado
se reevalúa desde sus rutas persistidas con el pilot.py guardado en el banco.
"""
from __future__ import annotations

import csv
import json
import math
import shutil
import time
import uuid
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from . import pilot as p
from .search import OBJECTIVE, SearchProblem

SCHEMA = 'methods-v2'
SUMMARY_METRICS = [
    'scenarios', 'alpha', 'mean_travel_minutes', 'cvar_travel_minutes',
    'cost_variance_population', 'std_travel_minutes', 'scenario_strict_feasibility',
    'mean_customer_violation_fraction', 'mean_lateness_minutes',
    'mean_depot_overtime_minutes', 'mean_waiting_minutes', 'mean_base_minutes',
    'mean_net_execution_delta_minutes', 'mean_late_customers', 'mean_recourse_reorders',
    'used_vehicles', 'evaluation_seconds', 'trace_validation_seconds',
]


def write_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(fields or dict.fromkeys(k for row in rows for k in row))
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def native(value):
    """Normaliza escalares del evaluador sellado sin modificar su implementación."""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return [native(item) for item in value.tolist()]
    if isinstance(value, dict):
        return {key: native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(item) for item in value]
    return value


def write_json(path, value):
    p.write_json(path, native(value))


def read_json(path):
    return json.loads(Path(path).read_text())


def validate_scenarios(scenarios, config):
    if set(scenarios) != set(config['scenarios']):
        raise ValueError('Splits del banco incompatibles.')
    seen = set()
    for split, count in config['scenarios'].items():
        seeds = scenarios[split]
        if (len(seeds) != count or len(set(seeds)) != count
                or any(isinstance(s, bool) or not isinstance(s, int) or s < 0 for s in seeds)
                or seen.intersection(seeds)):
            raise ValueError('Escenarios repetidos, solapados o con tamaño/tipo incorrecto.')
        seen.update(seeds)


def validate_config(config):
    if (config['schema_version'] != SCHEMA or config['role'] != 'development_only'
            or config['objective'] != OBJECTIVE or config['scenario_split'] != 'evaluation'
            or config['export_all_scenario_events'] is not True):
        raise ValueError('Contrato experimental incompatible.')
    if not math.isfinite(config['search_seconds']) or config['search_seconds'] <= 0:
        raise ValueError('Presupuesto debe ser positivo y finito.')
    seeds = config['seeds']
    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s < 0 for s in seeds):
        raise ValueError('Semillas de métodos inválidas.')
    if type(config['time_scale']) is not int or config['time_scale'] < 1:
        raise ValueError('Escala de tiempo inválida.')


def route_rows(x, routes, context):
    rows = []
    for vehicle, route in enumerate(routes):
        load = 0.0
        for position, node in enumerate([0, *route, 0]):
            load += float(x.demands[node])
            rows.append({**context, 'vehicle': vehicle, 'position': position,
                         'node': node, 'empty_vehicle': not bool(route),
                         'x': float(x.locations[node, 0]), 'y': float(x.locations[node, 1]),
                         'demand': float(x.demands[node]), 'cumulative_load': load,
                         'capacity': float(x.capacities[vehicle]),
                         'window_open': float(x.windows[node, 0]),
                         'window_close': float(x.windows[node, 1]),
                         'appear_time': float(x.appear_times[node])})
    return rows


def validate_trace(x, routes, trace, raw):
    for key, value in raw.items():
        actual = trace[key]
        if isinstance(value, float):
            if not math.isclose(actual, value, rel_tol=1e-12, abs_tol=1e-9):
                raise AssertionError(f'Traza distinta del evaluador: {key}')
        elif actual != value:
            raise AssertionError(f'Traza distinta del evaluador: {key}')
    p.validate_routes(x, trace['realized_routes'])
    if any(sorted(a) != sorted(b) for a, b in zip(routes, trace['realized_routes'])):
        raise AssertionError('El recurso cambió la asignación de vehículo.')
    events = trace['events']
    expected = x.n + sum(bool(r) for r in routes)
    if len(events) != expected:
        raise AssertionError('Eventos incompletos.')
    if any(type(e['vehicle']) is not int or not 0 <= e['vehicle'] < len(routes) for e in events):
        raise AssertionError('Vehículo inválido en eventos.')
    for vehicle, realized in enumerate(trace['realized_routes']):
        vehicle_events = [e for e in events if e['vehicle'] == vehicle]
        path = [0, *realized, 0] if realized else [0]
        if [(e['from'], e['to']) for e in vehicle_events] != list(zip(path, path[1:])):
            raise AssertionError('Arcos distintos de la ruta realizada.')
        previous = 0.0
        for event in vehicle_events:
            node = event['to']
            start = max(event['arrival'], x.windows[node, 0], x.appear_times[node]) if node else event['arrival']
            expected_late = max(0.0, start - x.windows[node, 1])
            if (not all(math.isfinite(event[k]) for k in ('departure', 'arrival', 'service_start', 'late'))
                    or not math.isclose(event['departure'], previous, abs_tol=1e-9)
                    or event['arrival'] < event['departure'] - 1e-9
                    or not math.isclose(event['service_start'], start, abs_tol=1e-9)
                    or not math.isclose(event['late'], expected_late, abs_tol=1e-9)):
                raise AssertionError('Horario de eventos incoherente.')
            previous = event['service_start']
    travel = sum(e['arrival'] - e['departure'] for e in events)
    waiting = sum(e['service_start'] - e['arrival'] for e in events)
    late = sum(e['late'] for e in events if e['to'] != 0)
    overtime = sum(e['late'] for e in events if e['to'] == 0)
    for value, key in [(travel, 'travel_minutes'), (waiting, 'waiting_minutes'),
                       (late, 'lateness_minutes'), (overtime, 'depot_overtime_minutes')]:
        if not math.isclose(value, raw[key], rel_tol=1e-10, abs_tol=1e-7):
            raise AssertionError(f'Eventos no reconcilian {key}.')


def evaluate_solution(x, solution, seeds, spec, policy, alpha, directory, context):
    """Nunca selecciona/tunea con evaluation; puntúa una solución ya congelada."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / 'environment.json', p.environment_report())
    routes = solution['routes']
    before = x.sha
    p.validate_routes(x, routes)
    summary, records = p.evaluate(x, routes, seeds, spec, policy, alpha)
    if [r['scenario_seed'] for r in records] != seeds:
        raise AssertionError('El evaluador cambió el conjunto u orden de escenarios.')
    trace_started = time.perf_counter()
    event_count = route_count = 0
    # Streaming: no acumular millones de eventos en RAM para n=300.
    with (directory / 'events.csv').open('w', newline='', encoding='utf-8') as ef, \
            (directory / 'realized_routes.csv').open('w', newline='', encoding='utf-8') as rf:
        ew = rw = None
        for record in records:
            seed = record['scenario_seed']
            trace = p.simulate(x, routes, seed, spec, policy, trace=True)
            validate_trace(x, routes, trace, record)
            trace_context = {**context, 'policy': policy, 'scenario_seed': str(seed)}
            for event_index, event in enumerate(trace['events']):
                row = {**trace_context, 'event_index': event_index, **event}
                if ew is None:
                    ew = csv.DictWriter(ef, fieldnames=list(row)); ew.writeheader()
                ew.writerow(row)
                event_count += 1
            for row in route_rows(x, trace['realized_routes'], trace_context):
                if rw is None:
                    rw = csv.DictWriter(rf, fieldnames=list(row)); rw.writeheader()
                rw.writerow(row)
                route_count += 1
    summary['trace_validation_seconds'] = time.perf_counter() - trace_started
    costs = [r['travel_minutes'] for r in records]
    for source in ['waiting_minutes', 'base_minutes', 'net_execution_delta_minutes',
                   'late_customers', 'recourse_reorders']:
        summary['mean_' + source] = float(np.mean([r[source] for r in records]))
    summary['std_travel_minutes'] = float(np.std(costs, ddof=0))
    # CVaR por integración de masa en la cola: control independiente de la fórmula de pilot.
    descending = sorted(costs, reverse=True)
    mass = len(costs) * (1 - alpha)
    whole = min(len(costs), int(math.floor(mass)))
    tail_sum = sum(descending[:whole])
    if whole < len(costs):
        tail_sum += (mass - whole) * descending[whole]
    if not math.isclose(tail_sum / mass, summary['cvar_travel_minutes'], rel_tol=1e-10):
        raise AssertionError('CVaR no coincide con integración de la cola.')
    if x.sha != before:
        raise AssertionError('La instancia cambió durante evaluación.')
    raw_rows = [{**context, 'policy': policy, **r, 'scenario_seed': str(r['scenario_seed'])}
                for r in records]
    write_csv(directory / 'scenario_metrics.csv', raw_rows)
    write_json(directory / 'summary.json', {**context, **summary})
    qc = {'status': 'passed', 'structural_valid': True, 'scenarios': len(records),
          'event_rows': event_count, 'realized_route_rows': route_count,
          'expected_event_rows': len(seeds) * (x.n + solution['nominal']['used_vehicles']),
          'expected_realized_route_rows': len(seeds) * (x.n + 2 * len(x.capacities)),
          'instance_unchanged': x.sha == before, 'trace_matches_evaluator': True,
          'recourse_preserves_vehicle_assignment': True, 'cvar_independently_checked': True,
          'nominal_feasible': solution['nominal']['strict_feasible'],
          'stochastic_feasibility_is_a_measurement': True}
    if qc['event_rows'] != qc['expected_event_rows'] or qc['realized_route_rows'] != qc['expected_realized_route_rows']:
        raise AssertionError('Número de filas de exportación incorrecto.')
    # Releer CSVs comprueba la persistencia, las semillas de 64 bits y el número de filas.
    for name, expected_rows in [('events.csv', event_count), ('realized_routes.csv', route_count),
                                ('scenario_metrics.csv', len(seeds))]:
        with (directory / name).open(newline='', encoding='utf-8') as stream:
            rows = csv.DictReader(stream)
            count = 0
            observed = set()
            for row in rows:
                observed.add(int(row['scenario_seed'])); count += 1
            if count != expected_rows or observed != set(seeds):
                raise AssertionError(f'Exportación incompleta: {name}')
    write_json(directory / 'validation.json', qc)
    return {**context, **summary}


def save_plot(x, routes, path, title):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 6))
    for vehicle, route in enumerate(routes):
        if not route:
            continue
        coords = x.locations[[0, *route, 0]]
        ax.plot(coords[:, 0], coords[:, 1], '-o', markersize=2, linewidth=.8,
                label=f'V{vehicle}')
    ax.scatter(*x.locations[0], marker='*', s=160, color='black', zorder=5)
    ax.set(title=title, xlabel='x (unidad fuente)', ylabel='y (unidad fuente)', aspect='equal')
    if sum(bool(r) for r in routes) <= 15:
        ax.legend(fontsize=6, ncol=3)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def _commit(directory, remote):
    p.seal_bundle(directory)
    _sync(directory, remote)


def _sync(local, remote):
    """Conservar staging interrumpido; promoverlo sólo si su hash es idéntico."""
    staging = remote.with_name(remote.name + '.partial')
    if not remote.exists() and staging.exists():
        try:
            complete = p.verify_bundle(staging) == p.verify_bundle(local)
        except (ValueError, OSError, json.JSONDecodeError):
            complete = False
        if complete:
            staging.rename(remote)
        else:
            staging.rename(staging.with_name(staging.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    p.sync_verified(local, remote)


def _recover(local, remote):
    if (local / 'CHECKSUMS.json').exists():
        p.verify_bundle(local)
        _sync(local, remote)
        return True
    if remote.exists():
        # Conservar la escritura local incompleta, sin sobrescribirla ni mezclarla.
        if local.exists():
            local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
        _sync(remote, local)
        return True
    if local.exists():
        local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    return False


def comparison_protocol(bank_hash, evaluator_hash, bank_config, config):
    """Mismo problema/evaluación/presupuesto; algoritmos y entrenamiento pueden diferir."""
    return {'schema_version': SCHEMA, 'bank_manifest_sha256': bank_hash,
            'evaluator_sha256': evaluator_hash,
            'runner_sha256': p.file_hash(__file__),
            'search_contract_sha256': p.file_hash(Path(__file__).with_name('search.py')),
            'objective': config['objective'], 'time_scale': config['time_scale'],
            'role': config['role'], 'solver_seeds': config['seeds'],
            'search_seconds': config['search_seconds'], 'scenario_split': config['scenario_split'],
            'policies': bank_config['policies'], 'alpha': bank_config['alpha_cvar'],
            'export_all_scenario_events': config['export_all_scenario_events']}


def provenance_columns(provenance):
    fields = ['training_id', 'model_checkpoint_sha256', 'fold_id', 'training_seed',
              'train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups',
              'training_seconds', 'validation_seconds', 'teacher_seconds', 'offline_cost_id',
              'offline_manifest_sha256', 'checkpoint_selection', 'training_scope']
    result = {key: provenance.get(key) for key in fields}
    for key in ['training_seconds', 'validation_seconds', 'teacher_seconds']:
        result[key] = provenance.get(key, 0.0)
        if isinstance(result[key], bool) or not isinstance(result[key], (float, int)) or not math.isfinite(result[key]) or result[key] < 0:
            raise ValueError(f'Costo offline inválido: {key}')
    for key, value in result.items():
        if isinstance(value, (dict, list, tuple)):
            result[key] = json.dumps(value, sort_keys=True, separators=(',', ':'))
    return result


def validate_learned_provenance(provenance, x, seed):
    groups = [provenance.get(role) for role in
              ['train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups']]
    if any(not isinstance(group, list) or not group for group in groups):
        raise ValueError('Faltan grupos de entrenamiento/validación/heldout del modelo.')
    train, validation, heldout = map(set, groups)
    if (train & validation or train & heldout or validation & heldout
            or x.provenance['geometry_group_sha256'] not in heldout
            or provenance.get('training_seed') != seed):
        raise ValueError('Procedencia del modelo tiene fuga de geometría o semilla distinta.')
    checkpoint = provenance.get('model_checkpoint_sha256', '')
    if not provenance.get('training_id') or len(checkpoint) != 64 or any(c not in '0123456789abcdef' for c in checkpoint):
        raise ValueError('Falta identidad/hash de checkpoint del modelo aprendido.')


def run_methods(bank, config, solvers, work_root, drive_root, run_id=None, validation_artifacts=(),
                method_availability=None, method_provenance=None):
    """solvers: {method: callable(problem, seed, config, output_dir)}.

    Rerun con run_id explícito recupera checkpoints; sin ID crea otra réplica
    trazable. La comparación usa comparison_id, sin confundir réplicas con N.
    """
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Experimentación exclusivamente en Google Colab.') from exc
    validate_config(config)
    bank = Path(bank)
    instances = sorted(p.load_bank(bank), key=lambda x: x.n)
    bank_config = read_json(bank / 'config.json')
    bank_hash = p.verify_bundle(bank)
    spec = p.Traffic(**bank_config['traffic'])
    nominal_spec = p.Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
    scenarios = {x.n: read_json(bank / f'scenarios/n{x.n:04d}.json') for x in instances}
    for value in scenarios.values():
        validate_scenarios(value, bank_config)
    if not solvers or not set(solvers).issubset(config['methods']):
        raise ValueError('Métodos solicitados incompatibles con configuración.')
    method_availability = method_availability or {name: 'enabled' for name in solvers}
    if set(method_availability) != set(solvers) or not set(method_availability.values()).issubset({'enabled', 'license_pending'}):
        raise ValueError('Disponibilidad de métodos inválida.')
    package = Path(__file__).parent
    source_hashes = {f.name: p.file_hash(f) for f in sorted(package.glob('*.py'))}
    method_provenance = method_provenance or {}
    protocol = comparison_protocol(bank_hash, p.file_hash(p.__file__), bank_config, config)
    comparison_id = p.digest(protocol)
    signature = p.digest({'protocol': protocol, 'config': config,
                          'methods': sorted(solvers), 'source_hashes': source_hashes,
                          'method_availability': method_availability, 'method_provenance': method_provenance})
    run_id = run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_') + uuid.uuid4().hex[:8]
    if Path(run_id).name != run_id or run_id in ('.', '..'):
        raise ValueError('run_id debe ser un nombre de carpeta simple.')
    root = Path(work_root) / 'method_runs' / bank.name / run_id
    remote = Path(drive_root) / 'method_runs' / bank.name / run_id
    print(f'RUN_ID = {run_id}\nDrive: {remote}', flush=True)
    if _recover(root / 'meta', remote / 'meta'):
        if read_json(root / 'meta/plan.json')['signature'] != signature:
            raise ValueError('No se puede reanudar con otro banco/código/protocolo/configuración/disponibilidad de licencia. Use RUN_ID nuevo.')
    else:
        meta = root / 'meta'
        meta.mkdir(parents=True)
        write_json(meta / 'plan.json', {'run_id': run_id, 'signature': signature,
                     'comparison_id': comparison_id, 'protocol': protocol, 'config': config,
                     'methods': sorted(solvers), 'source_hashes': source_hashes,
                     'method_availability': method_availability,
                     'method_provenance': method_provenance,
                     'bank_id': bank.name, 'role': 'development_only',
                     'created_utc': datetime.now(timezone.utc).isoformat()})
        write_json(meta / 'environment.json', p.environment_report())
        for source in package.glob('*.py'):
            destination = meta / 'code' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        for source in map(Path, validation_artifacts):
            destination = meta / 'validation' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        write_json(meta / 'metrics_schema.json', metric_schema())
        _commit(meta, remote / 'meta')
    if _recover(root / 'report', remote / 'report'):
        for relative, expected in read_json(root / 'report/bundles.json').items():
            if p.verify_bundle(remote / relative) != expected:
                raise ValueError(f'Checkpoint remoto modificado: {relative}')
            if not _recover(root / relative, remote / relative):
                raise ValueError(f'Checkpoint ausente: {relative}')
        print('Reporte completo recuperado y verificado.', flush=True)
        return root
    summaries, failures = [], []
    for x in instances:
        for method, solver in solvers.items():
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                job, remote_job = root / key, remote / key
                learned = method_provenance.get(method, {}).get(f'{x.sha}:{seed}', {})
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone') and not learned:
                    raise ValueError('Falta procedencia del modelo entrenado para este intento.')
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone'):
                    validate_learned_provenance(learned, x, seed)
                context = {'schema_version': SCHEMA, 'comparison_id': comparison_id,
                    'run_id': run_id, 'bank_id': bank.name, 'bank_manifest_sha256': bank_hash,
                    'evaluator_sha256': p.file_hash(p.__file__), 'method': method,
                    'solver_seed': seed, 'instance_id': x.instance_id, 'instance_sha256': x.sha,
                    'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                    'n_customers': x.n, 'objective_scope': OBJECTIVE,
                    'search_budget_seconds': config['search_seconds'],
                    'scenario_set_sha256': p.digest(scenarios[x.n]['evaluation']),
                    **provenance_columns(learned)}
                print(f'{key}: búsqueda / recuperación...', flush=True)
                solution_dir = job / 'solution'
                if not _recover(solution_dir, remote_job / 'solution'):
                    solution_dir.mkdir(parents=True)
                    started = time.perf_counter()
                    try:
                        problem = SearchProblem(x, spec, config['time_scale'])
                        preparation = time.perf_counter() - started
                        settings = {**config['methods'][method],
                                    'seconds': max(0.0, config['search_seconds'] - preparation)}
                        if settings['seconds'] <= 0:
                            solution = problem.result(None, 'time_limit_no_solution', preparation, [])
                        else:
                            solution = solver(problem, seed, settings, solution_dir)
                        solution['search_total_seconds'] = time.perf_counter() - started
                        solution['preparation_seconds'] = preparation
                        solution['nominal'] = None
                        if solution['routes'] is not None:
                            p.validate_routes(x, solution['routes'])
                            conservative = problem.score(solution['routes'])
                            solution['metadata'].update(search_violation_minutes=conservative[0],
                                search_objective_minutes=conservative[1],
                                conservative_feasible=conservative[0] <= 1e-9)
                            solution['nominal'] = p.simulate(x, solution['routes'], 0, nominal_spec,
                                                             'fixed_order', trace=True)
                            if conservative[0] <= 1e-9 and not solution['nominal']['strict_feasible']:
                                raise AssertionError('Solución conservadora viola el simulador nominal.')
                    except Exception as exc:
                        solution = {'routes': None, 'nominal': None, 'status': 'solver_error',
                                    'solve_seconds': time.perf_counter() - started,
                                    'search_total_seconds': time.perf_counter() - started,
                                    'history': [], 'metadata': {'error_type': type(exc).__name__,
                                    'error': str(exc), 'objective_scope': OBJECTIVE}}
                    write_json(solution_dir / 'solution.json', solution)
                    write_json(solution_dir / 'environment.json', p.environment_report())
                    # Read-back antes de evaluar: la solución persistida es la autoridad.
                    solution = read_json(solution_dir / 'solution.json')
                    write_json(solution_dir / 'identity.json', context)
                    write_csv(solution_dir / 'search_history.csv', solution['history'],
                              None if solution['history'] else ['iteration', 'elapsed_seconds'])
                    if solution['routes'] is not None:
                        write_csv(solution_dir / 'planned_routes.csv', route_rows(x, solution['routes'], context))
                        write_csv(solution_dir / 'nominal_events.csv',
                                  [{**context, **e} for e in solution['nominal']['events']])
                        save_plot(x, solution['routes'], solution_dir / 'routes.png',
                                  f'{method} · n={x.n} · seed={seed} · {solution["status"]}')
                    _commit(solution_dir, remote_job / 'solution')
                solution = read_json(solution_dir / 'solution.json')
                if read_json(solution_dir / 'identity.json') != context:
                    raise ValueError('Checkpoint de solución corresponde a otro intento.')
                meta = solution['metadata']
                base = {**context, 'solver_status': solution['status'],
                        'structural_valid': solution['routes'] is not None,
                        'nominal_feasible': solution['nominal']['strict_feasible'] if solution['nominal'] else None,
                        'conservative_feasible': meta.get('conservative_feasible'),
                        'search_violation_minutes': meta.get('search_violation_minutes'),
                        'search_objective_minutes': meta.get('search_objective_minutes'),
                        'solve_seconds': solution['solve_seconds'],
                        'search_total_seconds': solution['search_total_seconds'],
                        'best_bound': meta.get('best_bound'), 'relative_gap': meta.get('relative_gap'),
                        'surrogate_optimal': meta.get('surrogate_optimal'),
                        'model_load_seconds': meta.get('model_load_seconds', 0.0),
                        'tuning_seconds_this_run': 0.0}
                for policy in bank_config['policies']:
                    result_dir = job / policy
                    if solution['routes'] is None:
                        summary = {**base, 'policy': policy, 'evaluation_status': 'no_solution',
                                   **dict.fromkeys(SUMMARY_METRICS)}
                        failures.append({**context, 'policy': policy, 'status': solution['status'],
                                         'error': meta.get('error', '')})
                    else:
                        print(f'{key}: {policy}, {len(scenarios[x.n]["evaluation"])} escenarios...', flush=True)
                        if not _recover(result_dir, remote_job / policy):
                            evaluate_solution(x, solution, scenarios[x.n]['evaluation'], spec, policy,
                                              bank_config['alpha_cvar'], result_dir, context)
                            _commit(result_dir, remote_job / policy)
                        evaluated = read_json(result_dir / 'summary.json')
                        if any(evaluated.get(k) != v for k, v in context.items()) or evaluated.get('policy') != policy:
                            raise ValueError('Checkpoint de evaluación corresponde a otro intento o política.')
                        summary = {**base, **evaluated,
                                   'evaluation_status': 'validated'}
                    summaries.append(summary)
    if p.verify_bundle(bank) != bank_hash:
        raise AssertionError('El banco fue modificado.')
    report = root / 'report'
    report.mkdir(parents=True, exist_ok=True)
    write_csv(report / 'summary.csv', summaries)
    write_csv(report / 'failures.csv', failures, None if failures else ['method', 'n_customers', 'solver_seed', 'status'])
    write_json(report / 'summary.json', summaries)
    inventory = {'meta': p.verify_bundle(root / 'meta')}
    for x in instances:
        for method in solvers:
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                for part in ['solution', *bank_config['policies']]:
                    relative = f'{key}/{part}'
                    if (root / relative / 'CHECKSUMS.json').exists():
                        inventory[relative] = p.verify_bundle(root / relative)
                        if p.verify_bundle(remote / relative) != inventory[relative]:
                            raise ValueError(f'Copia Drive distinta: {relative}')
    write_json(report / 'bundles.json', inventory)
    save_summary_plot(summaries, report / 'metrics.png', bank_config['policies'])
    write_json(report / 'validation.json', {'status': 'completed', 'bank_unchanged': True,
        'attempts': len(instances) * len(solvers) * len(config['seeds']),
        'summary_rows': len(summaries), 'unsolved_policy_rows': len(failures),
        'not_a_claim_of_all_routes_feasible': True, 'ready_for_confirmatory_comparison': False})
    _commit(report, remote / 'report')
    print(f'COMPLETADO: exportaciones verificadas en {remote}', flush=True)
    return root


def save_summary_plot(summaries, path, policies):
    import matplotlib.pyplot as plt
    rows = [r for r in summaries if r['evaluation_status'] == 'validated']
    if not rows:
        return
    fig, axes = plt.subplots(len(policies), 2, figsize=(12, 4 * len(policies)), squeeze=False)
    for row, policy in enumerate(policies):
        for method in sorted({r['method'] for r in rows}):
            data = [r for r in rows if r['policy'] == policy and r['method'] == method]
            for column, metric in enumerate(['scenario_strict_feasibility', 'mean_travel_minutes']):
                axes[row, column].scatter([r['n_customers'] for r in data],
                                         [r[metric] for r in data], label=method, alpha=.65)
                axes[row, column].set(xlabel='Clientes (una geometría por tamaño)', ylabel=metric,
                                      title=f'{policy} · todas las semillas')
                axes[row, column].legend()
        axes[row, 0].set_ylim(-.03, 1.03)
    fig.suptitle('Piloto de desarrollo · costo de viaje y factibilidad deben interpretarse juntos')
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def metric_schema():
    return {'schema_version': SCHEMA, 'objective_scope': OBJECTIVE,
            'cost_scope': 'travel_only_not_full_recourse_cost',
            'scenario_strict_feasibility': 'Fracción de escenarios con cobertura, capacidad, ventanas y retorno válidos.',
            'cvar_travel_minutes': 'CVaR empírico con masa fraccionaria, alpha del banco.',
            'cost_variance_population': 'Varianza de viaje entre escenarios, ddof=0.',
            'mean_net_execution_delta_minutes': 'Viaje ejecutado menos viaje base planeado; puede ser negativo; no es Q óptimo.',
            'relative_gap': 'Gap Gurobi de la misma formulación conservadora; null si no disponible.',
            'search_total_seconds': 'Preparación y llamada al método; incluye construcción, extracción y artefactos propios del solver.',
            'evaluation_seconds': 'Tiempo informado por el evaluador común para Monte Carlo.',
            'trace_validation_seconds': 'Segunda pasada determinista, validación y escritura de trazas.',
            'missing_metrics': 'null en JSON / vacío en CSV cuando no hay solución; no convertir en cero.',
            'scenario_seed': 'Entero de 64 bits serializado en decimal: leer CSV como texto para evitar redondeo.',
            'independent_unit': 'geometry_group_sha256; semillas y escenarios no aumentan N.',
            'training_seconds': 'Costo total del modelo de ese fold/semilla, repetido por política; no sumar filas.',
            'teacher_seconds': 'Etiquetado compartido de geometrías train/validation; costos únicos en manifiesto offline.',
            'offline_cost_id': 'Identifica el inventario de costos únicos de entrenamiento/etiquetado.',
            'comparison_id': 'Mismo banco/evaluador/runner/objetivo/presupuesto/escenarios; no exige algoritmos iguales.',
            'solver_seed': 'En am_supervised/pomo/hbg_facs identifica entrenamiento y búsqueda; no equivale al mismo flujo RNG de ACO.',
            'comparison_keys': ['comparison_id', 'instance_sha256', 'solver_seed', 'policy'],
            'summary_metrics': SUMMARY_METRICS}


In [ ]:
%%writefile svrplab/attention.py
"""Attention Model supervisado propio para 03; ejecución sólo en Google Colab.

Encoder de conjuntos sin posición y decoder pointer con atención y máscaras
conservadoras. Inspirado en la familia AM de Kool et al. (ICLR 2019), pero NO
reproduce su arquitectura exacta, REINFORCE ni sus resultados. Las etiquetas
proceden de búsquedas de desarrollo, nunca de certificados del banco.

Cada depósito seleccionado cierra el vehículo actual y avanza al siguiente.
La flota mantiene su orden original, incluso con capacidades heterogéneas y
rutas vacías. Las construcciones bloqueadas se registran. Durante inferencia,
la completación opcional explora prefijos desde cero, se declara por separado
y no sustituye rutas por etiquetas. No se consultan escenarios estocásticos.

Fuentes técnicas: https://arxiv.org/abs/1803.08475 y
https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
"""
from __future__ import annotations

from collections.abc import Mapping
from dataclasses import asdict, dataclass
import math
import time

import numpy as np
import torch
from torch import nn

from .search import SearchProblem

IMPLEMENTATION = 'project_supervised_attention_v2'
INFERENCE_DEFAULTS = {'seconds': 120., 'n_samples': 16, 'permute_samples': False,
                      'completion_search_nodes': 2048}
MASK_VERSION = 'necessary_capacity_reachability_v2'
_TOL = 1e-9
_NODE_FEATURES = 9
_STATE_FEATURES = 6
_ACTION_FEATURES = 5


@dataclass(frozen=True)
class AttentionConfig:
    model_dim: int = 128
    num_heads: int = 8
    num_layers: int = 3
    dropout: float = .1
    tanh_clip: float = 10.

    def __post_init__(self):
        for name in ('model_dim', 'num_heads', 'num_layers'):
            value = getattr(self, name)
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
                raise ValueError(f'{name} debe ser entero positivo.')
        if self.model_dim % self.num_heads:
            raise ValueError('model_dim debe ser divisible entre num_heads.')
        for name in ('dropout', 'tanh_clip'):
            value = getattr(self, name)
            if (isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number))
                    or not math.isfinite(float(value))):
                raise ValueError(f'{name} debe ser un número finito.')
        if not 0 <= self.dropout < 1 or self.tanh_clip <= 0:
            raise ValueError('dropout debe pertenecer a [0,1), tanh_clip debe ser positivo.')

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_value(cls, config):
        if isinstance(config, cls):
            return config
        if config is None:
            return cls()
        if not isinstance(config, Mapping):
            raise ValueError('La configuración del modelo debe ser un mapping.')
        aliases = {'dim': 'model_dim', 'heads': 'num_heads', 'layers': 'num_layers'}
        values = dict(config)
        for short, full in aliases.items():
            if short in values:
                if full in values:
                    raise ValueError(f'No mezcle {short} y {full}.')
                values[full] = values.pop(short)
        unknown = set(values) - set(cls.__dataclass_fields__)
        if unknown:
            raise ValueError(f'Parámetros de modelo desconocidos: {sorted(unknown)}')
        return cls(**values)


def _node_order(problem, node_order):
    """Permuta sólo clientes; el token de depósito conserva la posición cero."""
    if node_order is None:
        return np.arange(problem.n + 1, dtype=np.int64)
    order = list(node_order)
    if (any(isinstance(j, (bool, np.bool_)) or not isinstance(j, (int, np.integer)) for j in order)
            or sorted(order) != list(range(1, problem.n + 1))):
        raise ValueError('node_order debe ser una permutación de los clientes 1..n, sin depósito.')
    return np.array([0, *order], dtype=np.int64)


def _scales(problem):
    xy = problem.x.locations - problem.x.locations[0]
    coord = max(1., float(np.max(np.abs(xy))))
    horizon = max(1., abs(float(problem.horizon)), float(np.max(np.abs(problem.close))),
                  float(np.max(np.abs(problem.open))))
    capacity = max(1., float(np.max(problem.capacities)))
    return coord, horizon, capacity


def _node_features(problem, order, scales):
    coord, horizon, capacity = scales
    locations = (problem.x.locations - problem.x.locations[0]) / coord
    features = np.column_stack((locations, problem.demands / capacity,
                                problem.open / horizon, problem.close / horizon,
                                problem.x.appear_times / horizon,
                                np.arange(problem.n + 1) == 0,
                                problem.travel[0] / horizon, problem.travel[:, 0] / horizon))
    if not np.isfinite(features).all():
        raise ValueError('Features del modelo no finitos.')
    return np.asarray(features[order], dtype=np.float32)


class AttentionPolicy(nn.Module):
    """Encoder permutation-equivariant y decoder pointer condicionado por flota."""
    def __init__(self, config=None):
        super().__init__()
        self.config = AttentionConfig.from_value(config)
        c = self.config
        self.node_embedding = nn.Linear(_NODE_FEATURES, c.model_dim)
        # Capas construidas separadamente: no comparten pesos ni inicialización.
        self.encoder = nn.ModuleList([
            nn.TransformerEncoderLayer(c.model_dim, c.num_heads,
                                       dim_feedforward=4 * c.model_dim,
                                       dropout=c.dropout, activation='gelu',
                                       batch_first=True, norm_first=True)
            for _ in range(c.num_layers)
        ])
        self.encoder_norm = nn.LayerNorm(c.model_dim)
        self.context = nn.Linear(2 * c.model_dim + _STATE_FEATURES, c.model_dim)
        self.action_embedding = nn.Linear(_ACTION_FEATURES, c.model_dim, bias=False)
        self.glimpse = nn.MultiheadAttention(c.model_dim, c.num_heads,
                                           dropout=c.dropout, batch_first=True)
        self.pointer_query = nn.Linear(c.model_dim, c.model_dim, bias=False)
        self.pointer_key = nn.Linear(c.model_dim, c.model_dim, bias=False)

    def encode(self, features):
        encoded = self.node_embedding(features).unsqueeze(0)
        for layer in self.encoder:
            encoded = layer(encoded)
        return self.encoder_norm(encoded).squeeze(0)

    def next_logits(self, encoded, current_local, state_features, action_features, blocked):
        if bool(blocked.all()):
            raise ValueError('No se calcula softmax de un estado sin acciones factibles.')
        query = self.context(torch.cat((encoded.mean(dim=0), encoded[current_local],
                                        state_features))).view(1, 1, -1)
        candidates = encoded + self.action_embedding(action_features)
        glimpse, _ = self.glimpse(query, candidates.unsqueeze(0), candidates.unsqueeze(0),
                                  key_padding_mask=blocked.unsqueeze(0), need_weights=False)
        logits = (self.pointer_key(candidates) * self.pointer_query(glimpse[0, 0])).sum(dim=-1)
        logits = self.config.tanh_clip * torch.tanh(logits / math.sqrt(self.config.model_dim))
        return logits.masked_fill(blocked, -torch.inf)


def _travel_lower_bound(problem):
    """Admissible all-pairs travel bound, including for nonmetric matrices.

    Intermediate vertices may already be visited: allowing them only relaxes
    the completion problem. Rounding each sum down avoids pruning a feasible
    completion due to numerical overestimation. This cache changes no data.
    """
    cached = getattr(problem, '_attention_travel_bound', None)
    if cached is not None:
        return cached
    lower = np.array(problem.travel, dtype=np.float64, copy=True)
    if not np.isfinite(lower).all() or (lower < 0).any():
        raise ValueError('La anticipación requiere tiempos finitos no negativos.')
    np.fill_diagonal(lower, 0.)
    for intermediate in range(problem.n + 1):
        via = np.nextafter(lower[:, intermediate, None] + lower[None, intermediate, :], -np.inf)
        np.minimum(lower, np.maximum(0., via), out=lower)
    lower.setflags(write=False)
    problem._attention_travel_bound = lower
    return lower


class _State:
    """Estado determinista CPU; sólo las features y la atención pasan al dispositivo."""
    def __init__(self, problem):
        self.problem = problem
        self.vehicle = 0
        self.node = 0
        self.clock = 0.
        self.load = 0.
        self.visited = np.zeros(problem.n + 1, dtype=bool)
        self.visited[0] = True
        self.routes = [[] for _ in range(problem.vehicles)]
        self.lower = _travel_lower_bound(problem)

    def clone(self):
        copied = object.__new__(type(self))
        copied.problem, copied.lower = self.problem, self.lower
        copied.vehicle, copied.node = self.vehicle, self.node
        copied.clock, copied.load = self.clock, self.load
        copied.visited = self.visited.copy()
        copied.routes = [list(route) for route in self.routes]
        return copied

    def _completion_possible_after(self, candidates, remaining, start):
        """Necessary conditions only, never a certificate of completion.

        Each pending customer must fit/reach either the current vehicle after
        the proposed action or a fresh future vehicle. Customers that cannot
        use any future vehicle must fit together in the current residual
        capacity. All conditions relax ordering and hence cannot exclude an
        actually feasible completion. Zero-demand customers remain mandatory.
        """
        p = self.problem
        pending = np.flatnonzero(remaining)
        if not len(candidates) or not len(pending):
            return np.ones(len(candidates), dtype=bool)
        future = p.capacities[self.vehicle + 1:]
        future_start = np.maximum(self.lower[0, pending], p.open[pending])
        future_ok = ((p.demands[pending] <= (float(future.max()) if len(future) else -1.) + _TOL)
                     & (future_start <= p.close[pending] + _TOL)
                     & (future_start + self.lower[pending, 0] <= p.horizon + _TOL))
        residual = p.capacities[self.vehicle] - self.load - p.demands[candidates]
        earliest = np.maximum(start[candidates, None] + self.lower[np.ix_(candidates, pending)],
                              p.open[pending][None, :])
        current_ok = ((p.demands[pending][None, :] <= residual[:, None] + _TOL)
                      & (earliest <= p.close[pending][None, :] + _TOL)
                      & (earliest + self.lower[pending, 0][None, :] <= p.horizon + _TOL))
        selected = candidates[:, None] == pending[None, :]
        individual = (selected | current_ok | future_ok[None, :]).all(axis=1)
        total_left = float(p.demands[pending].sum()) - p.demands[candidates]
        total_capacity = residual + float(future.sum())
        mandatory = (p.demands[pending] * ~future_ok)[None, :] * ~selected
        return (individual & (total_left <= total_capacity + (len(future) + 1) * _TOL)
                & (mandatory.sum(axis=1) <= residual + _TOL))

    @property
    def done(self):
        return self.vehicle == self.problem.vehicles

    def inputs(self, order, scales):
        p = self.problem
        _, horizon, capacity = scales
        remaining = ~self.visited
        arrival = self.clock + p.travel[self.node]
        start = np.maximum(arrival, p.open)
        returned = start + p.travel[:, 0]
        allowed = (remaining & (self.load + p.demands <= p.capacities[self.vehicle] + _TOL)
                   & (start <= p.close + _TOL)
                   & (start + self.lower[:, 0] <= p.horizon + _TOL))
        candidates = np.flatnonzero(allowed)
        allowed[candidates] &= self._completion_possible_after(candidates, remaining, start)
        # Cerrar el último vehículo no puede dejar clientes sin atender, ni de demanda cero.
        can_close = self.clock + p.travel[self.node, 0] <= p.horizon + _TOL
        if self.vehicle == p.vehicles - 1:
            can_close = can_close and not bool(remaining.any())
        elif remaining.any():
            future = p.capacities[self.vehicle + 1:]
            earliest = np.maximum(self.lower[0, remaining], p.open[remaining])
            can_close = (can_close and float(p.demands[remaining].sum()) <= float(future.sum()) + len(future) * _TOL
                         and float(p.demands[remaining].max()) <= float(future.max()) + _TOL
                         and bool((earliest <= p.close[remaining] + _TOL).all())
                         and bool((earliest + self.lower[remaining, 0] <= p.horizon + _TOL).all()))
        allowed[0] = can_close
        residual = float(p.capacities[self.vehicle]) - self.load
        state = np.array([self.clock / horizon, residual / capacity,
                          float(p.capacities[self.vehicle]) / capacity,
                          self.vehicle / max(1, p.vehicles - 1),
                          float(remaining.sum()) / max(1, p.n),
                          float(p.capacities[self.vehicle + 1:].sum()) /
                          max(1., float(p.capacities.sum()))], dtype=np.float32)
        action = np.column_stack((p.travel[self.node] / horizon, start / horizon,
                                  (p.close - start) / horizon, p.demands / capacity,
                                  returned / horizon))
        # El depósito significa regreso y cambio de vehículo, no servicio como cliente.
        action[0] = [p.travel[self.node, 0] / horizon,
                     (self.clock + p.travel[self.node, 0]) / horizon,
                     (p.horizon - self.clock - p.travel[self.node, 0]) / horizon,
                     0., (self.clock + p.travel[self.node, 0]) / horizon]
        return state, np.asarray(action[order], dtype=np.float32), ~allowed[order]

    def advance(self, customer):
        if customer == 0:
            self.vehicle += 1
            self.node, self.clock, self.load = 0, 0., 0.
        else:
            self.clock = max(self.clock + float(self.problem.travel[self.node, customer]),
                             float(self.problem.open[customer]))
            self.load += float(self.problem.demands[customer])
            self.node = customer
            self.visited[customer] = True
            self.routes[self.vehicle].append(customer)


def _tensors(state, order, scales, device):
    context, action, blocked = state.inputs(order, scales)
    return (torch.as_tensor(context, device=device), torch.as_tensor(action, device=device),
            torch.as_tensor(blocked, device=device))


def _prepare(model, problem, device, node_order):
    order = _node_order(problem, node_order)
    scales = _scales(problem)
    inverse = np.empty(problem.n + 1, dtype=np.int64)
    inverse[order] = np.arange(problem.n + 1)
    features = torch.as_tensor(_node_features(problem, order, scales), device=device)
    return model.encode(features), order, inverse, scales


def teacher_loss(model, problem, routes, device, node_order=None):
    """NLL media por decisión; el caller decide train/eval y ejecuta backward.

    Rechaza etiquetas sin cobertura/capacidad o no factibles en el modelo
    conservador. Devuelve (loss, {'decisions', 'nll_sum'}), con nll_sum numérico
    separado del grafo. Incluye un cierre por vehículo y clientes de demanda 0.
    No comprueba procedencia: el registro de profesores debe comprobarla antes.
    """
    violation, _ = problem.score(routes)
    if violation > _TOL:
        raise ValueError('La etiqueta no es factible bajo la matriz conservadora común.')
    encoded, order, inverse, scales = _prepare(model, problem, device, node_order)
    state = _State(problem)
    terms = []
    for route in routes:
        for customer in [*route, 0]:
            context, action, blocked = _tensors(state, order, scales, device)
            label = int(inverse[customer])
            if bool(blocked[label]):
                raise ValueError('Una acción del profesor está prohibida por las máscaras comunes.')
            logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
            term = -torch.log_softmax(logits, dim=-1)[label]
            if not bool(torch.isfinite(term)):
                raise FloatingPointError('NLL no finita; no continuar el entrenamiento.')
            terms.append(term)
            state.advance(int(customer))
    if not state.done or not bool(state.visited.all()) or not terms:
        raise ValueError('Secuencia del profesor incompleta.')
    total = torch.stack(terms).sum()
    return total / len(terms), {'decisions': len(terms), 'nll_sum': float(total.detach().cpu())}


def _sync(device):
    device = torch.device(device)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def _expired(deadline, device):
    if deadline is None:
        return False
    _sync(device)
    return time.perf_counter() >= deadline


def decode(model, problem, device, mode='greedy', generator=None, deadline=None, node_order=None):
    """Construye sin reparar; None significa bloqueo o tiempo, no infactibilidad probada.

    En sample, generator debe corresponder al dispositivo de PyTorch utilizado.
    deadline es un instante de perf_counter, compartido por todos los intentos.
    La sincronización CUDA incluye el trabajo asíncrono en dicho presupuesto.
    """
    if mode not in ('greedy', 'sample'):
        raise ValueError('mode debe ser greedy o sample.')
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            if _expired(deadline, device):
                return None
            encoded, order, inverse, scales = _prepare(model, problem, device, node_order)
            state = _State(problem)
            while not state.done:
                if _expired(deadline, device):
                    return None
                context, action, blocked = _tensors(state, order, scales, device)
                if bool(blocked.all()):
                    return None
                logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
                if not bool(torch.isfinite(logits[~blocked]).all()):
                    raise FloatingPointError('Logits no finitos en la inferencia.')
                if mode == 'greedy':
                    local = int(torch.argmax(logits).item())
                else:
                    probabilities = torch.softmax(logits, dim=-1)
                    local = int(torch.multinomial(probabilities, 1, generator=generator).item())
                state.advance(int(order[local]))
            # La validación independiente también se incluye en el tiempo de solve_attention.
            violation, _ = problem.score(state.routes)
            return state.routes if violation <= _TOL else None
    finally:
        model.train(was_training)


class UniformMaskedPolicy(nn.Module):
    """Untrained diagnostic: exactly uniform logits over the common mask."""
    def __init__(self, config=None):
        super().__init__()
        self.config = AttentionConfig.from_value(config)

    def encode(self, features):
        return features.new_zeros((len(features), 1))

    def next_logits(self, encoded, current_local, state_features, action_features, blocked):
        return encoded.new_zeros(len(blocked)).masked_fill(blocked, -torch.inf)


def policy_completion_search(model, problem, device, deadline, max_nodes=2048):
    """Policy-ordered depth-first completion, separately declared from a rollout.

    Restarts from the empty state, backtracks only through the shared action
    mask, and shares the caller's deadline. No external routes or solver are
    consulted. These search trajectories have no rollout probability and must
    not be added to on-policy training or replay as sampled trajectories.
    """
    if type(max_nodes) is not int or max_nodes < 0:
        raise ValueError('max_nodes debe ser entero no negativo.')
    stats = {'expanded_nodes': 0, 'backtracks': 0, 'termination': 'node_limit',
             'construction': 'policy_ordered_depth_first_completion',
             'is_policy_sample': False, 'mask_version': MASK_VERSION}
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            if max_nodes == 0 or _expired(deadline, device):
                stats['termination'] = 'time_limit' if max_nodes else 'disabled'
                return None, stats
            encoded, order, inverse, scales = _prepare(model, problem, device, None)
            stack = [(_State(problem), None)]
            while stack:
                if _expired(deadline, device):
                    stats['termination'] = 'time_limit'
                    return None, stats
                state, choices = stack.pop()
                if state.done:
                    if bool(state.visited.all()) and problem.score(state.routes)[0] <= _TOL:
                        stats['termination'] = 'feasible_found'
                        return state.routes, stats
                    stats['backtracks'] += 1
                    continue
                if choices is None:
                    if stats['expanded_nodes'] >= max_nodes:
                        return None, stats
                    stats['expanded_nodes'] += 1
                    context, action, blocked = _tensors(state, order, scales, device)
                    if bool(blocked.all()):
                        stats['backtracks'] += 1
                        continue
                    logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
                    if not bool(torch.isfinite(logits[~blocked]).all()):
                        raise FloatingPointError('Logits no finitos en búsqueda de completación.')
                    choices = [int(order[local]) for local in torch.argsort(
                        logits, descending=True, stable=True).cpu().tolist() if not bool(blocked[local])]
                if len(choices) > 1:
                    stack.append((state, choices[1:]))
                child = state.clone()
                child.advance(choices[0])
                stack.append((child, None))
            if not stack:
                stats['termination'] = 'tree_exhausted_under_mask'
            return None, stats
    finally:
        model.train(was_training)


def _inference_config(config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración de inferencia debe ser un mapping.')
    unknown = set(config or {}) - set(INFERENCE_DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros de inferencia desconocidos: {sorted(unknown)}')
    result = INFERENCE_DEFAULTS | dict(config or {})
    seconds = result['seconds']
    if (isinstance(seconds, (bool, np.bool_)) or not isinstance(seconds, (int, float, np.number))
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    samples = result['n_samples']
    if isinstance(samples, (bool, np.bool_)) or not isinstance(samples, (int, np.integer)) or samples < 0:
        raise ValueError('n_samples debe ser entero no negativo.')
    if not isinstance(result['permute_samples'], bool):
        raise ValueError('permute_samples debe ser booleano.')
    nodes = result['completion_search_nodes']
    if type(nodes) is not int or nodes < 0:
        raise ValueError('completion_search_nodes debe ser entero no negativo.')
    return result | {'seconds': float(seconds), 'n_samples': int(samples)}


def solve_attention(problem: SearchProblem, seed, config, model, device):
    """Un greedy y hasta n_samples, todos dentro del mismo límite de inferencia."""
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    effective = _inference_config(config)
    device = torch.device(device)
    _sync(device)
    started = time.perf_counter()
    deadline = started + effective['seconds']
    generator = torch.Generator(device=device).manual_seed(int(seed) % (2**63 - 1))
    rng = np.random.default_rng(int(seed))
    best, best_score, history = None, (math.inf, math.inf), []
    attempts = completed = failed = 0
    termination = 'sample_limit'
    completion_search = {'termination': 'not_needed', 'expanded_nodes': 0, 'is_policy_sample': False}
    for attempt in range(effective['n_samples'] + 1):
        if _expired(deadline, device):
            termination = 'time_limit'
            break
        mode = 'greedy' if attempt == 0 else 'sample'
        order = (rng.permutation(np.arange(1, problem.n + 1))
                 if mode == 'sample' and effective['permute_samples'] else None)
        attempts += 1
        routes = decode(model, problem, device, mode=mode, generator=generator,
                        deadline=deadline, node_order=order)
        if routes is None:
            failed += 1
        else:
            score = problem.score(routes)
            completed += 1
            if score < best_score:
                best, best_score = routes, score
                _sync(device)
                history.append({'iteration': attempt, 'elapsed_seconds': time.perf_counter() - started,
                                'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
                                'phase': mode})
        if _expired(deadline, device):
            termination = 'time_limit'
            break
    if best is None and effective['completion_search_nodes'] and not _expired(deadline, device):
        best, completion_search = policy_completion_search(
            model, problem, device, deadline, effective['completion_search_nodes'])
        if best is not None:
            best_score = problem.score(best)
            history.append({'iteration': attempts, 'elapsed_seconds': time.perf_counter() - started,
                            'violation_minutes': float(best_score[0]), 'travel_minutes': float(best_score[1]),
                            'phase': 'policy_completion_search'})
    if best is None and completion_search['termination'] == 'not_needed':
        completion_search['termination'] = ('no_remaining_budget' if effective['completion_search_nodes'] else 'disabled')
    if _expired(deadline, device):
        termination = 'time_limit'
    _sync(device)
    elapsed = time.perf_counter() - started
    result = problem.result(
        best, 'feasible' if best is not None else 'no_solution', elapsed, history,
        method='am_supervised', seed=int(seed), implementation=IMPLEMENTATION,
        model_config=model.config.to_dict(), effective_config=effective,
        termination=termination, attempts=attempts, completed_decodes=completed, failed_decodes=failed,
        raw_decode_complete_fraction=completed / attempts if attempts else None,
        completion_search=completion_search, mask_version=MASK_VERSION,
        solution_origin='policy_completion_search' if completion_search['termination'] == 'feasible_found' else
                        'raw_policy_decode' if best is not None else 'none',
        inference_device=str(device), budget_includes_construction=True,
        budget_includes_encoding=True, budget_overrun_seconds=max(0., elapsed - effective['seconds']),
        time_limit_granularity='synchronized_decoder_step_or_completed_validation',
        search_objective='conservative_upper_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        construction='from_scratch_without_certificates_or_teacher_fallback',
        seed_reproducibility='same_environment_and_sample_cutoff; wall_time_cutoff_may_differ',
        no_solution_scope='decoder_dead_end_or_budget_not_infeasibility_proof',
        training_included_in_solve_seconds=False)
    _sync(device)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - effective['seconds'])
    return result


In [ ]:
%%writefile svrplab/teacher_feasibility.py
"""Profesor de factibilidad independiente para 03; ejecutar sólo en Colab.

CP-SAT construye rutas desde cero sobre el mismo SearchProblem conservador.
No consulta certificados, solvers del certificador, escenarios ni rutas/hints
externos. No optimiza costo: OPTIMAL de CP-SAT en un modelo sin objetivo sólo
indica factibilidad. El caller registra tiempo y procedencia como costo docente.

La formulación sin índices de vehículo requiere que CADA capacidad pueda
transportar toda la demanda. Si no se cumple, devuelve unsupported_capacity;
no relaja ni agrega vehículos. AddMultipleCircuit exige que cada circuito pase
por el depósito, incluso cuando hay tiempos cero. Clientes de demanda cero son
obligatorios mediante self-loops falsos. Sobran vehículos: se conservan vacíos.

API primaria OR-Tools: https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html
"""
from __future__ import annotations

from collections.abc import Mapping
import importlib.metadata
import math
from numbers import Integral, Real
import time


IMPLEMENTATION = 'project_independent_cp_sat_teacher_v1'
DEFAULTS = {'seconds': 600., 'workers': 1}
_TOL = 1e-9
_MAX_SAFE_INTEGER = 2**60


class _BudgetExpired(Exception):
    pass


def _deadline(deadline):
    if time.perf_counter() >= deadline:
        raise _BudgetExpired


def validate_config(config=None):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración del profesor CP-SAT debe ser un mapping.')
    unknown = set(config or {}) - set(DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros del profesor desconocidos: {sorted(unknown)}')
    result = DEFAULTS | dict(config or {})
    seconds = result['seconds']
    if (isinstance(seconds, bool) or not isinstance(seconds, Real)
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    workers = result['workers']
    if isinstance(workers, bool) or not isinstance(workers, Integral) or workers != 1:
        raise ValueError('El profesor reproducible requiere workers=1.')
    return {'seconds': float(seconds), 'workers': 1}


def _cp_sat():
    # Importar este módulo no carga OR-Tools ni inicia ningún solver.
    try:
        from ortools.sat.python import cp_model
    except ImportError as exc:
        raise RuntimeError('El profesor requiere OR-Tools instalado en Google Colab.') from exc
    return cp_model


def _integer_problem(problem, deadline):
    """Reconstruye la rejilla ya fijada; nunca redondea una matriz distinta."""
    _deadline(deadline)
    scale = problem.time_scale
    if isinstance(scale, bool) or not isinstance(scale, Integral) or scale < 1:
        raise ValueError('SearchProblem.time_scale debe ser entero positivo.')
    scale = int(scale)
    matrix = []
    for i in range(problem.n + 1):
        _deadline(deadline)
        row = []
        for j in range(problem.n + 1):
            value = float(problem.travel[i, j]) * scale
            if not math.isfinite(value) or value < 0 or value > _MAX_SAFE_INTEGER:
                raise ValueError('La matriz de búsqueda tiene tiempos inválidos.')
            integer = round(value)
            if not math.isclose(value, integer, rel_tol=0., abs_tol=1e-6):
                raise ValueError('La matriz de búsqueda no pertenece a su time_scale; no redondearla de nuevo.')
            if i == j and integer != 0:
                raise ValueError('La diagonal de viajes debe ser cero.')
            row.append(int(integer))
        matrix.append(row)
    lower, upper = [0], [0]
    horizon_value = float(problem.horizon) * scale
    if not math.isfinite(horizon_value) or not 0 <= horizon_value <= _MAX_SAFE_INTEGER:
        raise ValueError('Horizonte inválido para CP-SAT.')
    horizon = math.floor(horizon_value)
    upper[0] = horizon
    for customer in range(1, problem.n + 1):
        _deadline(deadline)
        start = float(problem.open[customer]) * scale
        end = float(problem.close[customer]) * scale
        if not all(math.isfinite(value) and 0 <= value <= _MAX_SAFE_INTEGER for value in (start, end)):
            raise ValueError('Ventanas inválidas para CP-SAT.')
        lower.append(math.ceil(start))
        upper.append(min(math.floor(end), horizon))
    return {'matrix': matrix, 'lower': lower, 'upper': upper, 'horizon': horizon, 'time_scale': scale}


def _build_model(problem, data, cp, deadline):
    model = cp.CpModel()
    times = {}
    for customer in range(1, problem.n + 1):
        _deadline(deadline)
        lo, hi = data['lower'][customer], data['upper'][customer]
        if lo > hi:
            model.add(False)
        times[customer] = model.new_int_var(min(lo, hi), hi, f'start_{customer}')
    edges, variables = [], {}
    # Registrar incluso clientes aislados. El literal falso impide omitirlos.
    for customer in range(1, problem.n + 1):
        edges.append((customer, customer, model.new_constant(0)))
    departures = []
    for origin in range(problem.n + 1):
        _deadline(deadline)
        for target in range(problem.n + 1):
            if origin == target:
                continue
            transit = data['matrix'][origin][target]
            earliest = 0 if origin == 0 else data['lower'][origin]
            latest = data['horizon'] if target == 0 else data['upper'][target]
            if earliest + transit > latest:
                continue
            selected = model.new_bool_var(f'edge_{origin}_{target}')
            edges.append((origin, target, selected))
            variables[origin, target] = selected
            if origin == 0:
                departures.append(selected)
                model.add(times[target] >= transit).only_enforce_if(selected)
            elif target == 0:
                model.add(times[origin] + transit <= data['horizon']).only_enforce_if(selected)
            else:
                model.add(times[target] >= times[origin] + transit).only_enforce_if(selected)
    # No objective and no hints. The circuit constraint excludes subtours even
    # when all their arc durations are zero; time inequalities alone would not.
    model.add_multiple_circuit(edges)
    model.add(sum(departures) <= problem.vehicles)
    _deadline(deadline)
    return model, variables


def _extract_routes(problem, selected):
    departures = sorted(target for origin, target in selected if origin == 0)
    if len(departures) > problem.vehicles:
        raise ValueError('CP-SAT excedió la flota fija.')
    successors = {}
    for origin, target in selected:
        if origin:
            if origin in successors:
                raise ValueError('CP-SAT devolvió dos sucesores para un cliente.')
            successors[origin] = target
    seen, routes = set(), []
    for first in departures:
        route, customer = [], first
        while customer:
            if customer in seen or customer not in successors or not 1 <= customer <= problem.n:
                raise ValueError('La extracción encontró ciclo desconectado o cliente inválido.')
            seen.add(customer)
            route.append(customer)
            customer = successors[customer]
        routes.append(route)
    if seen != set(range(1, problem.n + 1)):
        raise ValueError('La solución CP-SAT no cubre todos los clientes.')
    return routes + [[] for _ in range(problem.vehicles - len(routes))]


def _finish(problem, routes, status, started, config, metadata, history):
    validation_started = time.perf_counter()
    result = problem.result(routes, status, validation_started - started, history, **metadata)
    finished = time.perf_counter()
    result['solve_seconds'] = finished - started
    result['metadata']['validation_seconds'] += finished - validation_started
    result['metadata']['budget_overrun_seconds'] = max(0., finished - started - config['seconds'])
    return result


def solve_feasibility(problem, seed, config=None):
    """Obtiene una etiqueta independiente, sin optimizar costo ni usar certificados.

    El presupuesto único incluye importación, preparación, modelo, solver,
    extracción y validación. No interrumpe la operación nativa en curso; deja
    constancia de cualquier exceso. UNKNOWN no prueba infactibilidad.
    """
    started = time.perf_counter()
    effective = validate_config(config)
    if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:
        raise ValueError('seed debe ser entero no negativo.')
    seed = int(seed)
    deadline = started + effective['seconds']
    metadata = {
        'method': 'cp_sat_feasibility', 'seed': seed, 'cp_sat_seed': seed % (2**31 - 1),
        'implementation': IMPLEMENTATION, 'effective_config': effective,
        'construction': 'independent_from_scratch_without_certificates_or_hints',
        'feasibility_only': True, 'surrogate_optimal': False, 'stochastic_optimal': False,
        'cost_optimization_performed': False, 'best_bound': None,
        'cp_sat_status': None, 'cp_sat_status_name': 'NOT_RUN', 'solver_version': None,
        'termination': 'not_started', 'search_solver_seconds': 0., 'model_seconds': 0.,
        'import_seconds': 0., 'validation_seconds': 0., 'response_stats': None,
        'time_scale': int(problem.time_scale), 'capacity_scope': 'nonbinding_for_each_vehicle',
        'conservative_integer_infeasible': False,
        'infeasibility_scope': 'conservative_integer_model_only_not_original_stochastic_problem',
        'optimal_status_scope': 'satisfaction_of_feasibility_constraints_not_minimum_route_cost',
        'budget_includes_imports': True, 'budget_includes_construction': True,
        'budget_includes_extraction_and_validation': True,
        'time_limit_granularity': 'model_row_or_native_solver_call_or_validation',
        'seed_reproducibility': 'same_ortools_environment_and_one_worker; wall_time_cutoff_may_differ',
    }
    routes, history, status = None, [], 'no_solution'
    try:
        _deadline(deadline)
        total_demand = math.fsum(float(value) for value in problem.demands[1:])
        if (not math.isfinite(total_demand) or total_demand < 0
                or any(not math.isfinite(float(value)) or float(value) < 0 for value in problem.demands)):
            raise ValueError('Demandas inválidas.')
        if any(not math.isfinite(float(capacity)) or float(capacity) < 0 for capacity in problem.capacities):
            raise ValueError('Capacidades inválidas.')
        if problem.vehicles < 1 or any(float(capacity) + _TOL < total_demand for capacity in problem.capacities):
            metadata['termination'] = 'unsupported_capacity'
            return _finish(problem, None, 'unsupported_capacity', started, effective, metadata, history)
        model_started = time.perf_counter()
        try:
            data = _integer_problem(problem, deadline)
        finally:
            metadata['model_seconds'] += time.perf_counter() - model_started
        if problem.n == 0:
            routes = [[] for _ in range(problem.vehicles)]
            metadata['termination'] = 'empty_instance_feasible'
            return _finish(problem, routes, 'feasible', started, effective, metadata, history)
        import_started = time.perf_counter()
        cp = _cp_sat()
        metadata['solver_version'] = importlib.metadata.version('ortools')
        metadata['import_seconds'] = time.perf_counter() - import_started
        _deadline(deadline)
        model_started = time.perf_counter()
        try:
            model, variables = _build_model(problem, data, cp, deadline)
            model_error = model.validate()
        finally:
            metadata['model_seconds'] += time.perf_counter() - model_started
        if model_error:
            metadata.update(termination='model_invalid', model_validation_error=model_error,
                            cp_sat_status_name='MODEL_INVALID')
            return _finish(problem, None, 'model_invalid', started, effective, metadata, history)
        solver = cp.CpSolver()
        solver.parameters.num_search_workers = effective['workers']
        solver.parameters.random_seed = metadata['cp_sat_seed']
        solver.parameters.stop_after_first_solution = True
        _deadline(deadline)
        solver.parameters.max_time_in_seconds = max(0., deadline - time.perf_counter())
        # A zero native limit can mean no limit in solver APIs: never invoke it.
        if solver.parameters.max_time_in_seconds <= 0:
            raise _BudgetExpired
        solver_started = time.perf_counter()
        result_code = solver.solve(model)
        metadata.update(search_solver_seconds=time.perf_counter() - solver_started,
                        cp_sat_status=int(result_code), cp_sat_status_name=solver.status_name(result_code),
                        response_stats=solver.response_stats(), model_arc_count=len(variables))
        if result_code in (cp.FEASIBLE, cp.OPTIMAL):
            validation_started = time.perf_counter()
            selected = [arc for arc, variable in variables.items() if solver.boolean_value(variable)]
            candidate = _extract_routes(problem, selected)
            violation, cost = problem.score(candidate)
            if not math.isfinite(violation) or not math.isfinite(cost) or violation > _TOL:
                raise ValueError('La etiqueta CP-SAT no pasó la validación conservadora común.')
            metadata['validation_seconds'] += time.perf_counter() - validation_started
            routes, status = candidate, 'feasible'
            metadata['termination'] = 'feasible_found'
            history.append({'phase': 'independent_feasibility_teacher',
                            'elapsed_seconds': time.perf_counter() - started,
                            'violation_minutes': float(violation), 'travel_minutes': float(cost)})
        elif result_code == cp.INFEASIBLE:
            metadata.update(termination='conservative_integer_infeasible', conservative_integer_infeasible=True)
        elif result_code == cp.MODEL_INVALID:
            status, metadata['termination'] = 'model_invalid', 'model_invalid'
        else:
            metadata['termination'] = 'unknown_no_feasible_solution'
    except _BudgetExpired:
        metadata['termination'] = 'time_limit_before_search'
    return _finish(problem, routes, status, started, effective, metadata, history)


In [ ]:
%%writefile svrplab/supervised_data.py
"""Corpus heurístico auditado y particiones por geometría para el método 03.

La construcción se ejecuta exclusivamente en Colab. ACO/Tabú y el respaldo
CP-SAT parten de cero: nunca leen certificados ni escenarios de evaluación.
Los cuatro folds del piloto son desarrollo cruzado, no un test confirmatorio.
"""
from __future__ import annotations

import math
import shutil
import time
from collections.abc import Mapping
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

from . import experiments as exp
from . import metaheuristics as mh
from . import pilot as p
from .search import OBJECTIVE, SearchProblem

SCHEMA = 'supervised-teachers-v2'
TEACHER_DEFAULTS = {
    'seeds': [101, 211, 307], 'seconds_per_method': 120., 'time_scale': 1000,
    'aco': {'max_iterations': 200, 'ants': 20, 'alpha': 1., 'beta': 2.,
            'evaporation': .1, 'q0': .2},
    'tabu': {'max_iterations': 200, 'neighborhood_samples': 80, 'tenure': 15,
             'initial_restarts': 20},
    'fallback': {'enabled': False, 'seeds': [401, 503, 607],
                 'seconds_per_attempt': 600., 'workers': 1},
}
_ROLES = ('train_geometry_groups', 'validation_geometry_groups',
          'heldout_geometry_groups')
_COSTS = ('teacher_search_seconds', 'teacher_preparation_seconds',
          'teacher_validation_seconds', 'teacher_wall_seconds')


def _require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Experimentación exclusivamente en Google Colab.') from exc


def _registry(instances):
    rows = []
    for x in instances:
        group = x.provenance.get('geometry_group_sha256')
        if (not isinstance(group, str) or len(group) != 64
                or any(c not in '0123456789abcdef' for c in group)):
            raise ValueError('Cada instancia requiere geometry_group_sha256 válido.')
        rows.append({'instance_id': x.instance_id, 'instance_sha256': x.sha,
                     'geometry_group_sha256': group, 'n_customers': x.n})
    if not rows or len({r['instance_id'] for r in rows}) != len(rows):
        raise ValueError('Registro de instancias vacío o con identificadores repetidos.')
    if len({r['instance_sha256'] for r in rows}) != len(rows):
        raise ValueError('Registro de instancias con hashes repetidos.')
    return sorted(rows, key=lambda r: (r['geometry_group_sha256'], r['instance_sha256']))


def make_geometry_folds(instances):
    """Cada grupo es heldout una vez; validación es el siguiente hash circular.

    Con las cuatro geometrías del piloto quedan dos grupos de entrenamiento,
    uno de validación y uno excluido de ambos. Ningún derivado cambia de rol.
    """
    instances = list(instances)
    groups = sorted({r['geometry_group_sha256'] for r in _registry(instances)})
    if len(groups) < 3:
        raise ValueError('Se requieren al menos tres geometrías independientes para los folds.')
    folds = []
    for i, heldout in enumerate(groups):
        validation = groups[(i + 1) % len(groups)]
        fold = {'fold_id': f'geometry_{i:02d}',
                'train_geometry_groups': [g for g in groups if g not in (heldout, validation)],
                'validation_geometry_groups': [validation],
                'heldout_geometry_groups': [heldout]}
        validate_fold(fold, instances)
        folds.append(fold)
    return folds


def validate_fold(fold, instances):
    """Rechaza cualquier mezcla de roles, grupo ajeno u omisión del registro."""
    known = {r['geometry_group_sha256'] for r in _registry(instances)}
    if (not isinstance(fold, Mapping) or not isinstance(fold.get('fold_id'), str)
            or not fold['fold_id'] or Path(fold['fold_id']).name != fold['fold_id']
            or fold['fold_id'] in ('.', '..')):
        raise ValueError('fold_id inválido.')
    used = set()
    for role in _ROLES:
        values = fold.get(role)
        if (not isinstance(values, (list, tuple)) or not values
                or any(not isinstance(v, str) for v in values)
                or len(set(values)) != len(values)):
            raise ValueError(f'Rol vacío, repetido o inválido: {role}')
        group_set = set(values)
        if not group_set <= known or used & group_set:
            raise ValueError('Grupos desconocidos o solapados entre train/validation/heldout.')
        used.update(group_set)
    if used != known or len(fold['heldout_geometry_groups']) != 1:
        raise ValueError('Todos los grupos deben tener exactamente un rol y un solo heldout.')
    return True



def validate_holdout_fold(fold, instances):
    """Partición final explícita: varias geometrías test, ninguna reutilizada.

    El runner final verifica además la reserva sellada y aporta su hash. Este
    validador comprueba exhaustividad/disjunción sin relajar los folds históricos.
    """
    known = {r['geometry_group_sha256'] for r in _registry(instances)}
    if (not isinstance(fold, Mapping) or fold.get('split_strategy') != 'independent_train_validation_test_v1'
            or not isinstance(fold.get('fold_id'), str) or not fold['fold_id']
            or Path(fold['fold_id']).name != fold['fold_id'] or fold['fold_id'] in ('.', '..')):
        raise ValueError('Partición holdout final inválida.')
    _check_hash(fold.get('split_registry_sha256'), 'split_registry_sha256')
    used = set()
    for role in _ROLES:
        values = fold.get(role)
        if (not isinstance(values, list) or not values or len(set(values)) != len(values)
                or any(not isinstance(value, str) for value in values)):
            raise ValueError('Rol holdout vacío o repetido.')
        group_set = set(values)
        if not group_set <= known or used & group_set:
            raise ValueError('Geometría ajena o compartida entre train/validation/test.')
        used.update(group_set)
    if used != known:
        raise ValueError('Partición holdout omite geometrías de la población.')
    return True

def _configuration(config):
    if not isinstance(config, Mapping) or set(config) - set(TEACHER_DEFAULTS):
        raise ValueError('Configuración de profesores inválida o con campos desconocidos.')
    cfg = {**TEACHER_DEFAULTS, **dict(config)}
    seeds = cfg['seeds']
    if (not isinstance(seeds, (list, tuple)) or not seeds
            or any(type(s) is not int or s < 0 for s in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de profesores inválidas o repetidas.')
    seconds = cfg['seconds_per_method']
    if (type(seconds) not in (int, float) or not math.isfinite(seconds) or seconds <= 0
            or type(cfg['time_scale']) is not int or cfg['time_scale'] <= 0):
        raise ValueError('Presupuesto y escala temporal deben ser positivos y finitos.')
    cfg['seeds'] = list(seeds)
    cfg['seconds_per_method'] = float(seconds)
    for name, defaults in [('aco', mh.ACO_DEFAULTS), ('tabu', mh.TABU_DEFAULTS)]:
        if not isinstance(cfg[name], Mapping) or 'seconds' in cfg[name]:
            raise ValueError('Use seconds_per_method para el presupuesto común de profesores.')
        cfg[name] = mh._configuration(defaults, {**cfg[name], 'seconds': float(seconds)})
        cfg[name].pop('seconds')
    fallback = cfg['fallback']
    if (not isinstance(fallback, Mapping)
            or set(fallback) - set(TEACHER_DEFAULTS['fallback'])):
        raise ValueError('Configuración de respaldo inválida.')
    fallback = {**TEACHER_DEFAULTS['fallback'], **dict(fallback)}
    if type(fallback['enabled']) is not bool:
        raise ValueError('fallback.enabled debe ser booleano.')
    seeds = fallback['seeds']
    if (not isinstance(seeds, (list, tuple)) or not seeds
            or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de respaldo inválidas.')
    seconds = fallback['seconds_per_attempt']
    if (type(seconds) not in (int, float) or not math.isfinite(seconds) or seconds <= 0
            or type(fallback['workers']) is not int or fallback['workers'] != 1):
        raise ValueError('Respaldo requiere presupuesto positivo y un worker.')
    cfg['fallback'] = {**fallback, 'seeds': list(seeds), 'seconds_per_attempt': float(seconds)}
    return cfg


def solve_feasibility(problem, seed, config):
    """Dependencia diferida: los folds de 04/05 no necesitan cargar un solver."""
    from .teacher_feasibility import solve_feasibility as solve
    return solve(problem, seed, config)


def _check_hash(value, name):
    if (not isinstance(value, str) or len(value) != 64
            or any(c not in '0123456789abcdef' for c in value)):
        raise ValueError(f'Hash inválido: {name}')


def _check_result(problem, result, spec):
    """Validez de etiqueta: estructura, cota conservadora y simulador nominal."""
    validation = {'eligible_label': False, 'structural_valid': False,
                  'conservative_feasible': False, 'nominal_feasible': False,
                  'reason': 'no_solution', 'objective_scope': OBJECTIVE}
    nominal = None
    try:
        routes = result.get('routes')
        if routes is None:
            return validation, nominal
        p.validate_routes(problem.x, routes)
        validation['structural_valid'] = True
        violation, cost = problem.score(routes)
        if not math.isfinite(violation) or not math.isfinite(cost):
            raise ValueError('Puntuación de profesor no finita.')
        validation.update(search_violation_minutes=float(violation),
                          search_objective_minutes=float(cost),
                          conservative_feasible=violation <= 1e-9)
        nominal_spec = p.Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
        # El 0 es una identidad formal: nominal_mean no muestrea ruido.
        nominal = p.simulate(problem.x, routes, 0, nominal_spec, 'fixed_order', trace=True)
        validation['nominal_feasible'] = bool(nominal['strict_feasible'])
        validation['eligible_label'] = validation['conservative_feasible'] and validation['nominal_feasible']
        validation['reason'] = ('accepted_heuristic_label' if validation['eligible_label'] else
                                'conservative_infeasible' if not validation['conservative_feasible'] else
                                'nominal_infeasible')
    except Exception as exc:
        validation.update(reason='validation_error', error_type=type(exc).__name__, error=str(exc))
    return validation, nominal


def _run_attempt(x, spec, cfg, method, seed, identity, directory):
    """Persiste incluso soluciones rechazadas; retorna KeyboardInterrupt diferido."""
    directory.mkdir(parents=True)
    started = time.perf_counter()
    preparation = search_seconds = validation_seconds = 0.
    result = {'routes': None, 'history': [], 'metadata': {}, 'status': 'solver_error',
              'solve_seconds': 0.}
    validation = {'eligible_label': False, 'structural_valid': False,
                  'conservative_feasible': False, 'nominal_feasible': False,
                  'reason': 'solver_error'}
    nominal, interrupt = None, None
    problem, search_started = None, None
    try:
        problem = SearchProblem(x, spec, cfg['time_scale'])
        preparation = time.perf_counter() - started
        search_started = time.perf_counter()
        budget = max(0., identity['search_budget_seconds'] - preparation)
        if method == 'cp_sat_feasibility':
            solver = solve_feasibility
            settings = {'workers': cfg['fallback']['workers'], 'seconds': budget}
        else:
            solver = {'aco': mh.solve_aco, 'tabu': mh.solve_tabu}[method]
            settings = {**cfg[method], 'seconds': budget}
        result = solver(problem, seed, settings)
        if not isinstance(result, dict):
            raise TypeError('El profesor debe devolver un diccionario de resultado.')
        search_seconds = time.perf_counter() - search_started
        search_started = None
        validation_started = time.perf_counter()
        validation, nominal = _check_result(problem, result, spec)
        validation_seconds = time.perf_counter() - validation_started
    except (Exception, KeyboardInterrupt) as exc:
        if search_started is not None:
            search_seconds = time.perf_counter() - search_started
        elif problem is None:
            preparation = time.perf_counter() - started
        if not isinstance(result, dict):
            result = {'routes': None, 'history': [], 'metadata': {}}
        result.update(status='interrupted' if isinstance(exc, KeyboardInterrupt) else 'solver_error')
        result['metadata'] = {**result.get('metadata', {}),
                              'error_type': type(exc).__name__, 'error': str(exc)}
        validation.update(eligible_label=False, reason=result['status'])
        if isinstance(exc, KeyboardInterrupt):
            interrupt = exc
    if x.sha != identity['instance_sha256']:
        raise AssertionError('El profesor modificó la instancia.')
    result.setdefault('history', [])
    result.setdefault('metadata', {})
    result.setdefault('routes', None)
    result.setdefault('solve_seconds', search_seconds)
    # JSON conserva rutas de intentos inválidos; el CSV sólo requiere estructura válida.
    exp.write_json(directory / 'result.json', result)
    exp.write_json(directory / 'nominal.json', nominal)
    exp.write_json(directory / 'validation.json', validation)
    exp.write_csv(directory / 'search_history.csv', result['history'],
                  None if result['history'] else ['iteration', 'elapsed_seconds'])
    rows = exp.route_rows(x, result['routes'], identity) if validation['structural_valid'] else []
    exp.write_csv(directory / 'planned_routes.csv', rows,
                  None if rows else ['instance_id', 'teacher_method', 'teacher_seed', 'vehicle', 'position', 'node'])
    if nominal is not None:
        exp.write_csv(directory / 'nominal_events.csv', nominal['events'])
    costs = {'teacher_preparation_seconds': float(preparation),
             'teacher_search_seconds': float(search_seconds),
             'teacher_validation_seconds': float(validation_seconds),
             'teacher_wall_seconds': time.perf_counter() - started,
             'wall_scope': 'preparation_search_validation_local_artifacts_before_final_record; excludes_drive_sync'}
    exp.write_json(directory / 'attempt.json', {**identity, **costs,
                   'status': result['status'], 'label_validation': validation,
                   'solver_termination': result['metadata'].get('termination'),
                   'solver_iterations': result['metadata'].get('iterations'),
                   'solver_error_type': result['metadata'].get('error_type'),
                   'solver_error': result['metadata'].get('error'),
                   'routes_sha256': p.digest(result['routes']) if result['routes'] is not None else None,
                   'construction': 'independent_from_scratch_without_certificates',
                   'label_kind': 'heuristic_not_proven_optimal'})
    return interrupt


def _cost_summary(attempts, groups):
    costs = []
    for group in sorted(groups):
        subset = [a for a in attempts if a['geometry_group_sha256'] == group]
        costs.append({'geometry_group_sha256': group, 'attempts': len(subset),
                      'accepted_attempts': sum(a['label_validation']['eligible_label'] for a in subset),
                      'rejected_attempts': sum(not a['label_validation']['eligible_label'] for a in subset),
                      **{key: sum(a[key] for a in subset) for key in _COSTS}})
    return costs


def build_teacher_corpus(instances, spec, config, work_root, drive_root, bank_id,
                         bank_sha256, evaluator_sha256):
    """Genera o recupera cada intento inmutable y retorna la raíz del corpus.

    El llamador debe cargar antes el banco aceptado mediante p.load_bank.
    Identidad: banco, registro íntegro, tráfico, código y configuración efectiva.
    Sólo se sella index/ al terminar: nunca la raíz que recibe nuevos intentos.
    """
    _require_colab()
    instances = list(instances)
    registry, cfg = _registry(instances), _configuration(config)
    folds = make_geometry_folds(instances)
    if not isinstance(bank_id, str) or Path(bank_id).name != bank_id or bank_id in ('', '.', '..'):
        raise ValueError('bank_id debe ser un nombre de carpeta simple.')
    _check_hash(bank_sha256, 'bank_sha256')
    _check_hash(evaluator_sha256, 'evaluator_sha256')
    if p.file_hash(p.__file__) != evaluator_sha256:
        raise ValueError('El evaluador activo no coincide con el banco sellado.')
    package = Path(__file__).parent
    source_hashes = {name: p.file_hash(package / name) for name in
                     ('supervised_data.py', 'metaheuristics.py', 'baselines.py', 'search.py', 'experiments.py')}
    source_hashes['pilot.py'] = evaluator_sha256
    if cfg['fallback']['enabled']:
        source_hashes['teacher_feasibility.py'] = p.file_hash(package / 'teacher_feasibility.py')
    protocol = {'schema_version': SCHEMA, 'role': 'development_geometry_crossfit',
                'bank_id': bank_id, 'bank_manifest_sha256': bank_sha256,
                'evaluator_sha256': evaluator_sha256, 'instances': registry,
                'traffic': asdict(spec), 'config': cfg, 'source_hashes': source_hashes,
                'objective_scope': OBJECTIVE, 'folds': folds,
                'labels': 'all_distinct_eligible_heuristic_final_incumbents',
                'fallback_trigger': 'zero_eligible_base_labels_for_instance_after_all_aco_tabu_attempts',
                'fallback_seed_policy': 'all_predeclared_seeds_even_after_success',
                'fallback_objective': 'feasibility_only_not_cost_optimality',
                'independent_unit': 'geometry_group_sha256'}
    corpus_id = p.digest(protocol)
    root = Path(work_root) / 'teacher_corpora' / bank_id / corpus_id
    remote = Path(drive_root) / 'teacher_corpora' / bank_id / corpus_id
    if root.resolve() == remote.resolve():
        raise ValueError('Trabajo local y Drive requieren directorios distintos.')
    print(f'TEACHER_CORPUS_ID = {corpus_id}\nDrive: {remote}', flush=True)
    if exp._recover(root / 'meta', remote / 'meta'):
        if exp.read_json(root / 'meta/protocol.json') != protocol:
            raise ValueError('Protocolo de profesores distinto del checkpoint.')
    else:
        exp.write_json(root / 'meta/protocol.json', protocol)
        exp.write_json(root / 'meta/identity.json', {'corpus_id': corpus_id,
                       'created_utc': datetime.now(timezone.utc).isoformat()})
        exp.write_json(root / 'meta/environment.json', p.environment_report())
        code_dir = root / 'meta/code'
        code_dir.mkdir()
        for name in source_hashes:
            shutil.copy2(Path(p.__file__) if name == 'pilot.py' else package / name, code_dir / name)
        exp._commit(root / 'meta', remote / 'meta')
    attempts, labels, seen, inventory = [], [], set(), {'meta': p.verify_bundle(root / 'meta')}
    by_sha = {x.sha: x for x in instances}
    def collect(row, method, seed, budget):
        x = by_sha[row['instance_sha256']]
        key = f'attempts/{x.sha}/{method}/seed_{seed}'
        identity = {**row, 'corpus_id': corpus_id, 'bank_id': bank_id,
                    'bank_manifest_sha256': bank_sha256,
                    'evaluator_sha256': evaluator_sha256,
                    'teacher_method': method, 'teacher_seed': seed,
                    'search_budget_seconds': budget, 'attempt_key': key}
        directory = root / key
        recovered = exp._recover(directory, remote / key)
        if not recovered:
            print(f'Profesor {method} · {x.instance_id} · seed={seed} · presupuesto={budget:g}s', flush=True)
            interrupt = _run_attempt(x, spec, cfg, method, seed, identity, directory)
            exp._commit(directory, remote / key)
            if interrupt is not None:
                raise interrupt
        attempt = exp.read_json(directory / 'attempt.json')
        if any(attempt.get(k) != v for k, v in identity.items()):
            raise ValueError('Identidad de intento incompatible con el corpus.')
        attempts.append(attempt)
        inventory[key] = p.verify_bundle(directory)
        if p.verify_bundle(remote / key) != inventory[key]:
            raise ValueError('Intento en Drive distinto del local.')
        validation = attempt['label_validation']
        print(f"  {'recuperado' if recovered else 'terminado'} · n={x.n} · {method} · seed={seed} · "
              f"estado={attempt['status']} · {validation['reason']} · "
              f"violación={validation.get('search_violation_minutes')} · "
              f"búsqueda={attempt['teacher_search_seconds']:.2f}s · "
              f"terminación={attempt.get('solver_termination')}", flush=True)
        if attempt.get('solver_error') or validation.get('error'):
            print('  error:', attempt.get('solver_error') or validation.get('error'), flush=True)
        label_key = (x.sha, attempt['routes_sha256'])
        if validation['eligible_label'] and label_key not in seen:
            seen.add(label_key)
            labels.append({**row, 'routes_sha256': attempt['routes_sha256'],
                           'attempt_key': key, 'teacher_method': method, 'teacher_seed': seed,
                           'objective_minutes': validation['search_objective_minutes']})

    for row in registry:
        for method in ('aco', 'tabu'):
            for seed in cfg['seeds']:
                collect(row, method, seed, cfg['seconds_per_method'])
        if cfg['fallback']['enabled'] and not any(
                label['instance_sha256'] == row['instance_sha256'] for label in labels):
            print(f"Respaldo independiente CP-SAT · n={row['n_customers']} · "
                  'sin etiquetas ACO/Tabú; no se leen certificados.', flush=True)
            for seed in cfg['fallback']['seeds']:
                collect(row, 'cp_sat_feasibility', seed, cfg['fallback']['seconds_per_attempt'])
    groups = {r['geometry_group_sha256'] for r in registry}
    labelled_instances = {label['instance_sha256'] for label in labels}
    missing_instances = [r for r in registry if r['instance_sha256'] not in labelled_instances]
    missing = sorted({r['geometry_group_sha256'] for r in missing_instances})
    costs = _cost_summary(attempts, groups)
    index = {'corpus_id': corpus_id, 'schema_version': SCHEMA,
             'status': 'no_labels_for_geometry' if missing else 'complete',
             'missing_geometry_groups': missing, 'protocol_sha256': p.digest(protocol),
             'missing_instance_sha256': [r['instance_sha256'] for r in missing_instances],
             'instances': registry, 'labels': labels, 'attempts': len(attempts),
             'teacher_costs_by_geometry': costs, 'bundles': inventory,
             'label_kind': 'heuristic_not_proven_optimal',
             'ready_for_confirmatory_comparison': False}
    if exp._recover(root / 'index', remote / 'index'):
        if exp.read_json(root / 'index/corpus.json') != index:
            raise ValueError('Índice de corpus distinto de los intentos recuperados.')
    else:
        exp.write_json(root / 'index/corpus.json', index)
        # Una fila por intento, incluidos errores, duplicados y rutas no elegibles.
        export_rows = [{**{k: v for k, v in a.items() if k != 'label_validation'},
                        **a['label_validation']} for a in attempts]
        exp.write_csv(root / 'index/teacher_attempts.csv', export_rows)
        exp.write_csv(root / 'index/teacher_costs_by_geometry.csv', costs)
        exp.write_csv(root / 'index/labels.csv', labels,
                      None if labels else ['instance_sha256', 'geometry_group_sha256', 'attempt_key'])
        exp.write_json(root / 'index/folds.json', folds)
        exp._commit(root / 'index', remote / 'index')
    if _registry(instances) != registry:
        raise AssertionError('El registro del banco cambió durante los profesores.')
    if missing:
        raise RuntimeError('Corpus sin etiquetas válidas para geometrías: ' + ', '.join(missing)
                           + '. Instancias: ' + ', '.join(str(r['n_customers']) for r in missing_instances)
                           + f'. Intentos y costos conservados en {remote}. '
                             'Revisar index/teacher_attempts.csv (reason, solver_termination, solver_error). '
                             'No se remuestrea ni se reutilizan certificados.')
    return root


def load_fold_examples(corpus_dir, instances, fold):
    """Retorna (train, validation, provenance), sin devolver rutas heldout.

    La verificación de integridad contempla todos los paquetes, pero sólo se
    abren result.json de grupos elegibles para entrenamiento o validación.
    """
    root = Path(corpus_dir)
    instances = list(instances)
    validate_fold(fold, instances)
    p.verify_bundle(root / 'index')
    meta_hash = p.verify_bundle(root / 'meta')
    index = exp.read_json(root / 'index/corpus.json')
    protocol = exp.read_json(root / 'meta/protocol.json')
    registry = _registry(instances)
    if (index.get('status') != 'complete' or index.get('instances') != registry
            or protocol.get('instances') != registry
            or index.get('corpus_id') != p.digest(protocol)
            or index.get('protocol_sha256') != p.digest(protocol)
            or index.get('bundles', {}).get('meta') != meta_hash
            or fold not in protocol.get('folds', [])):
        raise ValueError('Corpus incompleto, registro incompatible o fold ajeno al protocolo.')
    if protocol['evaluator_sha256'] != p.file_hash(p.__file__):
        raise ValueError('Evaluador del corpus distinto del módulo activo.')
    for key, expected in index['bundles'].items():
        relative = Path(key)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Ruta de paquete inválida en corpus.')
        if p.verify_bundle(root / relative) != expected:
            raise ValueError('Paquete de profesores modificado.')
    by_sha = {x.sha: x for x in instances}
    groups = {'train': set(fold['train_geometry_groups']),
              'validation': set(fold['validation_geometry_groups'])}
    examples = {'train': [], 'validation': []}
    for label in index['labels']:
        role = next((role for role, selected in groups.items()
                     if label['geometry_group_sha256'] in selected), None)
        if role is None:
            continue
        key = label['attempt_key']
        if key not in index['bundles']:
            raise ValueError('Etiqueta sin intento sellado.')
        attempt = exp.read_json(root / key / 'attempt.json')
        result = exp.read_json(root / key / 'result.json')
        x = by_sha.get(label['instance_sha256'])
        if (x is None or label['instance_id'] != x.instance_id
                or label['geometry_group_sha256'] != x.provenance['geometry_group_sha256']
                or any(attempt.get(k) != label[k] for k in
                       ('instance_sha256', 'geometry_group_sha256', 'attempt_key',
                        'teacher_method', 'teacher_seed', 'routes_sha256'))
                or attempt['label_validation']['eligible_label'] is not True
                or p.digest(result['routes']) != label['routes_sha256']):
            raise ValueError('Identidad o validación de etiqueta incompatible.')
        p.validate_routes(x, result['routes'])
        examples[role].append({'instance_id': x.instance_id, 'instance_sha256': x.sha,
                               'geometry_group_sha256': label['geometry_group_sha256'],
                               'routes': result['routes'], 'teacher': {
                                   'method': label['teacher_method'], 'seed': label['teacher_seed'],
                                   'attempt_key': key, 'objective_minutes': label['objective_minutes'],
                                   'label_kind': 'heuristic_not_proven_optimal'}})
    for role in groups:
        if {e['geometry_group_sha256'] for e in examples[role]} != groups[role]:
            raise ValueError(f'Faltan etiquetas de una geometría del rol {role}.')
        expected_instances = {x.sha for x in instances
                              if x.provenance['geometry_group_sha256'] in groups[role]}
        if {e['instance_sha256'] for e in examples[role]} != expected_instances:
            raise ValueError(f'Faltan etiquetas de una instancia del rol {role}.')
    costs = index['teacher_costs_by_geometry']
    selected = groups['train'] | groups['validation']
    provenance = {'corpus_id': index['corpus_id'], 'fold_id': fold['fold_id'],
                  'bank_id': protocol['bank_id'],
                  'bank_manifest_sha256': protocol['bank_manifest_sha256'],
                  'evaluator_sha256': protocol['evaluator_sha256'],
                  'teacher_costs_by_geometry': costs,
                  'selected_teacher_costs': {key: sum(c[key] for c in costs
                                                     if c['geometry_group_sha256'] in selected)
                                             for key in _COSTS},
                  **{f'{role}_teacher_costs': {
                      key: sum(c[key] for c in costs if c['geometry_group_sha256'] in group_set)
                      for key in _COSTS} for role, group_set in groups.items()},
                  'all_teacher_costs': {key: sum(c[key] for c in costs) for key in _COSTS},
                  'train_examples': len(examples['train']),
                  'validation_examples': len(examples['validation']),
                  'heldout_labels_exposed': False,
                  'independent_unit': 'geometry_group_sha256',
                  'role': 'development_geometry_crossfit'}
    return examples['train'], examples['validation'], provenance


In [ ]:
%%writefile svrplab/supervised.py
"""03: imitación supervisada por geometría, checkpoints y adaptador del runner.

Sólo Colab. Ningún escenario evaluation interviene en etiquetas, gradientes o
selección de checkpoint. El piloto no es una prueba final de generalización.
"""
from __future__ import annotations

import math
import random
import time
import uuid
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch

from . import experiments as e
from . import pilot as p
from .attention import AttentionPolicy, teacher_loss, solve_attention
from .search import SearchProblem
from .supervised_data import make_geometry_folds, validate_fold, load_fold_examples


def require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Entrenamiento y experimentación exclusivamente en Google Colab.') from exc


def validate_training_config(config):
    if (config['schema_version'] != 'supervised-03-v1' or config['role'] != 'development_only'
            or config['split_strategy'] != 'leave_one_geometry_out_with_inner_validation'):
        raise ValueError('Configuración supervisada incompatible.')
    settings = config['training']
    for name in ['epochs', 'steps_per_epoch', 'gradient_accumulation', 'patience']:
        if type(settings[name]) is not int or settings[name] < 1:
            raise ValueError(f'{name} debe ser entero positivo.')
    for name in ['learning_rate', 'gradient_clip_norm', 'seconds_per_model']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f'{name} debe ser finito y positivo.')
    for name in ['weight_decay', 'min_delta']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f'{name} debe ser finito y no negativo.')
    for name in ['permute_nodes', 'deterministic_algorithms']:
        if type(settings[name]) is not bool:
            raise ValueError(f'{name} debe ser booleano.')
    return True


def synchronize(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)


def choose_device(requested='auto'):
    if requested == 'auto':
        requested = 'cuda' if torch.cuda.is_available() else 'cpu'
    device = torch.device(requested)
    if device.type not in ('cpu', 'cuda') or (device.type == 'cuda' and not torch.cuda.is_available()):
        raise ValueError('Dispositivo no disponible: usar cpu, cuda o auto.')
    return device


def _source_hashes():
    directory = Path(__file__).parent
    return {name: p.file_hash(directory / name) for name in
            ('supervised.py', 'supervised_data.py', 'attention.py', 'search.py', 'metaheuristics.py', 'baselines.py')}


def _group_examples(examples, allowed):
    groups = {group: [] for group in allowed}
    for example in examples:
        group = example['geometry_group_sha256']
        if group not in groups:
            raise ValueError('Una etiqueta pertenece a una geometría no autorizada para este split.')
        groups[group].append(example)
    if not groups or any(not values for values in groups.values()):
        raise ValueError('Faltan etiquetas para alguna geometría del split.')
    return groups


def validation_nll(model, problems, examples, groups, device):
    """Promedio de NLL por decisión, luego etiqueta y geometría; sin heldout."""
    grouped = _group_examples(examples, groups)
    model.eval()
    losses = {}
    with torch.no_grad():
        for group, items in grouped.items():
            values = []
            for example in items:
                loss, _ = teacher_loss(model, problems[example['instance_sha256']],
                                       example['routes'], device)
                value = float(loss.detach().cpu())
                if not math.isfinite(value):
                    raise ValueError('NLL de validación no finita.')
                values.append(value)
            losses[group] = float(np.mean(values))
    return float(np.mean(list(losses.values()))), losses


def _cpu_state(model):
    return {key: tensor.detach().cpu().clone() for key, tensor in model.state_dict().items()}


def _load_checkpoint(path):
    # Sólo bundles propios, previamente verificados; evita pickle arbitrario.
    return torch.load(path, map_location='cpu', weights_only=True)


def _runtime_signature(device):
    return {'torch': torch.__version__, 'numpy': np.__version__, 'device_type': torch.device(device).type,
            'cuda': torch.version.cuda,
            'device_name': torch.cuda.get_device_name(device) if torch.device(device).type == 'cuda' else 'cpu'}


def train_fold(instances, spec, config, seed, fold, corpus, directory, remote, device):
    require_colab()
    validate_training_config(config)
    if type(seed) is not int or seed < 0:
        raise ValueError('La semilla de entrenamiento debe ser un entero no negativo.')
    validate_fold(fold, instances)
    directory, remote = Path(directory), Path(remote)
    device = choose_device(str(device))
    started = time.perf_counter()
    train, validation, provenance = load_fold_examples(corpus, instances, fold)
    grouped = _group_examples(train, fold['train_geometry_groups'])
    _group_examples(validation, fold['validation_geometry_groups'])
    identity = {'schema_version': config['schema_version'], 'fold': fold, 'seed': seed,
                'config': config, 'source_hashes': _source_hashes(),
                'corpus_id': provenance['corpus_id'], 'traffic': asdict(spec),
                'train_examples_sha256': p.digest(train), 'validation_examples_sha256': p.digest(validation)}
    if e._recover(directory / 'meta', remote / 'meta'):
        if e.read_json(directory / 'meta/identity.json') != identity:
            raise ValueError('Checkpoint de entrenamiento incompatible con corpus, código, fold o parámetros.')
    else:
        e.write_json(directory / 'meta/identity.json', identity)
        e.write_json(directory / 'meta/provenance.json', provenance)
        e.write_json(directory / 'meta/environment.json', p.environment_report())
        e._commit(directory / 'meta', remote / 'meta')
    if e._recover(directory / 'final', remote / 'final'):
        if e.read_json(directory / 'final/summary.json')['identity_sha256'] != p.digest(identity):
            raise ValueError('Modelo final de otro entrenamiento.')
        return directory / 'final'
    instances_by_sha = {x.sha: x for x in instances}
    for role, examples in [('train_geometry_groups', train), ('validation_geometry_groups', validation)]:
        for example in examples:
            instance = instances_by_sha.get(example['instance_sha256'])
            if (instance is None or instance.provenance['geometry_group_sha256'] != example['geometry_group_sha256']
                    or instance.provenance['geometry_group_sha256'] not in fold[role]):
                raise ValueError('Etiqueta fuera del registro o del rol de geometría autorizado.')
    selected = {ex['instance_sha256'] for ex in train + validation}
    problems = {sha: SearchProblem(instances_by_sha[sha], spec, config['teacher']['time_scale']) for sha in selected}
    for example in train + validation:
        if problems[example['instance_sha256']].score(example['routes'])[0] > 1e-9:
            raise ValueError('Etiqueta sin factibilidad conservadora.')
    settings = config['training']
    derived_seed = p.seed_for('supervised-03', fold['fold_id'], seed)
    random.seed(derived_seed)
    torch.manual_seed(derived_seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(derived_seed)
    torch.use_deterministic_algorithms(settings['deterministic_algorithms'])
    torch.backends.cudnn.benchmark = False
    rng = np.random.default_rng(derived_seed)
    model = AttentionPolicy(config['model']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['learning_rate'],
                                 weight_decay=settings['weight_decay'])
    best_state, best_loss, best_epoch, stale = None, math.inf, 0, 0
    history, elapsed_total, validation_total, last_epoch, terminal = [], 0.0, 0.0, 0, None
    checkpoint_numbers = set()
    for location in (directory, remote):
        if location.exists():
            checkpoint_numbers.update(int(child.name[6:]) for child in location.glob('epoch_*')
                                      if child.name[6:].isdigit() and (child / 'CHECKSUMS.json').exists())
    if checkpoint_numbers and sorted(checkpoint_numbers) != list(range(1, max(checkpoint_numbers) + 1)):
        raise ValueError('Faltan épocas selladas en la secuencia de checkpoints.')
    for epoch in sorted(checkpoint_numbers):
        if not e._recover(directory / f'epoch_{epoch:04d}', remote / f'epoch_{epoch:04d}'):
            raise ValueError('Checkpoint de época no recuperable.')
    if checkpoint_numbers:
        last_epoch = max(checkpoint_numbers)
        latest = directory / f'epoch_{last_epoch:04d}'
        progress = e.read_json(latest / 'progress.json')
        if (progress['identity_sha256'] != p.digest(identity)
                or progress['runtime_signature'] != _runtime_signature(device)):
            raise ValueError('No se reanuda entrenamiento parcial con otro entorno/fold. Use el entorno original o una configuración nueva.')
        state = _load_checkpoint(latest / 'state.pt')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        best_state = state['best_model']
        best_loss, best_epoch, stale = progress['best_validation_nll'], progress['best_epoch'], progress['stale_epochs']
        history = progress['history']
        elapsed_total, validation_total = progress['training_seconds'], progress['validation_seconds']
        rng.bit_generator.state = progress['numpy_generator_state']
        random.setstate((progress['python_rng_state'][0], tuple(progress['python_rng_state'][1]), progress['python_rng_state'][2]))
        torch.set_rng_state(state['torch_rng'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(state['cuda_rng'])
        terminal = progress['stop_reason']
    synchronize(device)
    elapsed_total += time.perf_counter() - started
    groups = sorted(grouped)
    for epoch in range(last_epoch + 1, settings['epochs'] + 1):
        if terminal:
            break
        if elapsed_total >= settings['seconds_per_model']:
            terminal = 'time_limit'
            break
        epoch_started = time.perf_counter()
        model.train()
        values, updates, decisions = [], 0, 0
        for offset in range(0, settings['steps_per_epoch'], settings['gradient_accumulation']):
            if elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                break
            batch_size = min(settings['gradient_accumulation'], settings['steps_per_epoch'] - offset)
            optimizer.zero_grad(set_to_none=True)
            for _ in range(batch_size):
                group = groups[int(rng.integers(len(groups)))]
                items = grouped[group]
                example = items[int(rng.integers(len(items)))]
                problem = problems[example['instance_sha256']]
                order = rng.permutation(np.arange(1, problem.n + 1)).tolist() if settings['permute_nodes'] else None
                loss, stats = teacher_loss(model, problem, example['routes'], device, node_order=order)
                if not bool(torch.isfinite(loss).item()):
                    raise ValueError('Pérdida de entrenamiento no finita.')
                (loss / batch_size).backward()
                values.append(float(loss.detach().cpu()))
                decisions += stats['decisions']
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), settings['gradient_clip_norm'], error_if_nonfinite=True)
            if not math.isfinite(float(norm)):
                raise ValueError('Gradientes no finitos.')
            optimizer.step(); updates += 1
        if not values:
            elapsed_total += time.perf_counter() - epoch_started
            terminal = 'time_limit'
            break
        synchronize(device)
        val_started = time.perf_counter()
        val_loss, group_losses = validation_nll(model, problems, validation, fold['validation_geometry_groups'], device)
        synchronize(device)
        val_seconds = time.perf_counter() - val_started
        if val_loss < best_loss - settings['min_delta']:
            best_loss, best_epoch, best_state, stale = val_loss, epoch, _cpu_state(model), 0
        else:
            stale += 1
        elapsed_total += time.perf_counter() - epoch_started
        validation_total += val_seconds
        terminal = ('patience' if stale >= settings['patience'] else
                    'time_limit' if elapsed_total >= settings['seconds_per_model'] else
                    'epochs' if epoch == settings['epochs'] else None)
        history.append({'epoch': epoch, 'train_nll': float(np.mean(values)), 'validation_nll': val_loss,
                        'best_validation_nll': best_loss, 'best_epoch': best_epoch,
                        'examples_sampled': len(values), 'decisions': decisions, 'optimizer_updates': updates,
                        'training_seconds': elapsed_total, 'validation_seconds': validation_total})
        target = directory / f'epoch_{epoch:04d}'
        # Preservar una escritura interrumpida de esta época antes de repetirla.
        if e._recover(target, remote / target.name):
            raise ValueError('La época nueva ya existe; no sobrescribir un checkpoint.')
        target.mkdir(parents=True, exist_ok=True)
        torch.save({'model': _cpu_state(model), 'optimizer': optimizer.state_dict(), 'best_model': best_state,
                    'torch_rng': torch.get_rng_state(),
                    'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}, target / 'state.pt')
        e.write_json(target / 'progress.json', {'epoch': epoch, 'identity_sha256': p.digest(identity),
                     'runtime_signature': _runtime_signature(device), 'best_validation_nll': best_loss,
                     'best_epoch': best_epoch, 'stale_epochs': stale, 'history': history,
                     'training_seconds': elapsed_total, 'validation_seconds': validation_total,
                     'numpy_generator_state': rng.bit_generator.state, 'python_rng_state': random.getstate(),
                     'validation_by_geometry': group_losses, 'stop_reason': terminal})
        e.write_json(target / 'environment.json', p.environment_report())
        e._commit(target, remote / target.name)
        print(f'{fold["fold_id"]} seed={seed} epoch={epoch}: train={history[-1]["train_nll"]:.5f}, '
              f'val={val_loss:.5f}, best={best_epoch}', flush=True)
    if best_state is None:
        raise RuntimeError('No hay checkpoint entrenado y seleccionado; no evaluar pesos aleatorios.')
    final = directory / 'final'
    final.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, final / 'best.pt')
    summary = {'identity_sha256': p.digest(identity), 'fold': fold, 'training_seed': seed,
               'training_seconds': elapsed_total, 'validation_seconds': validation_total,
               'training_budget_seconds': settings['seconds_per_model'],
               'training_budget_overrun_seconds': max(0., elapsed_total - settings['seconds_per_model']),
               'training_time_scope': 'setup_and_epoch_compute_validation_excludes_epoch_and_final_checkpoint_persistence',
               'teacher_seconds': provenance['selected_teacher_costs']['teacher_wall_seconds'],
               'best_epoch': best_epoch, 'best_validation_nll': best_loss, 'epochs_completed': len(history),
               'stop_reason': terminal, 'model_config': config['model'], 'traffic': asdict(spec),
               'corpus_id': provenance['corpus_id'], 'checkpoint_selection': 'inner_validation_teacher_nll',
               'training_scope': 'development_geometry_crossfit_surrogate_imitation',
               'train_examples': len(train), 'validation_examples': len(validation),
               'train_geometries': len(grouped), 'heldout_used_for_training_or_selection': False,
               'model_checkpoint_sha256': p.file_hash(final / 'best.pt')}
    e.write_json(final / 'summary.json', summary)
    e.write_json(final / 'history.json', history)
    e.write_csv(final / 'history.csv', history)
    save_learning_plot(history, final / 'learning.png', fold['fold_id'], seed)
    e._commit(final, remote / 'final')
    del model, optimizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return final


def save_learning_plot(history, path, fold_id, seed):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    for metric in ['train_nll', 'validation_nll']:
        ax.plot([row['epoch'] for row in history], [row[metric] for row in history], label=metric)
    ax.set(title=f'{fold_id} · seed={seed} · NLL de imitación', xlabel='Época', ylabel='NLL por decisión')
    ax.legend(); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def train_crossfit(instances, spec, config, seeds, corpus, work_root, drive_root, bank_id,
                   bank_sha256, evaluator_sha256, device='auto'):
    require_colab()
    validate_training_config(config)
    if (not seeds or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de entrenamiento vacías, repetidas o inválidas.')
    if evaluator_sha256 != p.file_hash(p.__file__):
        raise ValueError('Evaluador de entrenamiento distinto del módulo sellado activo.')
    device = choose_device(device)
    folds = make_geometry_folds(instances)
    identity = {'config': config, 'seeds': seeds, 'folds': folds,
                'corpus_id': Path(corpus).name, 'bank_sha256': bank_sha256,
                'evaluator_sha256': evaluator_sha256, 'source_hashes': _source_hashes()}
    training_id = 'attention03_' + p.digest(identity)[:20]
    root = Path(work_root) / 'supervised_training' / bank_id / training_id
    remote = Path(drive_root) / 'supervised_training' / bank_id / training_id
    print('TRAINING_ID =', training_id, '| dispositivo:', device, flush=True)
    if e._recover(root / 'meta', remote / 'meta'):
        if e.read_json(root / 'meta/identity.json') != identity:
            raise ValueError('Identidad de entrenamiento inconsistente.')
    else:
        e.write_json(root / 'meta/identity.json', identity)
        e.write_json(root / 'meta/folds.json', folds)
        for name in _source_hashes():
            target = root / 'meta/code' / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(Path(__file__).with_name(name).read_bytes())
        e._commit(root / 'meta', remote / 'meta')
    models = []
    for fold in folds:
        for seed in seeds:
            relative = Path(fold['fold_id']) / f'seed_{seed}'
            attempt_started = time.perf_counter()
            try:
                final = train_fold(instances, spec, config, seed, fold, corpus,
                                   root / relative, remote / relative, device)
            except Exception as exc:
                failure = root / 'failures' / (fold['fold_id'] + f'_seed_{seed}_' + uuid.uuid4().hex[:8])
                e.write_json(failure / 'failure.json', {'fold': fold, 'seed': seed, 'training_id': training_id,
                             'error_type': type(exc).__name__, 'error': str(exc),
                             'observed_call_wall_seconds_including_io': time.perf_counter() - attempt_started,
                             'completed_checkpoints_preserved': True})
                e.write_json(failure / 'environment.json', p.environment_report())
                e._commit(failure, remote / 'failures' / failure.name)
                raise
            models.append({**e.read_json(final / 'summary.json'), 'relative_directory': str(relative / 'final'),
                           'bundle_sha256': p.verify_bundle(final)})
    # Toda selección terminó antes de que el notebook invoque evaluate sobre heldout.
    if not e._recover(root / 'index', remote / 'index'):
        _, _, provenance = load_fold_examples(corpus, instances, folds[0])
        e.write_json(root / 'index/models.json', models)
        e.write_json(root / 'index/offline_costs.json', {'offline_cost_id': training_id,
                     'teacher_corpus_id': provenance['corpus_id'],
                     'teacher_costs_unique': provenance['all_teacher_costs'],
                     'training_seconds_unique': sum(row['training_seconds'] for row in models),
                     'validation_seconds_included_in_training': sum(row['validation_seconds'] for row in models),
                     'models_trained': len(models), 'geometry_count': len(folds),
                     'summary_times_must_not_be_summed_across_policies_or_scenarios': True})
        e.write_csv(root / 'index/training_summary.csv', [{k: v for k, v in row.items()
                                                         if not isinstance(v, (dict, list))} for row in models])
        e._commit(root / 'index', remote / 'index')
    if e.read_json(root / 'index/models.json') != models:
        raise ValueError('Inventario de modelos distinto de los checkpoints.')
    return root


def make_solver(training_root, instances, spec, device='auto', expected_bank_sha256=None):
    """Carga sólo el modelo que dejó fuera la geometría de la instancia solicitada."""
    root, device = Path(training_root), choose_device(device)
    p.verify_bundle(root / 'meta')
    identity = e.read_json(root / 'meta/identity.json')
    if expected_bank_sha256 is not None and identity['bank_sha256'] != expected_bank_sha256:
        raise ValueError('Entrenamiento vinculado a otro banco.')
    p.verify_bundle(root / 'index')
    index = e.read_json(root / 'index/models.json')
    registry, provenance = {}, {}
    for model in index:
        final = root / model['relative_directory']
        if p.verify_bundle(final) != model['bundle_sha256'] or p.file_hash(final / 'best.pt') != model['model_checkpoint_sha256']:
            raise ValueError('Checkpoint de modelo modificado.')
        if model['traffic'] != asdict(spec):
            raise ValueError('Modelo entrenado con otro tráfico conocido.')
        fold = model['fold']
        validate_fold(fold, instances)
        for x in instances:
            if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']:
                continue
            key = f'{x.sha}:{model["training_seed"]}'
            if key in registry:
                raise ValueError('Dos modelos compiten por una observación heldout.')
            registry[key] = (final, model)
            provenance[key] = {key_: model[key_] for key_ in ['training_seed', 'training_seconds',
                'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256', 'checkpoint_selection', 'training_scope']}
            provenance[key].update(training_id=root.name, fold_id=fold['fold_id'],
                train_geometry_groups=fold['train_geometry_groups'],
                validation_geometry_groups=fold['validation_geometry_groups'],
                heldout_geometry_groups=fold['heldout_geometry_groups'], offline_cost_id=root.name,
                offline_manifest_sha256=p.verify_bundle(root / 'index'))

    def solver(problem, seed, config, output_dir):
        started = time.perf_counter()
        key = f'{problem.x.sha}:{seed}'
        if key not in registry:
            raise ValueError('No existe modelo fuera de geometría para esta instancia/semilla.')
        final, record = registry[key]
        if p.file_hash(final / 'best.pt') != record['model_checkpoint_sha256']:
            raise ValueError('Checkpoint cambió antes de inferencia.')
        model = AttentionPolicy(record['model_config']).to(device)
        model.load_state_dict(_load_checkpoint(final / 'best.pt'))
        model.eval(); synchronize(device)
        loading = time.perf_counter() - started
        remaining = config['seconds'] - loading
        try:
            if remaining <= 0:
                result = problem.result(None, 'time_limit_model_load', loading, [])
            else:
                result = solve_attention(problem, seed, {**config, 'seconds': remaining}, model, device)
            result['metadata'].update(model_load_seconds=loading,
                                      model_checkpoint_sha256=record['model_checkpoint_sha256'],
                                      fold_id=record['fold']['fold_id'])
            synchronize(device)
            result['solve_seconds'] = time.perf_counter() - started
            return result
        finally:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    return solver, {'am_supervised': provenance}


In [ ]:
import importlib
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
for name in list(sys.modules):
    if name == 'svrplab' or name.startswith('svrplab.'):
        del sys.modules[name]
importlib.invalidate_caches()
import torch
import pandas as pd
from IPython.display import display, Image
from svrplab import pilot as p
from svrplab import experiments as exp
from svrplab import supervised_data as sd
from svrplab import supervised as sup
exp.validate_config(CONFIG)
sup.validate_training_config(SUPERVISED_CONFIG)
if CONFIG['time_scale'] != SUPERVISED_CONFIG['teacher']['time_scale']:
    raise ValueError('Profesores e inferencia deben usar la misma matriz conservadora.')
instances = sorted(p.load_bank(BANK), key=lambda x: x.n)
bank_config = json.loads((BANK / 'config.json').read_text())
if bank_config['protocol_version'] not in SUPPORTED_BANK_PROTOCOLS:
    raise ValueError('Protocolo de banco incompatible.')
TRAFFIC = p.Traffic(**bank_config['traffic'])
BANK_SHA = p.verify_bundle(BANK)
EVALUATOR_SHA = p.file_hash(p.__file__)
DEVICE = sup.choose_device(TRAIN_DEVICE)
print('PyTorch:', torch.__version__, '| dispositivo:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(DEVICE), '| CUDA:', torch.version.cuda)
else:
    print('CPU: entrenamiento permitido en Colab, puede ser más lento.')
folds = sd.make_geometry_folds(instances)
fold_rows = [{'fold_id': fold['fold_id'], 'role': role, 'n_customers': x.n,
              'instance_sha256': x.sha, 'geometry_group_sha256': x.provenance['geometry_group_sha256']}
             for fold in folds for role in ('train_geometry_groups','validation_geometry_groups','heldout_geometry_groups')
             for x in instances if x.provenance['geometry_group_sha256'] in fold[role]]
display(pd.DataFrame(fold_rows))
print('Banco aceptado para desarrollo; folds fijados antes de etiquetar y entrenar.')


## 4. Pruebas funcionales, únicamente en Colab


In [ ]:
%%writefile tests/test_baselines.py
"""Conventional baselines: run exclusively in Colab, never locally."""
from pathlib import Path
import sys
import time

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import baselines as b
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, capacities=(3, 3, 3), demands=(0, 0, 1, 1), windows=None):
    n = len(demands) - 1
    x = freeze_instance({
        'instance_id': 'baseline-manual-fixture', 'depot_indices': [0],
        'locations': [[j, 0] for j in range(n + 1)],
        'demands': list(demands), 'capacities': list(capacities),
        'windows': windows or [[0, 100]] * (n + 1),
        'appear_times': [0] * (n + 1), 'provenance': {'kind': 'manual_test_fixture'},
    })
    return SearchProblem(x, Traffic(mode='no_noise'))


def test_edd_fixed_fleet_zero_demand_and_determinism():
    p = problem()
    before = p.x.sha
    a, c = b.solve_edd(p, 11), b.solve_edd(p, 47)
    assert a['status'] == c['status'] == 'feasible'
    assert a['routes'] == c['routes']
    assert validate_routes(p.x, a['routes'])
    assert len(a['routes']) == len(p.capacities)
    assert p.x.sha == before
    assert a['metadata']['seed_affects_search'] is False
    assert a['metadata']['reference_is_bound'] is False


def test_edd_inserts_before_existing_customer_when_append_would_be_late():
    p = problem(capacities=(5,), demands=(0, 1, 1), windows=[[0, 100], [5, 6], [0, 8]])
    # Customer 1 has the earlier closing time, but its opening time imposes
    # waiting. Insert customer 2 before it instead of appending after waiting.
    p.travel = np.array([[0., 1., 1.], [1., 0., 4.], [1., 4., 0.]])
    result = b.solve_edd(p, 11)
    assert result['routes'] == [[2, 1]]
    assert result['status'] == 'feasible'
    assert p.score([[1, 2]])[0] > 0


def test_impossible_window_returns_complete_diagnostic_without_feasibility_claim():
    p = problem(windows=[[0, 100]] + [[0, .01]] * 3)
    result = b.solve_edd(p, 11)
    assert result['status'] == 'conservative_infeasible'
    assert validate_routes(p.x, result['routes'])
    assert result['metadata']['search_violation_minutes'] > 0
    assert result['metadata']['surrogate_optimal'] is False


def test_capacity_dead_end_never_drops_customers_or_adds_vehicles():
    p = problem(capacities=(1,), demands=(0, 0, 1, 1))
    result = b.solve_edd(p, 11)
    assert result['routes'] is None and result['status'] == 'no_solution'
    assert result['metadata']['termination'] == 'capacity_dead_end'


def test_deadline_stops_incomplete_construction_without_returning_partial_routes(monkeypatch):
    clock = [0.]

    def now():
        clock[0] += .1
        return clock[0]

    monkeypatch.setattr(b.time, 'perf_counter', now)
    routes, reason = b.construct_insertion(problem(), np.random.default_rng(11), .25)
    assert routes is None and reason == 'time_limit'


def test_local_search_repairs_lateness_without_extra_vehicle_or_customer_loss():
    p = problem(capacities=(5,), demands=(0, 1, 1), windows=[[0, 100], [5, 6], [0, 8]])
    p.travel = np.array([[0., 1., 1.], [1., 0., 4.], [1., 4., 0.]])
    before = [[1, 2]]
    routes, stats = b.improve_feasible_routes(p, before, np.random.default_rng(11), time.perf_counter() + 30)
    assert before == [[1, 2]]
    assert p.score(routes)[0] == 0.
    assert validate_routes(p.x, routes)
    assert stats['accepted_moves'] >= 1


def test_local_search_improves_travel_and_never_replaces_feasible_with_late():
    p = problem(capacities=(3, 3, 3))
    before = [[1], [2], [3]]
    routes, stats = b.improve_feasible_routes(p, before, np.random.default_rng(11), time.perf_counter() + 30)
    assert p.score(routes) <= p.score(before)
    assert p.score(routes)[0] <= 1e-9
    assert validate_routes(p.x, routes)
    assert stats['evaluated_neighbors'] > 0


def test_local_search_zero_budget_retains_input_incumbent():
    p = problem()
    before = [[1], [2], [3]]
    routes, stats = b.improve_feasible_routes(p, before, np.random.default_rng(11), -1.)
    assert routes == before
    assert stats['accepted_moves'] == stats['evaluated_neighbors'] == 0
    assert stats['termination'] == 'time_limit'


def test_ortools_fixed_fleet_capacity_and_zero_demand_customer():
    pytest.importorskip('ortools')
    p = problem(capacities=(1, 1, 1))
    before = p.x.sha
    result = b.solve_ortools_routing(p, 29, {'seconds': 2.})
    assert result['status'] == 'feasible'
    assert validate_routes(p.x, result['routes'])
    assert len(result['routes']) == 3 and p.x.sha == before
    meta = result['metadata']
    assert not meta['surrogate_optimal'] and not meta['stochastic_optimal']
    assert not meta['reference_is_bound']
    assert meta['budget_includes_construction'] and meta['seed_affects_search'] is False
    assert meta['bound_scope'] == 'no_optimality_bound_from_routing_search'


def test_ortools_tight_windows_use_same_conservative_contract():
    pytest.importorskip('ortools')
    p = problem(capacities=(5,), demands=(0, 1, 1), windows=[[0, 100], [5, 6], [0, 8]])
    p.travel = np.array([[0., 1., 1.], [1., 0., 4.], [1., 4., 0.]])
    result = b.solve_ortools_routing(p, 11, {'seconds': 2.})
    assert result['routes'] == [[2, 1]]
    assert p.score(result['routes'])[0] == 0.


def test_ortools_zero_budget_does_not_import_solver(monkeypatch):
    monkeypatch.setattr(b, '_ortools', lambda: pytest.fail('No solver import without budget.'))
    result = b.solve_ortools_routing(problem(), 11, {'seconds': 0.})
    assert result['routes'] is None and result['metadata']['termination'] == 'time_limit'


def test_solver_import_is_part_of_budget(monkeypatch):
    clock = [0.]
    monkeypatch.setattr(b.time, 'perf_counter', lambda: clock[0])

    def delayed_import():
        clock[0] = 11.
        return object(), object()

    monkeypatch.setattr(b, '_ortools', delayed_import)
    result = b.solve_ortools_routing(problem(), 11, {'seconds': 10.})
    assert result['routes'] is None
    assert result['solve_seconds'] == 11.
    assert result['metadata']['budget_overrun_seconds'] == 1.


def test_integer_grid_cannot_be_loosened_to_find_solution():
    assert b._integer(1.25, 1000) == 1250
    with pytest.raises(ValueError, match='rejilla'):
        b._integer(1.2505, 1000)


def test_ortools_never_relabels_infeasible_edd_as_solver_solution():
    pytest.importorskip('ortools')
    p = problem(windows=[[0, 100]] + [[0, .01]] * 3)
    result = b.solve_ortools_routing(p, 11, {'seconds': 2.})
    assert result['routes'] is None and result['status'] == 'no_solution'
    assert result['metadata']['solution_origin'] == 'none'
    assert result['metadata']['edd_diagnostic_score'][0] > 0
    assert not result['metadata']['returned_paid_edd_fallback']


@pytest.mark.parametrize('solver,config', [
    (b.solve_edd, {'unknown': 1}), (b.solve_edd, {'seconds': float('nan')}),
    (b.solve_ortools_routing, {'seconds': -1}),
    (b.solve_ortools_routing, {'capacity_scale': 0}),
    (b.solve_ortools_routing, {'capacity_scale': True}),
])
def test_invalid_baseline_configuration_is_rejected(solver, config):
    with pytest.raises(ValueError):
        solver(problem(), 11, config)


In [ ]:
%%writefile tests/test_metaheuristics.py
"""Pruebas para ejecutar exclusivamente en Colab; no ejecutarlas localmente."""
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import metaheuristics as mh
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, impossible_window=False, impossible_capacity=False):
    x = freeze_instance({
        'instance_id': 'metaheuristic-manual-fixture', 'depot_indices': [0],
        'locations': [[0, 0], [1, 0], [2, 0], [3, 0], [-2, 0], [-3, 0]],
        'demands': [0, 0, 2, 2, 2, 2],
        'capacities': [3, 3] if impossible_capacity else [4, 4],
        'windows': [[0, 100]] + [[0, .1 if impossible_window else 80]] * 5,
        'appear_times': [0] * 6, 'provenance': {'kind': 'test_fixture'},
    })
    return SearchProblem(x, Traffic())


def small_config(solver, **overrides):
    config = {'seconds': 60, 'max_iterations': 4, 'local_search_seconds': 10., 'local_search_moves': 4}
    config.update({'ants': 4} if solver is mh.solve_aco else
                  {'initial_restarts': 4, 'neighborhood_samples': 30, 'tenure': 3})
    return config | overrides


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_complete_routes_capacity_and_zero_demand_customer(solver):
    p = problem()
    before = p.x.sha
    result = solver(p, 31, small_config(solver))
    assert result['status'] == 'feasible'
    assert validate_routes(p.x, result['routes'])
    assert sorted(j for route in result['routes'] for j in route) == [1, 2, 3, 4, 5]
    assert len(result['routes']) == 2
    assert p.score(result['routes'])[0] <= 1e-9
    assert p.x.sha == before
    assert result['solve_seconds'] >= 0
    assert result['metadata']['budget_includes_construction'] is True
    progress = [(row['violation_minutes'], row['travel_minutes']) for row in result['history']]
    assert progress and all(b <= a for a, b in zip(progress, progress[1:]))


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_impossible_windows_report_violation_without_losing_clients(solver):
    p = problem(impossible_window=True)
    result = solver(p, 19, small_config(solver))
    assert result['status'] == 'conservative_infeasible'
    assert validate_routes(p.x, result['routes'])
    assert p.score(result['routes'])[0] > 0
    # Es el estado de las rutas; no una prueba de infactibilidad del problema.
    assert result['metadata']['nominal_scope'] == 'conservative_upper_mean_matrix_not_stochastic_feasibility'


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_impossible_capacity_returns_explicit_no_solution(solver):
    result = solver(problem(impossible_capacity=True), 3, small_config(solver))
    assert result['routes'] is None
    assert result['status'] == 'no_solution'
    assert result['metadata']['termination'] == 'capacity_screen_failed'


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_iteration_limited_seed_is_reproducible(solver):
    p = problem()
    first = solver(p, 41, small_config(solver))
    second = solver(p, 41, small_config(solver))
    assert first['metadata']['termination'] == second['metadata']['termination'] == 'iteration_limit'
    assert first['routes'] == second['routes']
    assert first['status'] == second['status']
    assert [(r['violation_minutes'], r['travel_minutes']) for r in first['history']] == [
        (r['violation_minutes'], r['travel_minutes']) for r in second['history']]


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_zero_budget_does_not_construct(monkeypatch, solver):
    def forbidden(*args, **kwargs):
        raise AssertionError('La construcción no debe iniciarse sin presupuesto.')
    monkeypatch.setattr(mh, '_construct', forbidden)
    result = solver(problem(), 5, small_config(solver, seconds=0))
    assert result['status'] == 'no_solution' and result['routes'] is None
    assert result['metadata']['termination'] == 'time_limit'


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_construction_consumes_same_budget_and_preserves_completed_incumbent(monkeypatch, solver):
    clock = [0.]
    calls = []

    def construction(*args, **kwargs):
        calls.append(True)
        clock[0] = 11.
        return [[1, 2, 3], [4, 5]], 'complete'

    monkeypatch.setattr(mh.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(mh, '_construct', construction)
    result = solver(problem(), 7, small_config(solver, seconds=10))
    assert len(calls) == 1
    assert result['routes'] == [[1, 2, 3], [4, 5]]
    assert result['metadata']['termination'] == 'time_limit'
    assert result['solve_seconds'] == 11.
    assert result['metadata']['budget_overrun_seconds'] == 1.


@pytest.mark.parametrize('kind,first,second,inverse_first,inverse_second', [
    ('relocate', (0, 1), (1, 1), (1, 1), (0, 1)),
    ('swap', (0, 0), (1, 1), (0, 0), (1, 1)),
    ('2opt', (0, 0), (0, 2), (0, 0), (0, 2)),
    ('2opt_star', (0, 1), (1, 1), (0, 1), (1, 1)),
])
def test_tabu_blocks_actual_inverse_and_respects_aspiration_and_expiry(
        kind, first, second, inverse_first, inverse_second):
    before = [[1, 2, 3], [4, 5]]
    after = mh._apply_move(before, kind, first, second)
    restored = mh._apply_move(after, kind, inverse_first, inverse_second)
    assert restored == before
    removed = mh._arcs(before) - mh._arcs(after)
    inverse_added = mh._arcs(restored) - mh._arcs(after)
    assert removed and removed == inverse_added
    memory = {arc: 8 for arc in removed}
    assert not mh._tabu_admissible(inverse_added, memory, 6, (0., 10.), (0., 10.))
    assert mh._tabu_admissible(inverse_added, memory, 6, (0., 9.), (0., 10.))
    # Menor viaje nunca compensa mayor violación en la regla de aspiración.
    assert not mh._tabu_admissible(inverse_added, memory, 6, (1., 1.), (0., 10.))
    assert mh._tabu_admissible(inverse_added, memory, 8, (0., 10.), (0., 10.))


@pytest.mark.parametrize('solver,config', [
    (mh.solve_aco, {'q0': 1.1}), (mh.solve_aco, {'evaporation': 0}),
    (mh.solve_aco, {'ants': 1.2}), (mh.solve_aco, {'seconds': float('nan')}),
    (mh.solve_tabu, {'tenure': 0}), (mh.solve_tabu, {'initial_restarts': 0}),
    (mh.solve_tabu, {'neighborhood_samples': True}), (mh.solve_tabu, {'unknown': 1}),
])
def test_invalid_configuration_is_rejected(solver, config):
    with pytest.raises(ValueError):
        solver(problem(), 1, config)


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_time_only_mode_does_not_stop_at_legacy_200_iterations(monkeypatch, solver):
    clock = [0.]
    monkeypatch.setattr(mh.time, 'perf_counter', lambda: clock[0])

    def construction(*args, **kwargs):
        clock[0] += 1.
        return [[1, 2, 3], [4, 5]], 'complete'

    def neighbor(*args, **kwargs):
        clock[0] += 1.
        return None

    monkeypatch.setattr(mh, '_construct', construction)
    monkeypatch.setattr(mh, '_random_neighbor', neighbor)
    config = small_config(solver, seconds=205., max_iterations=None, local_search_seconds=0.)
    if solver is mh.solve_aco:
        config['ants'] = 1
    else:
        config.update(initial_restarts=1, neighborhood_samples=1)
    result = solver(problem(), 11, config)
    assert result['metadata']['iterations'] > 200
    assert result['metadata']['termination'] == 'time_limit'
    assert result['metadata']['effective_config']['max_iterations'] is None


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_paid_local_search_is_called_and_cannot_use_extra_budget(monkeypatch, solver):
    clock, calls = [0.], []
    monkeypatch.setattr(mh.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(mh, '_construct', lambda *a, **k: ([[1, 2, 3], [4, 5]], 'complete'))

    def improve(problem, routes, rng, deadline, max_moves):
        calls.append((deadline, max_moves))
        assert deadline <= 10.
        clock[0] = 11.
        return routes, {'evaluated_neighbors': 3, 'accepted_moves': 1}

    monkeypatch.setattr(mh, 'improve_feasible_routes', improve)
    result = solver(problem(), 11, small_config(solver, seconds=10., local_search_seconds=20.))
    assert len(calls) == 1
    assert result['metadata']['budget_includes_repair']
    assert result['metadata']['local_search_evaluated_neighbors'] == 3
    assert result['metadata']['local_search_accepted_moves'] == 1
    assert result['metadata']['termination'] == 'time_limit'
    assert result['solve_seconds'] == 11.


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_both_methods_keep_ordinary_edd_incumbent_when_later_construction_fails(monkeypatch, solver):
    calls = [0]

    def construct(*args, **kwargs):
        calls[0] += 1
        return ([[1, 2, 3], [4, 5]], 'complete') if calls[0] == 1 else (None, 'capacity_dead_end')

    monkeypatch.setattr(mh, '_construct', construct)
    result = solver(problem(), 11, small_config(solver, local_search_seconds=0.))
    assert result['status'] == 'feasible'
    assert result['metadata']['initialization'] == 'paid_edd_feasible_cheapest_insertion'
    assert result['metadata']['capacity_dead_ends'] > 0
    assert result['metadata']['budget_includes_construction']


In [ ]:
%%writefile tests/test_experiments.py
"""Controles del protocolo 01/02. Ejecutar exclusivamente desde Google Colab.

Los casos son manuales: no generan el banco, no buscan certificados y no usan
ninguna ruta testigo del banco como solución de un método.
"""
import copy
import csv
import json
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import pilot as p
from svrplab import experiments as e
from svrplab.search import SearchProblem


def manual_instance(windows=None, capacities=None):
    return p.freeze_instance({
        'instance_id': 'manual_runner', 'depot_indices': [0],
        'locations': [[0, 0], [3, 0], [6, 0]],
        'demands': [0, 0, 1], 'capacities': capacities or [5, 5],
        'windows': windows or [[0, 100], [5, 10], [0, 100]],
        'appear_times': [0, 0, 0], 'provenance': {'kind': 'manual_test'},
    })


def manual_solution(x, routes=None, spec=None):
    routes = [[1, 2], []] if routes is None else routes
    spec = p.Traffic(mode='no_noise') if spec is None else spec
    return {'routes': routes, 'nominal': p.simulate(x, routes, 0, spec, trace=True)}


def csv_rows(path):
    with Path(path).open(newline='', encoding='utf-8') as stream:
        return list(csv.DictReader(stream))


@pytest.mark.parametrize('scenarios', [
    {'train': [1, 2], 'selection': [2], 'evaluation': [4, 5]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, 4]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, True]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, 5.0]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, -1]},
    {'train': [1, 2], 'evaluation': [4, 5]},
])
def test_scenario_contract_rejects_leakage_or_malformed_splits(scenarios):
    with pytest.raises(ValueError):
        e.validate_scenarios(scenarios, {'scenarios': {'train': 2, 'selection': 1, 'evaluation': 2}})


def test_scenario_contract_preserves_large_seeds_and_original_order():
    scenarios = {'train': [2**64 - 1, 7], 'selection': [3], 'evaluation': [2**53 + 1, 1]}
    before = copy.deepcopy(scenarios)
    e.validate_scenarios(scenarios, {'scenarios': {'train': 2, 'selection': 1, 'evaluation': 2}})
    assert scenarios == before


def test_csv_seeds_have_lossless_decimal_roundtrip(tmp_path):
    seeds = [2**53 + 1, 2**63 + 17, 2**64 - 1]
    path = tmp_path / 'seeds.csv'
    e.write_csv(path, [{'scenario_seed': str(seed), 'metric': 1.25} for seed in seeds])
    rows = csv_rows(path)
    assert [row['scenario_seed'] for row in rows] == [str(seed) for seed in seeds]
    assert [int(row['scenario_seed']) for row in rows] == seeds


def test_search_objective_and_clock_have_manual_values():
    x = manual_instance()
    problem = SearchProblem(x, p.Traffic(mode='no_noise'))
    before = x.sha
    stats = problem.route_stats([1, 2], 0)
    # 0->1: 3, espera 2; 1->2: 3; 2->0: 6. Viaje 12, retorno 14.
    assert stats['travel_minutes'] == 12
    assert stats['service_starts'] == [5, 8]
    assert stats['return_minutes'] == 14
    assert problem.score([[1, 2], []]) == (0, 12)
    assert x.sha == before


def test_search_objective_records_lateness_and_return_separately_from_distance():
    x = manual_instance([[0, 8], [0, 2], [0, 4]])
    problem = SearchProblem(x, p.Traffic(mode='no_noise'))
    stats = problem.route_stats([1, 2], 0)
    assert stats['lateness_minutes'] == 3  # (3-2) + (6-4).
    assert stats['depot_overtime_minutes'] == 4  # 12-8.
    assert problem.score([[1, 2], []]) == (7, 12)
    result = problem.result([[1, 2], []], 'time_limit', .1, [])
    assert result['metadata']['conservative_feasible'] is False
    assert result['metadata']['search_violation_minutes'] == 7
    assert result['metadata']['search_objective_minutes'] == 12


def test_search_never_accepts_missing_zero_demand_customer_or_excess_capacity():
    problem = SearchProblem(manual_instance(), p.Traffic(mode='no_noise'))
    with pytest.raises(ValueError):
        problem.score([[2], []])
    problem = SearchProblem(manual_instance(capacities=[.5, .5]), p.Traffic(mode='no_noise'))
    with pytest.raises(ValueError):
        problem.score([[1, 2], []])


@pytest.mark.parametrize('damage', ['metric', 'missing_event', 'invalid_customer', 'invalid_vehicle',
                                    'broken_arc', 'broken_clock'])
def test_trace_qc_rejects_corruption_even_when_aggregate_totals_match(damage):
    x = manual_instance()
    routes = [[1, 2], []]
    spec = p.Traffic(mode='no_noise')
    raw = p.simulate(x, routes, 1, spec)
    trace = p.simulate(x, routes, 1, spec, trace=True)
    if damage == 'metric':
        trace['travel_minutes'] += 1
    elif damage == 'missing_event':
        trace['events'].pop()
    elif damage == 'invalid_customer':
        trace['events'][0]['to'] = 99
    elif damage == 'invalid_vehicle':
        trace['events'][0]['vehicle'] = 99
    elif damage == 'broken_arc':
        trace['events'][1]['from'] = 0
    else:
        # Conserva duración, espera y tardanza agregadas, pero rompe continuidad.
        for name in ('departure', 'arrival', 'service_start'):
            trace['events'][1][name] += 1
    with pytest.raises((AssertionError, ValueError)):
        e.validate_trace(x, routes, trace, raw)


def test_trace_qc_rejects_vehicle_reassignment_with_same_metrics():
    x = manual_instance([[0, 100], [0, 100], [0, 100]])
    routes = [[1], [2]]
    spec = p.Traffic(mode='no_noise')
    raw = p.simulate(x, routes, 1, spec)
    trace = p.simulate(x, routes, 1, spec, trace=True)
    trace['realized_routes'] = [[2], [1]]
    with pytest.raises(AssertionError, match='asignación'):
        e.validate_trace(x, routes, trace, raw)


@pytest.mark.parametrize('policy', ['fixed_order', 'causal_edd'])
def test_evaluation_exports_common_metrics_and_every_route_event(tmp_path, policy):
    x = manual_instance()
    before = x.sha
    solution = manual_solution(x)
    seeds = [2**53 + 1, 2**63 + 17, 2**64 - 1]
    spec = p.Traffic(mode='no_noise')
    context = {'method': 'manual', 'instance_sha256': x.sha, 'solver_seed': 11}
    expected, raw = p.evaluate(x, solution['routes'], seeds, spec, policy, .5)
    actual = e.evaluate_solution(x, solution, seeds, spec, policy, .5, tmp_path, context)
    for name, value in expected.items():
        if name != 'evaluation_seconds':
            assert actual[name] == value
    assert actual['mean_travel_minutes'] == actual['cvar_travel_minutes'] == 12
    assert actual['mean_waiting_minutes'] == 2
    assert actual['scenario_strict_feasibility'] == 1
    assert actual['mean_net_execution_delta_minutes'] == 0
    assert actual['std_travel_minutes'] == 0
    assert x.sha == before
    exported = csv_rows(tmp_path / 'scenario_metrics.csv')
    assert [int(row['scenario_seed']) for row in exported] == seeds
    assert [float(row['travel_minutes']) for row in exported] == [row['travel_minutes'] for row in raw]
    events = csv_rows(tmp_path / 'events.csv')
    routes = csv_rows(tmp_path / 'realized_routes.csv')
    assert len(events) == 9  # Dos clientes y un retorno por escenario.
    assert len(routes) == 18  # Dos clientes y dos depósitos por vehículo/escenario.
    for seed in seeds:
        subset = [row for row in routes if int(row['scenario_seed']) == seed]
        assert [int(row['node']) for row in subset if row['vehicle'] == '0'] == [0, 1, 2, 0]
        empty = [row for row in subset if row['vehicle'] == '1']
        assert [row['node'] for row in empty] == ['0', '0']
        assert all(row['empty_vehicle'] == 'True' for row in empty)
    qc = json.loads((tmp_path / 'validation.json').read_text())
    assert qc['status'] == 'passed' and qc['nominal_feasible'] is True
    assert qc['event_rows'] == qc['expected_event_rows'] == 9
    assert qc['realized_route_rows'] == qc['expected_realized_route_rows'] == 18


def test_structurally_valid_but_temporally_infeasible_routes_remain_in_results(tmp_path):
    x = manual_instance([[0, 8], [0, 2], [0, 4]])
    spec = p.Traffic(mode='no_noise')
    actual = e.evaluate_solution(x, manual_solution(x), [1, 2], spec, 'fixed_order', .5,
                                 tmp_path, {'method': 'manual'})
    assert actual['scenario_strict_feasibility'] == 0
    assert actual['mean_lateness_minutes'] == 3
    assert actual['mean_depot_overtime_minutes'] == 4
    assert actual['mean_customer_violation_fraction'] == 1
    assert actual['mean_travel_minutes'] == 12
    qc = json.loads((tmp_path / 'validation.json').read_text())
    assert qc['status'] == 'passed' and qc['nominal_feasible'] is False
    assert len(csv_rows(tmp_path / 'scenario_metrics.csv')) == 2


def test_depot_only_failure_remains_exportable_without_numpy_boolean_errors(tmp_path):
    # No hay clientes tardíos; sólo falla el retorno. El evaluador sellado puede
    # producir numpy.bool_ en strict_feasible: la exportación debe normalizarlo.
    x = manual_instance([[0, 8], [0, 100], [0, 100]])
    summary = e.evaluate_solution(x, manual_solution(x), [1, 2], p.Traffic(mode='no_noise'),
                                 'fixed_order', .5, tmp_path, {'method': 'manual'})
    assert summary['scenario_strict_feasibility'] == 0
    assert summary['mean_customer_violation_fraction'] == 0
    assert summary['mean_lateness_minutes'] == 0
    assert summary['mean_depot_overtime_minutes'] == 4
    qc = json.loads((tmp_path / 'validation.json').read_text())
    assert qc['status'] == 'passed' and qc['nominal_feasible'] is False


def test_invalid_routes_fail_before_creating_a_validated_result(tmp_path):
    x = manual_instance()
    with pytest.raises(ValueError):
        e.evaluate_solution(x, {'routes': [[2], []], 'nominal': {}}, [1],
                            p.Traffic(mode='no_noise'), 'fixed_order', .95, tmp_path, {})
    assert not (tmp_path / 'validation.json').exists()


def test_stochastic_exports_match_shared_evaluator_and_fractional_cvar(tmp_path):
    x = manual_instance([[0, 100], [5, 10], [0, 100]])
    routes = [[1, 2], []]
    spec = p.Traffic()
    seeds = [13, 17, 19, 23]
    solution = manual_solution(x, routes, p.Traffic(mode='nominal_mean'))
    expected, records = p.evaluate(x, routes, seeds, spec, 'fixed_order', .625)
    summary = e.evaluate_solution(x, solution, seeds, spec, 'fixed_order', .625,
                                  tmp_path, {'method': 'manual'})
    for name, value in expected.items():
        if name != 'evaluation_seconds':
            assert summary[name] == value
    # Cola de masa 1.5: peor costo más media observación del segundo peor.
    costs = sorted((record['travel_minutes'] for record in records), reverse=True)
    assert summary['cvar_travel_minutes'] == pytest.approx((costs[0] + .5 * costs[1]) / 1.5)
    rows = csv_rows(tmp_path / 'scenario_metrics.csv')
    assert [int(row['scenario_seed']) for row in rows] == seeds
    assert [float(row['travel_minutes']) for row in rows] == [r['travel_minutes'] for r in records]


def test_recourse_route_export_records_actual_order_without_hiding_failure(tmp_path):
    x = manual_instance([[0, 100], [0, 1], [0, 20]])
    summary = e.evaluate_solution(x, manual_solution(x), [1], p.Traffic(mode='no_noise'),
                                 'causal_edd', .95, tmp_path, {'method': 'manual'})
    rows = csv_rows(tmp_path / 'realized_routes.csv')
    assert [int(row['node']) for row in rows if row['vehicle'] == '0'] == [0, 2, 1, 0]
    assert summary['mean_recourse_reorders'] == 1
    assert summary['scenario_strict_feasibility'] == 0
    assert summary['mean_late_customers'] == 1


def test_recovery_preserves_incomplete_local_files_and_restores_verified_remote(tmp_path):
    local, remote = tmp_path / 'local', tmp_path / 'remote'
    p.write_json(remote / 'result.json', {'status': 'completed'})
    expected = p.seal_bundle(remote)
    p.write_json(local / 'uncommitted.json', {'status': 'interrupted'})
    assert e._recover(local, remote) is True
    assert p.verify_bundle(local) == expected
    preserved = list(tmp_path.glob('local.interrupted_*'))
    assert len(preserved) == 1
    assert json.loads((preserved[0] / 'uncommitted.json').read_text())['status'] == 'interrupted'
    assert not (local / 'uncommitted.json').exists()


def test_recovery_detects_modified_checkpoint_instead_of_reusing_it(tmp_path):
    local, remote = tmp_path / 'local', tmp_path / 'remote'
    p.write_json(local / 'result.json', {'metric': 12})
    p.seal_bundle(local)
    p.write_json(local / 'result.json', {'metric': 0})
    with pytest.raises(ValueError, match='modificado'):
        e._recover(local, remote)
    assert not remote.exists()


def test_comparison_contract_allows_other_algorithms_but_not_other_budgets_or_data():
    config = {'objective': 'conservative_upper_travel_minutes_v1', 'time_scale': 1000,
              'role': 'development_only', 'seeds': [11], 'search_seconds': 120.,
              'scenario_split': 'evaluation', 'export_all_scenario_events': True,
              'methods': {'aco': {'ants': 20}}}
    bank_config = {'policies': ['fixed_order', 'causal_edd'], 'alpha_cvar': .95}
    first = e.comparison_protocol('a' * 64, 'b' * 64, bank_config, config)
    learned = copy.deepcopy(config)
    learned['methods'] = {'am_supervised': {'n_samples': 16}}
    assert e.comparison_protocol('a' * 64, 'b' * 64, bank_config, learned) == first
    assert e.comparison_protocol('c' * 64, 'b' * 64, bank_config, config) != first
    assert e.comparison_protocol('a' * 64, 'b' * 64, bank_config, config | {'search_seconds': 60.}) != first


def test_learned_provenance_rejects_training_geometry_in_evaluation_and_false_costs():
    x = manual_instance()
    payload = x.payload()
    payload['provenance']['geometry_group_sha256'] = 'a' * 64
    x = p.freeze_instance(payload)
    provenance = {'training_id': 'trained', 'training_seed': 11, 'model_checkpoint_sha256': 'b' * 64,
                  'train_geometry_groups': ['c' * 64], 'validation_geometry_groups': ['d' * 64],
                  'heldout_geometry_groups': ['a' * 64]}
    e.validate_learned_provenance(provenance, x, 11)
    with pytest.raises(ValueError, match='fuga'):
        e.validate_learned_provenance(provenance | {'train_geometry_groups': ['a' * 64]}, x, 11)
    with pytest.raises(ValueError, match='semilla'):
        e.validate_learned_provenance(provenance, x, 29)
    with pytest.raises(ValueError, match='Costo offline'):
        e.provenance_columns(provenance | {'training_seconds': True})


In [ ]:
%%writefile tests/test_attention.py
"""Pruebas funcionales de 03: ejecutar exclusivamente en Google Colab."""
import math
from pathlib import Path
import sys

import numpy as np
import pytest

torch = pytest.importorskip('torch')
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import attention as am
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, capacities=(1, 2, 4), demands=(0, 0, 2, 4),
            windows=None, appear_times=None):
    n = len(demands) - 1
    x = freeze_instance({
        'instance_id': 'attention-manual-fixture', 'depot_indices': [0],
        'locations': [[j, 0] for j in range(n + 1)], 'demands': list(demands),
        'capacities': list(capacities),
        'windows': windows or [[0, 100]] * (n + 1),
        'appear_times': appear_times or [0] * (n + 1),
        'provenance': {'kind': 'test_fixture'},
    })
    return SearchProblem(x, Traffic(mode='no_noise'))


def policy():
    torch.manual_seed(19)
    return am.AttentionPolicy({'model_dim': 16, 'num_heads': 4,
                               'num_layers': 2, 'dropout': 0.})


def test_teacher_loss_finite_gradients_zero_demand_and_empty_vehicles():
    p = problem()
    model = policy()
    before = p.x.sha
    loss, stats = am.teacher_loss(model, p, [[], [1, 2], [3]], 'cpu')
    assert stats['decisions'] == p.n + p.vehicles
    assert math.isfinite(stats['nll_sum'])
    assert loss.ndim == 0 and torch.isfinite(loss)
    loss.backward()
    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    assert gradients and all(torch.isfinite(gradient).all() for gradient in gradients)
    assert any(bool((gradient != 0).any()) for gradient in gradients)
    assert p.x.sha == before


def test_teacher_loss_is_equivariant_to_input_customer_order_in_eval():
    p = problem()
    model = policy().eval()
    routes = [[], [1, 2], [3]]
    first, first_stats = am.teacher_loss(model, p, routes, 'cpu')
    permuted, permuted_stats = am.teacher_loss(model, p, routes, 'cpu', node_order=[3, 1, 2])
    assert first_stats['decisions'] == permuted_stats['decisions']
    assert torch.allclose(first, permuted, atol=2e-5, rtol=2e-5)


@pytest.mark.parametrize('routes', [
    [[], [2], [3]],             # Falta cliente de demanda cero.
    [[1], [2, 2], [3]],         # Repetido.
    [[2], [1], [3]],            # Capacidad del primer vehículo.
    [[0], [1, 2], [3]],         # Depósito no es cliente.
])
def test_invalid_teacher_is_rejected(routes):
    with pytest.raises(ValueError):
        am.teacher_loss(policy(), problem(), routes, 'cpu')


def test_teacher_that_misses_window_is_rejected():
    p = problem(windows=[[0, 100], [0, .5], [0, 100], [0, 100]])
    with pytest.raises(ValueError, match='conservadora'):
        am.teacher_loss(policy(), p, [[], [1, 2], [3]], 'cpu')


def test_masks_enforce_appearance_window_and_depot_return():
    p = problem(capacities=(10,), demands=(0, 1, 1, 1),
                windows=[[0, 10], [0, .5], [0, 10], [0, 10]],
                appear_times=[0, 0, 9, 0])
    state = am._State(p)
    _, _, blocked = state.inputs(np.arange(4), am._scales(p))
    assert blocked.tolist() == [True, True, True, True]
    # Cliente 1 llega después del cierre; cliente 2 espera hasta 9 y regresaría a 11.
    state.advance(3)
    _, _, after = state.inputs(np.arange(4), am._scales(p))
    assert after[3]                         # No revisitar.
    assert after[0]                         # Clientes pendientes en último vehículo.
    assert am.decode(policy(), p, 'cpu') is None


def test_empty_routes_allowed_and_last_vehicle_must_serve_zero_demand():
    p = problem(capacities=(1, 1), demands=(0, 0))
    state = am._State(p)
    order, scales = np.arange(2), am._scales(p)
    assert not state.inputs(order, scales)[2][0]
    state.advance(0)
    assert state.inputs(order, scales)[2].tolist() == [True, False]
    state.advance(1)
    assert state.inputs(order, scales)[2].tolist() == [False, True]
    state.advance(0)
    assert state.done and state.routes == [[], [1]]


def test_heterogeneous_capacity_masks_and_future_capacity_reservation():
    p = problem(capacities=(4, 2), demands=(0, 4, 2))
    state = am._State(p)
    order, scales = np.arange(3), am._scales(p)
    _, _, blocked = state.inputs(order, scales)
    assert blocked.tolist() == [True, False, True]
    state.advance(1)
    assert state.inputs(order, scales)[2].tolist() == [False, True, True]
    state.advance(0)
    assert state.inputs(order, scales)[2].tolist() == [True, True, False]


def test_greedy_preserves_fleet_coverage_and_original_node_ids(monkeypatch):
    p = problem()
    before = p.x.sha
    model = policy()

    def ranked(encoded, current_local, state_features, action_features, blocked):
        return torch.arange(len(blocked), dtype=encoded.dtype).masked_fill(blocked, -torch.inf)

    monkeypatch.setattr(model, 'next_logits', ranked)
    routes = am.decode(model, p, 'cpu', node_order=[3, 1, 2])
    assert routes is not None and validate_routes(p.x, routes)
    assert p.score(routes)[0] <= 1e-9
    assert p.x.sha == before
    assert model.training                   # Restaurar modo previo tras inferencia.


def test_sampling_reproducible_with_explicit_generator():
    p = problem(capacities=(10,))
    model = policy()
    first = am.decode(model, p, 'cpu', mode='sample', generator=torch.Generator().manual_seed(7))
    second = am.decode(model, p, 'cpu', mode='sample', generator=torch.Generator().manual_seed(7))
    assert first is not None and first == second
    assert validate_routes(p.x, first)


def test_dead_end_never_uses_teacher_or_repair():
    p = problem(capacities=(1,), demands=(0, 2))
    result = am.solve_attention(p, 21, {'seconds': 60, 'n_samples': 2}, policy(), 'cpu')
    assert result['status'] == 'no_solution' and result['routes'] is None
    assert result['metadata']['completed_decodes'] == 0
    assert result['metadata']['failed_decodes'] == 3
    assert result['metadata']['no_solution_scope'] == 'decoder_dead_end_or_budget_not_infeasibility_proof'


def test_zero_budget_does_not_start_decode(monkeypatch):
    def forbidden(*args, **kwargs):
        raise AssertionError('No decodificar sin presupuesto.')
    monkeypatch.setattr(am, 'decode', forbidden)
    result = am.solve_attention(problem(), 2, {'seconds': 0, 'n_samples': 2}, policy(), 'cpu')
    assert result['routes'] is None
    assert result['metadata']['attempts'] == 0
    assert result['metadata']['termination'] == 'time_limit'


def test_completed_incumbent_survives_time_limit_without_extra_budget(monkeypatch):
    clock = [0.]
    deadlines = []
    p = problem()

    def completed(model, problem, device, **kwargs):
        deadlines.append(kwargs['deadline'])
        clock[0] = 11.
        return [[], [1, 2], [3]]

    monkeypatch.setattr(am.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(am, 'decode', completed)
    result = am.solve_attention(p, 7, {'seconds': 10, 'n_samples': 16}, policy(), 'cpu')
    assert deadlines == [10.]
    assert result['routes'] == [[], [1, 2], [3]]
    assert result['metadata']['termination'] == 'time_limit'
    assert result['metadata']['budget_includes_encoding'] is True
    assert result['solve_seconds'] == 11.
    assert result['metadata']['budget_overrun_seconds'] == 1.


def test_best_sample_is_selected_using_common_conservative_cost(monkeypatch):
    p = problem(capacities=(10,))
    candidates = iter([[[2, 1, 3]], [[1, 3, 2]], [[1, 2, 3]]])
    monkeypatch.setattr(am, 'decode', lambda *args, **kwargs: next(candidates))
    result = am.solve_attention(p, 11, {'seconds': 60, 'n_samples': 2}, policy(), 'cpu')
    assert result['routes'] == [[1, 3, 2]]     # Empata al mínimo; conserva el primero.
    assert result['metadata']['attempts'] == 3
    assert result['metadata']['completed_decodes'] == 3
    assert result['metadata']['search_objective_minutes'] == 6.
    assert [row['travel_minutes'] for row in result['history']] == [8., 6.]


@pytest.mark.parametrize('config', [
    {'model_dim': 15, 'num_heads': 4}, {'num_layers': 0}, {'dropout': 1.},
    {'tanh_clip': float('nan')}, {'unknown': 1}, {'dim': 16, 'model_dim': 16},
])
def test_model_configuration_rejects_invalid_values(config):
    with pytest.raises(ValueError):
        am.AttentionPolicy(config)


@pytest.mark.parametrize('order', [[0, 1, 2], [1, 1, 3], [1., 2., 3.], [True, 2, 3]])
def test_invalid_customer_permutation_rejected(order):
    with pytest.raises(ValueError, match='permutación'):
        am.decode(policy(), problem(), 'cpu', node_order=order)


def test_lookahead_prevents_serving_late_customer_before_urgent_one():
    p = problem(capacities=(10,), demands=(0, 1, 1),
                windows=[[0, 100], [0, 1.5], [0, 100]])
    blocked = am._State(p).inputs(np.arange(3), am._scales(p))[2]
    assert blocked.tolist() == [True, False, True]
    assert am.decode(policy(), p, 'cpu') == [[1, 2]]


def test_lookahead_never_masks_prefix_of_a_feasible_completion_nonmetric_or_metric():
    # Exhaustive tiny contract belongs in Colab only. No search on the bank.
    from itertools import permutations
    p = problem(capacities=(3, 3), demands=(0, 0, 2, 1),
                windows=[[0, 20], [0, 4], [0, 8], [0, 12]])
    for nonmetric in (False, True):
        if nonmetric:
            p.travel[1, 0] = 30.  # Returning through another customer can be shorter.
            p.travel[0, 3] = 15.
            p.__dict__.pop('_attention_travel_bound', None)
        checked = 0
        for permutation in permutations(range(1, p.n + 1)):
            for split in range(p.n + 1):
                routes = [list(permutation[:split]), list(permutation[split:])]
                try:
                    violation, _ = p.score(routes)
                except ValueError:
                    continue
                if violation > 1e-9:
                    continue
                state = am._State(p)
                for route in routes:
                    for customer in [*route, 0]:
                        assert not state.inputs(np.arange(p.n + 1), am._scales(p))[2][customer]
                        state.advance(customer)
                checked += 1
        assert checked > 0


def test_policy_completion_search_is_bounded_and_reports_non_sample_origin():
    p, model = problem(capacities=(10,)), policy()
    routes, stats = am.policy_completion_search(model, p, 'cpu', None, max_nodes=100)
    assert validate_routes(p.x, routes) and p.score(routes)[0] == 0.
    assert stats['expanded_nodes'] <= 100 and stats['termination'] == 'feasible_found'
    assert stats['is_policy_sample'] is False
    assert model.training
    routes, stats = am.policy_completion_search(model, p, 'cpu', None, max_nodes=1)
    assert routes is None and stats['termination'] == 'node_limit'


def test_completion_search_shares_inference_budget_and_is_not_counted_as_raw_decode(monkeypatch):
    p = problem(capacities=(10,))
    observed = []
    monkeypatch.setattr(am, 'decode', lambda *args, **kwargs: None)
    def search(model, problem, device, deadline, max_nodes):
        observed.append((deadline, max_nodes))
        return [[1, 2, 3]], {'termination': 'feasible_found', 'expanded_nodes': 4,
                             'is_policy_sample': False}
    monkeypatch.setattr(am, 'policy_completion_search', search)
    monkeypatch.setattr(am.time, 'perf_counter', lambda: 10.)
    result = am.solve_attention(p, 1, {'seconds': 30., 'n_samples': 2,
                                     'completion_search_nodes': 10}, policy(), 'cpu')
    assert observed == [(40., 10)]
    assert result['metadata']['solution_origin'] == 'policy_completion_search'
    assert result['metadata']['completed_decodes'] == 0
    assert result['metadata']['raw_decode_complete_fraction'] == 0.


def test_uniform_masked_policy_has_no_learned_parameters_and_equal_allowed_logits():
    model = am.UniformMaskedPolicy()
    assert list(model.parameters()) == []
    logits = model.next_logits(torch.zeros(3, 1), 0, None, None,
                               torch.tensor([True, False, False]))
    assert logits[0] == -torch.inf and logits[1] == logits[2] == 0.


In [ ]:
%%writefile tests/test_teacher_feasibility.py
"""Profesor CP-SAT: controles exclusivamente en Google Colab; local sólo AST."""
from pathlib import Path
import sys
from types import SimpleNamespace

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import teacher_feasibility as teacher
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, capacities=(4, 4, 4), demands=(0, 0, 1, 1), windows=None, locations=None):
    n = len(demands) - 1
    instance = freeze_instance({
        'instance_id': 'independent-teacher-manual-fixture', 'depot_indices': [0],
        'locations': locations or [[j, 0] for j in range(n + 1)],
        'demands': list(demands), 'capacities': list(capacities),
        'windows': windows or [[0, 100]] * (n + 1), 'appear_times': [0] * (n + 1),
        'provenance': {'kind': 'manual_test_fixture'},
    })
    return SearchProblem(instance, Traffic(mode='no_noise'))


def test_feasibility_preserves_fixed_fleet_and_zero_demand_customer():
    pytest.importorskip('ortools')
    p = problem(capacities=(2, 3, 4, 5))
    before = p.x.sha
    result = teacher.solve_feasibility(p, 211, {'seconds': 30, 'workers': 1})
    assert result['status'] == 'feasible'
    assert validate_routes(p.x, result['routes'])
    assert len(result['routes']) == 4 and any(not route for route in result['routes'])
    assert sorted(customer for route in result['routes'] for customer in route) == [1, 2, 3]
    assert p.score(result['routes'])[0] <= 1e-9 and p.x.sha == before
    meta = result['metadata']
    assert meta['cp_sat_status_name'] in ('OPTIMAL', 'FEASIBLE')
    assert meta['feasibility_only'] and not meta['cost_optimization_performed']
    assert not meta['surrogate_optimal'] and not meta['stochastic_optimal']
    assert meta['best_bound'] is None and meta['seed'] == meta['cp_sat_seed'] == 211
    assert meta['model_seconds'] >= 0 and meta['search_solver_seconds'] >= 0
    assert meta['validation_seconds'] >= 0 and meta['solver_version']
    assert result['solve_seconds'] >= meta['search_solver_seconds']


def test_zero_time_cycles_cannot_skip_customers_or_create_disconnected_routes():
    pytest.importorskip('ortools')
    p = problem(capacities=(1,), demands=(0, 0, 0, 0), locations=[[0, 0]] * 4)
    result = teacher.solve_feasibility(p, 7, {'seconds': 30})
    assert result['status'] == 'feasible' and len(result['routes']) == 1
    assert sorted(result['routes'][0]) == [1, 2, 3]
    assert result['metadata']['search_objective_minutes'] == 0.
    assert not result['metadata']['surrogate_optimal']


def test_infeasible_window_has_only_conservative_integer_scope():
    pytest.importorskip('ortools')
    p = problem(demands=(0, 1), windows=[[0, 100], [0, .5]])
    result = teacher.solve_feasibility(p, 11, {'seconds': 30})
    assert result['routes'] is None and result['status'] == 'no_solution'
    assert result['metadata']['cp_sat_status_name'] == 'INFEASIBLE'
    assert result['metadata']['conservative_integer_infeasible']
    assert result['metadata']['infeasibility_scope'] == 'conservative_integer_model_only_not_original_stochastic_problem'


def test_unsupported_binding_capacity_does_not_load_solver_or_relax_fleet(monkeypatch):
    def forbidden():
        raise AssertionError('No cargar solver fuera del alcance de capacidad.')

    monkeypatch.setattr(teacher, '_cp_sat', forbidden)
    p = problem(capacities=(1, 3), demands=(0, 2, 1))
    result = teacher.solve_feasibility(p, 1, {'seconds': 30})
    assert result['routes'] is None and result['status'] == 'unsupported_capacity'
    assert result['metadata']['termination'] == 'unsupported_capacity'
    assert result['metadata']['cp_sat_status_name'] == 'NOT_RUN'
    assert not result['metadata']['conservative_integer_infeasible']


def test_nonfinite_capacity_is_rejected_before_solver(monkeypatch):
    p = problem()
    p.capacities = p.capacities.copy()
    p.capacities[0] = float('nan')
    monkeypatch.setattr(teacher, '_cp_sat', lambda: pytest.fail('No cargar solver con capacidad NaN.'))
    with pytest.raises(ValueError, match='Capacidades'):
        teacher.solve_feasibility(p, 11)


def test_zero_budget_does_not_load_dependency_or_construct_model(monkeypatch):
    def forbidden(*args):
        raise AssertionError('No importar/construir sin presupuesto.')

    monkeypatch.setattr(teacher, '_cp_sat', forbidden)
    monkeypatch.setattr(teacher, '_integer_problem', forbidden)
    result = teacher.solve_feasibility(problem(), 11, {'seconds': 0})
    assert result['routes'] is None
    assert result['metadata']['termination'] == 'time_limit_before_search'
    assert result['metadata']['search_solver_seconds'] == 0.
    assert result['metadata']['solver_version'] is None


def test_import_consumes_shared_budget_before_build(monkeypatch):
    p, clock = problem(), [0.]

    def expensive_import():
        clock[0] = 11.
        return object()

    monkeypatch.setattr(teacher.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(teacher, '_cp_sat', expensive_import)
    monkeypatch.setattr(teacher.importlib.metadata, 'version', lambda *args: 'test-double')
    result = teacher.solve_feasibility(p, 11, {'seconds': 10})
    assert result['routes'] is None and result['solve_seconds'] == 11.
    assert result['metadata']['import_seconds'] == 11.
    assert result['metadata']['budget_overrun_seconds'] == 1.
    assert result['metadata']['search_solver_seconds'] == 0.


def test_build_consumes_shared_budget_and_never_starts_solver_after_expiry(monkeypatch):
    p, clock = problem(), [0.]

    def expensive_build(problem, data, cp, deadline):
        assert deadline == 10.
        clock[0] = 11.
        teacher._deadline(deadline)

    monkeypatch.setattr(teacher.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(teacher, '_cp_sat', lambda: object())
    monkeypatch.setattr(teacher.importlib.metadata, 'version', lambda *args: 'test-double')
    monkeypatch.setattr(teacher, '_build_model', expensive_build)
    result = teacher.solve_feasibility(p, 11, {'seconds': 10})
    assert result['routes'] is None and result['metadata']['model_seconds'] == 11.
    assert result['metadata']['search_solver_seconds'] == 0.
    assert result['metadata']['termination'] == 'time_limit_before_search'


def test_completed_feasibility_survives_deadline_and_optimal_status_is_not_cost_optimal(monkeypatch):
    p, clock = problem(demands=(0, 0, 1)), [0.]
    calls = []

    class Solver:
        def __init__(self):
            self.parameters = SimpleNamespace()

        def solve(self, model):
            calls.append(vars(self.parameters).copy())
            clock[0] = 11.
            return 4

        def status_name(self, code):
            return 'OPTIMAL'

        def response_stats(self):
            return 'manual solver control-plane fixture'

        def boolean_value(self, variable):
            return True

    cp = SimpleNamespace(CpSolver=Solver, FEASIBLE=2, OPTIMAL=4, INFEASIBLE=3, MODEL_INVALID=1)
    monkeypatch.setattr(teacher.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(teacher, '_cp_sat', lambda: cp)
    monkeypatch.setattr(teacher.importlib.metadata, 'version', lambda *args: 'test-double')
    monkeypatch.setattr(teacher, '_build_model', lambda *args: (
        SimpleNamespace(validate=lambda: ''), {(0, 1): True, (1, 2): True, (2, 0): True}))
    result = teacher.solve_feasibility(p, 2**40 + 7, {'seconds': 10})
    assert result['routes'] == [[1, 2], [], []] and result['status'] == 'feasible'
    assert calls[0]['max_time_in_seconds'] == 10.
    assert calls[0]['num_search_workers'] == 1 and calls[0]['stop_after_first_solution']
    assert calls[0]['random_seed'] == (2**40 + 7) % (2**31 - 1)
    assert result['metadata']['budget_overrun_seconds'] == 1.
    assert result['metadata']['cp_sat_status_name'] == 'OPTIMAL'
    assert not result['metadata']['surrogate_optimal'] and result['metadata']['best_bound'] is None


def test_integer_matrix_reconstruction_does_not_silently_round_new_travel():
    p = problem()
    data = teacher._integer_problem(p, float('inf'))
    assert data['matrix'][0][1] == p.time_scale
    assert data['time_scale'] == p.time_scale
    # Mutable SearchProblem matrix belongs only to this manual fixture.
    p.travel[0, 1] += .0004
    with pytest.raises(ValueError, match='time_scale'):
        teacher._integer_problem(p, float('inf'))


def test_extraction_rejects_disconnected_cycles_and_restores_empty_vehicles():
    p = problem(demands=(0, 0, 1))
    assert teacher._extract_routes(p, [(0, 1), (1, 2), (2, 0)]) == [[1, 2], [], []]
    with pytest.raises(ValueError, match='cubre'):
        teacher._extract_routes(p, [(1, 2), (2, 1)])
    with pytest.raises(ValueError, match='sucesores'):
        teacher._extract_routes(p, [(0, 1), (1, 2), (1, 0), (2, 0)])


@pytest.mark.parametrize('config', [[], {'unknown': 1}, {'seconds': -1}, {'seconds': True},
                                    {'seconds': float('nan')}, {'workers': 0}, {'workers': 2}, {'workers': True}])
def test_invalid_configuration_is_rejected(config):
    with pytest.raises(ValueError):
        teacher.validate_config(config)


@pytest.mark.parametrize('seed', [-1, True, 1.5, '11'])
def test_invalid_solver_seed_is_rejected(seed):
    with pytest.raises(ValueError, match='seed'):
        teacher.solve_feasibility(problem(), seed, {'seconds': 0})


In [ ]:
%%writefile tests/test_supervised_data.py
"""Contratos de datos supervisados: ejecutar exclusivamente desde Google Colab.

Fixtures manuales y profesores simulados: no usan certificados ni el banco real.
Estas pruebas tampoco se ejecutan en el equipo local del autor.
"""
import copy
import json
import shutil
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import experiments as exp
from svrplab import pilot as p
from svrplab import supervised_data as sd
from svrplab.search import SearchProblem


def manual_instances(duplicate_geometry=False):
    instances = []
    for i in range(4):
        locations = [[0, 0], [i + 1, 0], [0, i + 1]]
        group = p.digest({'locations': locations, 'depot_indices': [0]})
        payload = {'instance_id': f'manual_geometry_{i}', 'depot_indices': [0],
                   'locations': locations, 'demands': [0, 0, 1], 'capacities': [2, 2],
                   'windows': [[0, 1000], [0, 1000], [0, 1000]],
                   'appear_times': [0, 0, 0],
                   'provenance': {'geometry_group_sha256': group, 'kind': 'manual_test'}}
        instances.append(p.freeze_instance(payload))
        if duplicate_geometry:
            derived = copy.deepcopy(payload)
            derived['instance_id'] += '_other_window'
            derived['windows'][1][0] = 1
            instances.append(p.freeze_instance(derived))
    return instances


def config():
    return {'seeds': [101, 211], 'seconds_per_method': 120., 'time_scale': 1000,
            'aco': {'max_iterations': 1, 'ants': 1},
            'tabu': {'max_iterations': 1, 'neighborhood_samples': 1,
                     'initial_restarts': 1, 'tenure': 1}}


def teacher(problem, seed, settings):
    return problem.result([[1, 2], []], 'feasible', .01, [], seed=seed,
                          construction='manual_test_double')


def build(tmp_path, instances, settings=None):
    return sd.build_teacher_corpus(instances, p.Traffic(), config() if settings is None else settings,
                                   tmp_path / 'work', tmp_path / 'drive',
                                   'manual_bank', 'a' * 64, p.file_hash(p.__file__))


def fallback_config():
    return config() | {'fallback': {'enabled': True, 'seeds': [401, 503, 607],
                                    'seconds_per_attempt': 600., 'workers': 1}}


def no_solution(problem, seed, settings):
    return problem.result(None, 'no_solution', .01, [], seed=seed,
                          construction='manual_test_double')


def test_four_folds_preserve_all_derivatives_and_hold_out_every_geometry_once():
    instances = manual_instances(duplicate_geometry=True)
    before = [x.sha for x in instances]
    folds = sd.make_geometry_folds(instances)
    groups = {x.provenance['geometry_group_sha256'] for x in instances}
    assert len(folds) == 4
    assert {f['heldout_geometry_groups'][0] for f in folds} == groups
    for fold in folds:
        assert sd.validate_fold(fold, instances)
        assert len(fold['train_geometry_groups']) == 2
        assert len(fold['validation_geometry_groups']) == 1
        assigned = [group for role in sd._ROLES for group in fold[role]]
        assert len(assigned) == len(set(assigned)) == 4
        # El rol usa el grupo; ambas ventanas tienen necesariamente el mismo rol.
        for x in instances:
            assert sum(x.provenance['geometry_group_sha256'] in fold[r] for r in sd._ROLES) == 1
    assert before == [x.sha for x in instances]
    assert sd.make_geometry_folds(list(reversed(instances))) == folds


@pytest.mark.parametrize('damage', ['overlap', 'unknown', 'empty', 'omission', 'duplicate', 'two_heldout'])
def test_fold_rejects_leakage_unknown_groups_and_incomplete_registry(damage):
    instances = manual_instances()
    fold = sd.make_geometry_folds(instances)[0]
    if damage == 'overlap':
        fold['train_geometry_groups'].append(fold['heldout_geometry_groups'][0])
    elif damage == 'unknown':
        fold['train_geometry_groups'][0] = 'f' * 64
    elif damage == 'empty':
        fold['validation_geometry_groups'] = []
    elif damage == 'omission':
        fold['train_geometry_groups'].pop()
    elif damage == 'duplicate':
        fold['train_geometry_groups'].append(fold['train_geometry_groups'][0])
    else:
        fold['heldout_geometry_groups'].append(fold['train_geometry_groups'].pop())
    with pytest.raises(ValueError):
        sd.validate_fold(fold, instances)


def test_too_few_independent_geometries_cannot_be_rescued_by_other_windows():
    instances = manual_instances(duplicate_geometry=True)[:4]
    assert len(instances) == 4
    with pytest.raises(ValueError, match='tres geometrías'):
        sd.make_geometry_folds(instances)


@pytest.mark.parametrize('change', [{'seeds': []}, {'seeds': [1, 1]}, {'seeds': [True]},
                                    {'seconds_per_method': 0}, {'time_scale': 0},
                                    {'aco': {'seconds': 100}}, {'certificate': 'forbidden'}])
def test_teacher_configuration_rejects_ambiguous_or_invalid_protocol(change):
    with pytest.raises(ValueError):
        sd._configuration(config() | change)


def test_corpus_saves_every_attempt_deduplicates_labels_and_resumes_without_search(tmp_path, monkeypatch):
    monkeypatch.setattr(sd.mh, 'solve_aco', teacher)
    monkeypatch.setattr(sd.mh, 'solve_tabu', teacher)
    instances = manual_instances()
    root = build(tmp_path, instances)
    index = exp.read_json(root / 'index/corpus.json')
    assert index['status'] == 'complete'
    assert index['attempts'] == 16
    assert len(index['labels']) == 4  # Las réplicas coincidentes no multiplican etiquetas.
    assert not (root / 'CHECKSUMS.json').exists()  # Sólo paquetes inmutables.
    assert len(list((root / 'attempts').glob('*/*/*/attempt.json'))) == 16
    assert len(list((root / 'attempts').glob('*/*/*/planned_routes.csv'))) == 16
    assert sum(c['attempts'] for c in index['teacher_costs_by_geometry']) == 16
    remote = tmp_path / 'drive' / 'teacher_corpora' / 'manual_bank' / root.name
    assert p.verify_bundle(remote / 'index') == p.verify_bundle(root / 'index')
    checksum_before = p.verify_bundle(root / 'index')

    def forbidden(*args, **kwargs):
        raise AssertionError('No debe volver a ejecutarse un intento sellado.')

    monkeypatch.setattr(sd.mh, 'solve_aco', forbidden)
    monkeypatch.setattr(sd.mh, 'solve_tabu', forbidden)
    assert build(tmp_path, instances) == root
    assert p.verify_bundle(root / 'index') == checksum_before


def test_loading_fold_never_opens_heldout_routes_and_costs_include_all_attempts(tmp_path, monkeypatch):
    monkeypatch.setattr(sd.mh, 'solve_aco', teacher)
    monkeypatch.setattr(sd.mh, 'solve_tabu', teacher)
    instances = manual_instances(duplicate_geometry=True)
    root = build(tmp_path, instances)
    fold = sd.make_geometry_folds(instances)[0]
    heldout_shas = {x.sha for x in instances
                    if x.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups']}
    original = exp.read_json

    def guarded(path):
        path = Path(path)
        if path.name == 'result.json' and any(sha in path.parts for sha in heldout_shas):
            raise AssertionError('Las etiquetas heldout no deben llegar al entrenador.')
        return original(path)

    monkeypatch.setattr(exp, 'read_json', guarded)
    train, validation, provenance = sd.load_fold_examples(root, instances, fold)
    assert len(train) == 4 and len(validation) == 2
    assert {r['geometry_group_sha256'] for r in train} == set(fold['train_geometry_groups'])
    assert {r['geometry_group_sha256'] for r in validation} == set(fold['validation_geometry_groups'])
    assert all(r['instance_sha256'] not in heldout_shas for r in train + validation)
    assert provenance['heldout_labels_exposed'] is False
    assert sum(c['attempts'] for c in provenance['teacher_costs_by_geometry']) == 32
    for key in sd._COSTS:
        assert 0 <= provenance['selected_teacher_costs'][key] <= provenance['all_teacher_costs'][key]


def test_zero_labels_fail_after_persisting_all_failures_and_costs(tmp_path, monkeypatch):
    def fail(problem, seed, settings):
        raise RuntimeError('Fallo deliberado del profesor manual.')

    monkeypatch.setattr(sd.mh, 'solve_aco', fail)
    monkeypatch.setattr(sd.mh, 'solve_tabu', fail)
    instances = manual_instances()
    with pytest.raises(RuntimeError, match='Corpus sin etiquetas válidas'):
        build(tmp_path, instances)
    indexes = list((tmp_path / 'drive').glob('teacher_corpora/manual_bank/*/index/corpus.json'))
    assert len(indexes) == 1
    root = indexes[0].parent.parent
    index = json.loads(indexes[0].read_text())
    assert index['status'] == 'no_labels_for_geometry'
    assert index['attempts'] == 16 and not index['labels']
    assert len(index['missing_geometry_groups']) == 4
    assert sum(c['rejected_attempts'] for c in index['teacher_costs_by_geometry']) == 16
    attempts = [exp.read_json(path) for path in root.glob('attempts/*/*/*/attempt.json')]
    assert len(attempts) == 16
    assert all(a['status'] == 'solver_error' and a['teacher_search_seconds'] >= 0 for a in attempts)
    assert all(not a['label_validation']['eligible_label'] for a in attempts)
    with pytest.raises(ValueError, match='incompleto'):
        sd.load_fold_examples(root, instances, sd.make_geometry_folds(instances)[0])


def test_rejected_solution_is_saved_but_not_used_as_a_label(tmp_path, monkeypatch):
    instances = manual_instances()

    def malformed(problem, seed, settings):
        return {'routes': [[1], []], 'history': [], 'solve_seconds': .01,
                'status': 'claimed_feasible', 'metadata': {'deliberately_invalid': True}}

    monkeypatch.setattr(sd.mh, 'solve_aco', malformed)
    monkeypatch.setattr(sd.mh, 'solve_tabu', teacher)
    root = build(tmp_path, instances)
    index = exp.read_json(root / 'index/corpus.json')
    assert all(label['teacher_method'] == 'tabu' for label in index['labels'])
    assert sum(c['rejected_attempts'] for c in index['teacher_costs_by_geometry']) == 8
    attempt_dir = next(root.glob('attempts/*/aco/seed_101'))
    assert exp.read_json(attempt_dir / 'result.json')['routes'] == [[1], []]
    assert exp.read_json(attempt_dir / 'validation.json')['structural_valid'] is False


def test_label_eligibility_recomputes_conservative_score_and_nominal_independently(monkeypatch):
    x = manual_instances()[0]
    problem = SearchProblem(x, p.Traffic())
    result = teacher(problem, 101, {})
    # La afirmación del solver no reemplaza ninguna comprobación del corpus.
    result['metadata']['conservative_feasible'] = True
    monkeypatch.setattr(problem, 'score', lambda routes: (2., 12.))
    validation, _ = sd._check_result(problem, result, p.Traffic())
    assert validation['structural_valid']
    assert not validation['eligible_label']
    assert validation['reason'] == 'conservative_infeasible'
    assert validation['search_violation_minutes'] == 2.
    monkeypatch.setattr(problem, 'score', lambda routes: (0., 12.))
    observed_modes = []

    def failing_nominal(instance, routes, seed, spec, policy, trace):
        observed_modes.append(spec.mode)
        return {'strict_feasible': False, 'events': []}

    monkeypatch.setattr(p, 'simulate', failing_nominal)
    validation, _ = sd._check_result(problem, result, p.Traffic())
    assert validation['conservative_feasible']
    assert not validation['eligible_label']
    assert validation['reason'] == 'nominal_infeasible'
    assert observed_modes == ['nominal_mean']


def test_changed_label_artifact_cannot_be_loaded(tmp_path, monkeypatch):
    monkeypatch.setattr(sd.mh, 'solve_aco', teacher)
    monkeypatch.setattr(sd.mh, 'solve_tabu', teacher)
    instances = manual_instances()
    root = build(tmp_path, instances)
    path = next(root.glob('attempts/*/*/*/result.json'))
    path.write_text('{}')
    with pytest.raises(ValueError, match='modificado'):
        sd.load_fold_examples(root, instances, sd.make_geometry_folds(instances)[0])


@pytest.mark.parametrize('change', [
    {'enabled': 1}, {'seeds': []}, {'seeds': [401, 401]}, {'seeds': [True]},
    {'seconds_per_attempt': 0}, {'seconds_per_attempt': float('inf')},
    {'workers': 0}, {'workers': True}, {'workers': 2}, {'unknown': True},
])
def test_fallback_configuration_rejects_invalid_or_undeclared_protocol(change):
    settings = fallback_config()
    settings['fallback'].update(change)
    with pytest.raises(ValueError):
        sd._configuration(settings)


def test_fallback_is_disabled_for_legacy_teacher_configuration():
    normalized = sd._configuration(config())
    assert normalized['fallback']['enabled'] is False
    assert normalized['seeds'] == [101, 211]
    assert normalized['seconds_per_method'] == 120.


def test_fallback_runs_all_declared_seeds_and_accounts_every_attempt(tmp_path, monkeypatch):
    instances = manual_instances()
    settings = fallback_config()
    base_calls, fallback_calls = [], []

    def missing_base(problem, seed, solver_settings):
        base_calls.append((problem.x.sha, seed))
        return no_solution(problem, seed, solver_settings)

    def independent_fallback(problem, seed, solver_settings):
        # Ambos profesores y todas sus semillas deben haber terminado primero.
        assert sum(sha == problem.x.sha for sha, _ in base_calls) == 4
        fallback_calls.append((problem.x.sha, seed, dict(solver_settings)))
        assert set(solver_settings) == {'seconds', 'workers'}
        assert 0 < solver_settings['seconds'] <= 600.
        assert solver_settings['workers'] == 1
        return teacher(problem, seed, solver_settings)

    monkeypatch.setattr(sd.mh, 'solve_aco', missing_base)
    monkeypatch.setattr(sd.mh, 'solve_tabu', missing_base)
    monkeypatch.setattr(sd, 'solve_feasibility', independent_fallback)
    root = build(tmp_path, instances, settings)
    index = exp.read_json(root / 'index/corpus.json')
    assert index['status'] == 'complete'
    assert index['attempts'] == 28 and len(index['labels']) == 4
    assert len(base_calls) == 16 and len(fallback_calls) == 12
    assert {(sha, seed) for sha, seed, _ in fallback_calls} == {
        (x.sha, seed) for x in instances for seed in settings['fallback']['seeds']}
    assert {row['teacher_method'] for row in index['labels']} == {'cp_sat_feasibility'}
    assert {row['instance_sha256'] for row in index['labels']} == {x.sha for x in instances}
    assert index['missing_instance_sha256'] == []
    attempts = [exp.read_json(path) for path in root.glob('attempts/*/*/*/attempt.json')]
    fallback_attempts = [a for a in attempts if a['teacher_method'] == 'cp_sat_feasibility']
    assert len(attempts) == 28 and len(fallback_attempts) == 12
    assert all(a['search_budget_seconds'] == 600. for a in fallback_attempts)
    assert all(a['search_budget_seconds'] == 120. for a in attempts if a not in fallback_attempts)
    assert all(a['construction'] == 'independent_from_scratch_without_certificates' for a in attempts)
    assert all(a['label_kind'] == 'heuristic_not_proven_optimal' for a in attempts)
    assert sum(c['accepted_attempts'] for c in index['teacher_costs_by_geometry']) == 12
    assert sum(c['rejected_attempts'] for c in index['teacher_costs_by_geometry']) == 16
    for field in sd._COSTS:
        assert all(a[field] >= 0 for a in attempts)
        assert sum(c[field] for c in index['teacher_costs_by_geometry']) == pytest.approx(
            sum(a[field] for a in attempts))
    fold = sd.make_geometry_folds(instances)[0]
    train, validation, provenance = sd.load_fold_examples(root, instances, fold)
    assert len(train) == 2 and len(validation) == 1
    assert all(e['teacher']['method'] == 'cp_sat_feasibility' for e in train + validation)
    assert provenance['heldout_labels_exposed'] is False
    for field in sd._COSTS:
        assert provenance['all_teacher_costs'][field] == pytest.approx(sum(a[field] for a in attempts))


def test_eligible_base_labels_skip_fallback_for_every_instance(tmp_path, monkeypatch):
    fallback_calls = []

    def forbidden_fallback(problem, seed, settings):
        fallback_calls.append((problem.x.sha, seed))
        raise AssertionError('Una etiqueta base elegible no necesita respaldo.')

    monkeypatch.setattr(sd.mh, 'solve_aco', no_solution)
    monkeypatch.setattr(sd.mh, 'solve_tabu', teacher)
    monkeypatch.setattr(sd, 'solve_feasibility', forbidden_fallback)
    root = build(tmp_path, manual_instances(), fallback_config())
    index = exp.read_json(root / 'index/corpus.json')
    assert index['status'] == 'complete' and index['attempts'] == 16
    assert not fallback_calls
    assert all(label['teacher_method'] == 'tabu' for label in index['labels'])
    assert not list(root.glob('attempts/*/cp_sat_feasibility'))


@pytest.mark.parametrize('failure', ['no_solution', 'malformed', 'solver_error'])
def test_failed_fallback_persists_rejections_and_does_not_complete_corpus(tmp_path, monkeypatch, failure):
    fallback_calls = []

    def failed_fallback(problem, seed, settings):
        fallback_calls.append((problem.x.sha, seed))
        if failure == 'solver_error':
            raise RuntimeError('Fallo deliberado de respaldo manual.')
        if failure == 'malformed':
            return {'routes': [[1], []], 'history': [], 'solve_seconds': .01,
                    'status': 'claimed_feasible', 'metadata': {}}
        return no_solution(problem, seed, settings)

    monkeypatch.setattr(sd.mh, 'solve_aco', no_solution)
    monkeypatch.setattr(sd.mh, 'solve_tabu', no_solution)
    monkeypatch.setattr(sd, 'solve_feasibility', failed_fallback)
    instances = manual_instances()
    with pytest.raises(RuntimeError, match='Corpus sin etiquetas válidas'):
        build(tmp_path, instances, fallback_config())
    paths = list((tmp_path / 'drive').glob('teacher_corpora/manual_bank/*/index/corpus.json'))
    assert len(paths) == 1
    root = paths[0].parent.parent
    index = exp.read_json(paths[0])
    assert index['status'] == 'no_labels_for_geometry'
    assert index['attempts'] == 28 and not index['labels']
    assert len(fallback_calls) == 12
    assert set(index['missing_instance_sha256']) == {x.sha for x in instances}
    assert sum(c['rejected_attempts'] for c in index['teacher_costs_by_geometry']) == 28
    assert p.verify_bundle(root / 'index')
    attempts = [exp.read_json(path) for path in root.glob('attempts/*/cp_sat_feasibility/*/attempt.json')]
    assert len(attempts) == 12
    assert all(not a['label_validation']['eligible_label'] for a in attempts)
    expected_reason = {'no_solution': 'no_solution', 'malformed': 'validation_error',
                       'solver_error': 'solver_error'}[failure]
    assert all(a['label_validation']['reason'] == expected_reason for a in attempts)
    assert all(a['teacher_wall_seconds'] >= 0 for a in attempts)
    with pytest.raises(ValueError, match='incompleto'):
        sd.load_fold_examples(root, instances, sd.make_geometry_folds(instances)[0])


def test_fallback_recovery_from_drive_reuses_sealed_attempts_and_costs(tmp_path, monkeypatch):
    monkeypatch.setattr(sd.mh, 'solve_aco', no_solution)
    monkeypatch.setattr(sd.mh, 'solve_tabu', no_solution)
    monkeypatch.setattr(sd, 'solve_feasibility', teacher)
    instances = manual_instances()
    settings = fallback_config()
    root = build(tmp_path, instances, settings)
    before = exp.read_json(root / 'index/corpus.json')
    checksum_before = p.verify_bundle(root / 'index')
    solver_calls = []

    def forbidden_solver(problem, seed, solver_settings):
        solver_calls.append((problem.x.sha, seed))
        raise AssertionError('Los intentos recuperados no repiten búsquedas.')

    monkeypatch.setattr(sd.mh, 'solve_aco', forbidden_solver)
    monkeypatch.setattr(sd.mh, 'solve_tabu', forbidden_solver)
    monkeypatch.setattr(sd, 'solve_feasibility', forbidden_solver)
    shutil.rmtree(root)  # Simula la pérdida del runtime; Drive conserva los paquetes.
    assert build(tmp_path, instances, settings) == root
    assert solver_calls == []
    assert exp.read_json(root / 'index/corpus.json') == before
    assert p.verify_bundle(root / 'index') == checksum_before


def test_interrupted_fallback_is_charged_once_and_remaining_seeds_resume(tmp_path, monkeypatch):
    instances = manual_instances()
    settings = fallback_config()
    base_calls, fallback_calls = [], []

    def missing_base(problem, seed, solver_settings):
        base_calls.append((problem.x.sha, seed))
        return no_solution(problem, seed, solver_settings)

    def interrupt_once(problem, seed, solver_settings):
        fallback_calls.append((problem.x.sha, seed))
        if len(fallback_calls) == 1:
            raise KeyboardInterrupt('Interrupción manual del primer respaldo.')
        return teacher(problem, seed, solver_settings)

    monkeypatch.setattr(sd.mh, 'solve_aco', missing_base)
    monkeypatch.setattr(sd.mh, 'solve_tabu', missing_base)
    monkeypatch.setattr(sd, 'solve_feasibility', interrupt_once)
    with pytest.raises(KeyboardInterrupt, match='primer respaldo'):
        build(tmp_path, instances, settings)
    saved = list((tmp_path / 'drive').glob(
        'teacher_corpora/manual_bank/*/attempts/*/cp_sat_feasibility/*/attempt.json'))
    assert len(saved) == 1
    interrupted_attempt = exp.read_json(saved[0])
    assert interrupted_attempt['status'] == 'interrupted'
    assert interrupted_attempt['label_validation']['eligible_label'] is False
    assert interrupted_attempt['teacher_wall_seconds'] >= 0
    interrupted_checksum = p.verify_bundle(saved[0].parent)
    root = build(tmp_path, instances, settings)
    index = exp.read_json(root / 'index/corpus.json')
    assert index['status'] == 'complete' and index['attempts'] == 28
    assert len(base_calls) == 16 and len(fallback_calls) == 12
    assert len(set(fallback_calls)) == 12
    assert exp.read_json(saved[0]) == interrupted_attempt
    assert p.verify_bundle(saved[0].parent) == interrupted_checksum
    assert sum(c['accepted_attempts'] for c in index['teacher_costs_by_geometry']) == 11
    assert sum(c['rejected_attempts'] for c in index['teacher_costs_by_geometry']) == 17


def test_fallback_coverage_is_per_instance_with_shared_geometry(tmp_path, monkeypatch):
    instances = manual_instances(duplicate_geometry=True)
    missing = {x.sha for x in instances if x.instance_id.endswith('_other_window')}
    fallback_calls = []

    def base_teacher(problem, seed, settings):
        return (no_solution if problem.x.sha in missing else teacher)(problem, seed, settings)

    def targeted_fallback(problem, seed, settings):
        fallback_calls.append((problem.x.sha, seed))
        return teacher(problem, seed, settings)

    monkeypatch.setattr(sd.mh, 'solve_aco', base_teacher)
    monkeypatch.setattr(sd.mh, 'solve_tabu', base_teacher)
    monkeypatch.setattr(sd, 'solve_feasibility', targeted_fallback)
    root = build(tmp_path, instances, fallback_config())
    index = exp.read_json(root / 'index/corpus.json')
    assert index['status'] == 'complete'
    assert index['attempts'] == 44 and len(index['labels']) == 8
    assert len(fallback_calls) == 12
    assert {sha for sha, _ in fallback_calls} == missing
    assert {r['instance_sha256'] for r in index['labels']} == {x.sha for x in instances}
    fold = sd.make_geometry_folds(instances)[0]
    train, validation, _ = sd.load_fold_examples(root, instances, fold)
    expected_train = {x.sha for x in instances
                      if x.provenance['geometry_group_sha256'] in fold['train_geometry_groups']}
    expected_validation = {x.sha for x in instances
                           if x.provenance['geometry_group_sha256'] in fold['validation_geometry_groups']}
    assert {e['instance_sha256'] for e in train} == expected_train
    assert {e['instance_sha256'] for e in validation} == expected_validation


def test_labelled_geometry_cannot_hide_unlabelled_derived_instance(tmp_path, monkeypatch):
    instances = manual_instances(duplicate_geometry=True)
    missing = {x.sha for x in instances if x.instance_id.endswith('_other_window')}

    def incomplete_teacher(problem, seed, settings):
        return (no_solution if problem.x.sha in missing else teacher)(problem, seed, settings)

    monkeypatch.setattr(sd.mh, 'solve_aco', incomplete_teacher)
    monkeypatch.setattr(sd.mh, 'solve_tabu', incomplete_teacher)
    with pytest.raises(RuntimeError, match='Corpus sin etiquetas válidas'):
        build(tmp_path, instances)
    paths = list((tmp_path / 'drive').glob('teacher_corpora/manual_bank/*/index/corpus.json'))
    assert len(paths) == 1
    index = exp.read_json(paths[0])
    assert index['status'] == 'no_labels_for_geometry'
    assert index['attempts'] == 32 and len(index['labels']) == 4
    assert set(index['missing_instance_sha256']) == missing
    assert set(index['missing_geometry_groups']) == {
        x.provenance['geometry_group_sha256'] for x in instances}


def test_fold_loader_rechecks_instance_coverage_even_with_complete_geometry_labels(tmp_path, monkeypatch):
    monkeypatch.setattr(sd.mh, 'solve_aco', teacher)
    monkeypatch.setattr(sd.mh, 'solve_tabu', teacher)
    instances = manual_instances(duplicate_geometry=True)
    root = build(tmp_path, instances)
    fold = sd.make_geometry_folds(instances)[0]
    missing = next(x.sha for x in instances if x.instance_id.endswith('_other_window')
                   and x.provenance['geometry_group_sha256'] in fold['train_geometry_groups'])
    index = exp.read_json(root / 'index/corpus.json')
    index['labels'] = [label for label in index['labels'] if label['instance_sha256'] != missing]
    exp.write_json(root / 'index/corpus.json', index)
    # El índice es íntegro en disco, pero semánticamente omite una instancia train.
    p.seal_bundle(root / 'index')
    with pytest.raises(ValueError, match='Faltan etiquetas de una instancia'):
        sd.load_fold_examples(root, instances, fold)


In [ ]:
%%writefile tests/test_supervised.py
"""Pruebas funcionales de 03: ejecutar exclusivamente en Google Colab.

Usan tres geometrías manuales de dos clientes y profesores simulados. No usan
certificados ni generan instancias experimentales. No ejecutar localmente.
"""
from dataclasses import asdict
from pathlib import Path
import sys

import numpy as np
import pytest

torch = pytest.importorskip('torch')
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import experiments as e
from svrplab import pilot as p
from svrplab import supervised as s
from svrplab.search import OBJECTIVE
from svrplab.supervised_data import make_geometry_folds


def manual_instances():
    instances = []
    for i in range(3):
        xy = [[0, 0], [i + 1, 0], [0, i + 1]]
        instances.append(p.freeze_instance({
            'instance_id': f'supervised_manual_{i}', 'depot_indices': [0],
            'locations': xy, 'demands': [0, 0, 1], 'capacities': [2, 2],
            'windows': [[0, 100], [0, 100], [0, 100]], 'appear_times': [0, 0, 0],
            'provenance': {'kind': 'manual_test',
                           'geometry_group_sha256': p.digest({'locations': xy})},
        }))
    return instances


def config():
    return {
        'schema_version': 'supervised-03-v1', 'role': 'development_only',
        'split_strategy': 'leave_one_geometry_out_with_inner_validation',
        'teacher': {'time_scale': 1000},
        'model': {'model_dim': 16, 'num_heads': 2, 'num_layers': 1,
                  'dropout': 0., 'tanh_clip': 10.},
        'training': {'epochs': 2, 'steps_per_epoch': 2, 'gradient_accumulation': 1,
                     'learning_rate': .001, 'weight_decay': 0., 'gradient_clip_norm': 1.,
                     'patience': 5, 'min_delta': 0., 'seconds_per_model': 120.,
                     'permute_nodes': True, 'deterministic_algorithms': True},
    }


def examples_for(instances, fold):
    def role_examples(role):
        return [{'instance_sha256': x.sha,
                 'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                 'routes': [[1, 2], []], 'label_kind': 'manual_test_double'}
                for x in instances if x.provenance['geometry_group_sha256'] in fold[role]]
    provenance = {'corpus_id': 'manual_corpus', 'heldout_labels_exposed': False,
                  'selected_teacher_costs': {'teacher_wall_seconds': 2.5},
                  'all_teacher_costs': {'teacher_wall_seconds': 3.75}}
    return (role_examples('train_geometry_groups'),
            role_examples('validation_geometry_groups'), provenance)


@pytest.fixture
def setup(monkeypatch):
    # El notebook que ejecuta pytest ya exige Colab; sólo aislamos la guardia del trainer.
    monkeypatch.setattr(s, 'require_colab', lambda: None)
    monkeypatch.setattr(p, 'environment_report', lambda: {'test_fixture': True})
    monkeypatch.setattr(s, 'save_learning_plot', lambda *args: None)
    instances = manual_instances()
    fold = make_geometry_folds(instances)[0]
    monkeypatch.setattr(s, 'load_fold_examples', lambda corpus, xs, f: examples_for(xs, f))
    return instances, fold


def train(tmp_path, instances, fold, label='run', settings=None):
    return s.train_fold(instances, p.Traffic(mode='no_noise'), settings or config(),
                        11, fold, tmp_path / 'manual_corpus',
                        tmp_path / label / 'local', tmp_path / label / 'drive', 'cpu')


@pytest.mark.parametrize('key,value', [
    ('epochs', 0), ('steps_per_epoch', True), ('gradient_accumulation', 0),
    ('patience', -1), ('learning_rate', 0), ('gradient_clip_norm', float('nan')),
    ('seconds_per_model', float('inf')), ('weight_decay', -1),
    ('min_delta', True), ('permute_nodes', 1), ('deterministic_algorithms', 'yes'),
])
def test_training_configuration_rejects_invalid_settings(key, value):
    settings = config()
    settings['training'][key] = value
    with pytest.raises(ValueError):
        s.validate_training_config(settings)


@pytest.mark.parametrize('key,value', [('schema_version', 'other'), ('role', 'final_test'),
                                     ('split_strategy', 'random_rows')])
def test_training_configuration_rejects_other_protocols(key, value):
    with pytest.raises(ValueError):
        s.validate_training_config(config() | {key: value})


def test_validation_weights_geometry_equally_and_excludes_gradient_graphs(monkeypatch):
    model = torch.nn.Linear(1, 1).train()
    examples = [
        {'geometry_group_sha256': 'a', 'instance_sha256': 'a1', 'routes': []},
        {'geometry_group_sha256': 'b', 'instance_sha256': 'b1', 'routes': []},
        {'geometry_group_sha256': 'b', 'instance_sha256': 'b2', 'routes': []},
        {'geometry_group_sha256': 'b', 'instance_sha256': 'b3', 'routes': []},
    ]
    calls = []

    def loss_stub(model, problem, routes, device):
        assert not model.training and not torch.is_grad_enabled()
        calls.append(problem)
        return torch.tensor(float(problem)), {'decisions': 1000, 'nll_sum': 0.}

    monkeypatch.setattr(s, 'teacher_loss', loss_stub)
    mean, by_group = s.validation_nll(model, {'a1': 2, 'b1': 4, 'b2': 6, 'b3': 8},
                                      examples, ['a', 'b'], 'cpu')
    assert by_group == {'a': 2., 'b': 6.}
    assert mean == 4. and mean != np.mean(calls)


def test_grouping_rejects_heldout_labels_and_missing_geometry():
    example = {'geometry_group_sha256': 'heldout'}
    with pytest.raises(ValueError, match='no autorizada'):
        s._group_examples([example], ['train'])
    with pytest.raises(ValueError, match='Faltan etiquetas'):
        s._group_examples([example], ['heldout', 'missing'])


def test_best_state_is_a_cpu_copy_and_can_restore_original_parameters(tmp_path):
    model = s.AttentionPolicy(config()['model'])
    best = s._cpu_state(model)
    original = {key: value.clone() for key, value in best.items()}
    with torch.no_grad():
        for parameter in model.parameters():
            parameter.add_(1.)
    assert all(tensor.device.type == 'cpu' for tensor in best.values())
    assert all(torch.equal(best[key], value) for key, value in original.items())
    path = tmp_path / 'best.pt'
    torch.save(best, path)
    model.load_state_dict(s._load_checkpoint(path))
    assert all(torch.equal(model.state_dict()[key], value) for key, value in original.items())


def test_train_fold_uses_only_authorized_geometries_and_final_resume_is_immutable(tmp_path, monkeypatch, setup):
    instances, fold = setup
    real_problem = s.SearchProblem
    seen = set()

    def guarded(x, *args, **kwargs):
        assert x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']
        seen.add(x.sha)
        return real_problem(x, *args, **kwargs)

    monkeypatch.setattr(s, 'SearchProblem', guarded)
    final = train(tmp_path, instances, fold)
    before = p.verify_bundle(final)
    summary = e.read_json(final / 'summary.json')
    authorized = {x.sha for x in instances
                  if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']}
    assert seen == authorized
    assert summary['heldout_used_for_training_or_selection'] is False
    assert summary['epochs_completed'] == 2 and summary['teacher_seconds'] == 2.5
    assert summary['training_seconds'] >= summary['validation_seconds'] >= 0
    assert summary['model_checkpoint_sha256'] == p.file_hash(final / 'best.pt')
    assert p.verify_bundle(tmp_path / 'run/drive/final') == before
    latest = s._load_checkpoint(final.parent / 'epoch_0002/state.pt')
    assert latest['optimizer']['state'] and latest['cuda_rng'] == []
    assert latest['torch_rng'].dtype == torch.uint8

    def forbidden(*args, **kwargs):
        raise AssertionError('Una recuperación final no debe construir problemas ni entrenar.')

    monkeypatch.setattr(s, 'SearchProblem', forbidden)
    assert train(tmp_path, instances, fold) == final
    assert p.verify_bundle(final) == before
    with (final / 'best.pt').open('ab') as stream:
        stream.write(b'corruption')
    with pytest.raises(ValueError):
        train(tmp_path, instances, fold)


def test_train_fold_rejects_heldout_label_before_constructing_problem(tmp_path, monkeypatch, setup):
    instances, fold = setup
    training, validation, provenance = examples_for(instances, fold)
    heldout = next(x for x in instances
                   if x.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups'])
    training.append({'instance_sha256': heldout.sha,
                     'geometry_group_sha256': heldout.provenance['geometry_group_sha256'],
                     'routes': [[1, 2], []]})
    monkeypatch.setattr(s, 'load_fold_examples', lambda *args: (training, validation, provenance))

    def forbidden(*args, **kwargs):
        raise AssertionError('Rechazar fuga antes de construir los problemas.')

    monkeypatch.setattr(s, 'SearchProblem', forbidden)
    with pytest.raises(ValueError, match='no autorizada'):
        train(tmp_path, instances, fold)


def test_training_budget_exhausted_by_setup_never_updates_or_exports_random_weights(tmp_path, monkeypatch, setup):
    instances, fold = setup
    settings = config()
    settings['training']['seconds_per_model'] = 1e-12

    def forbidden(*args, **kwargs):
        raise AssertionError('El presupuesto agotado no debe iniciar un batch o validación.')

    monkeypatch.setattr(s, 'teacher_loss', forbidden)
    with pytest.raises(RuntimeError, match='No hay checkpoint entrenado'):
        train(tmp_path, instances, fold, settings=settings)
    assert not list(tmp_path.glob('**/final/best.pt'))


def test_forged_group_cannot_hide_a_heldout_instance_sha(tmp_path, monkeypatch, setup):
    instances, fold = setup
    training, validation, provenance = examples_for(instances, fold)
    heldout = next(x for x in instances if x.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups'])
    training[0]['instance_sha256'] = heldout.sha
    monkeypatch.setattr(s, 'load_fold_examples', lambda *args: (training, validation, provenance))
    with pytest.raises(ValueError, match='rol de geometría'):
        train(tmp_path, instances, fold)


def test_changing_heldout_windows_cannot_change_training_inputs_or_weights(tmp_path, monkeypatch, setup):
    instances, fold = setup
    first = train(tmp_path, instances, fold, 'first')
    changed = []
    for x in instances:
        if x.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups']:
            payload = x.payload()
            payload['windows'][1][0] = 20
            changed.append(p.freeze_instance(payload))
        else:
            changed.append(x)
    first_state = s._load_checkpoint(first / 'best.pt')
    second = train(tmp_path, changed, fold, 'second')
    second_state = s._load_checkpoint(second / 'best.pt')
    assert first_state.keys() == second_state.keys()
    assert all(torch.equal(first_state[key], second_state[key]) for key in first_state)
    assert e.read_json(first / 'summary.json')['identity_sha256'] == e.read_json(second / 'summary.json')['identity_sha256']


def test_partial_resume_restores_optimizer_and_rng_without_retraining_first_epoch(tmp_path, monkeypatch, setup):
    instances, fold = setup
    reference = train(tmp_path, instances, fold, 'reference')
    real_commit = e._commit

    def interrupt_after_epoch_one(local, remote):
        real_commit(local, remote)
        if Path(local).name == 'epoch_0001':
            raise KeyboardInterrupt('Simular desconexión después del checkpoint completo.')

    monkeypatch.setattr(e, '_commit', interrupt_after_epoch_one)
    with pytest.raises(KeyboardInterrupt):
        train(tmp_path, instances, fold, 'interrupted')
    first_epoch = tmp_path / 'interrupted/local/epoch_0001'
    original_checksum = p.verify_bundle(first_epoch)
    monkeypatch.setattr(e, '_commit', real_commit)
    resumed = train(tmp_path, instances, fold, 'interrupted')
    assert p.verify_bundle(first_epoch) == original_checksum
    assert e.read_json(resumed / 'summary.json')['epochs_completed'] == 2
    expected = s._load_checkpoint(reference.parent / 'epoch_0002/state.pt')
    actual = s._load_checkpoint(resumed.parent / 'epoch_0002/state.pt')
    for name in ['model', 'best_model']:
        assert all(torch.equal(expected[name][key], actual[name][key]) for key in expected[name])
    assert torch.equal(expected['torch_rng'], actual['torch_rng'])
    for key, values in expected['optimizer']['state'].items():
        for field, value in values.items():
            if isinstance(value, torch.Tensor):
                assert torch.equal(value, actual['optimizer']['state'][key][field])
            else:
                assert value == actual['optimizer']['state'][key][field]


def test_runner_preserves_training_provenance_when_inference_fails(tmp_path, monkeypatch, setup):
    instances, fold = setup
    x = next(item for item in instances
             if item.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups'])
    bank = tmp_path / 'manual_bank'
    bank_config = {'traffic': asdict(p.Traffic(mode='no_noise')),
                   'policies': ['fixed_order', 'causal_edd'], 'alpha_cvar': .5,
                   'scenarios': {'train': 1, 'selection': 1, 'evaluation': 1}}
    e.write_json(bank / 'config.json', bank_config)
    e.write_json(bank / f'scenarios/n{x.n:04d}.json', {'train': [1], 'selection': [2], 'evaluation': [3]})
    p.seal_bundle(bank)
    monkeypatch.setattr(p, 'load_bank', lambda path: [x])
    runner_config = {'schema_version': e.SCHEMA, 'role': 'development_only',
                     'objective': OBJECTIVE,
                     'time_scale': 1000, 'seeds': [11], 'search_seconds': 120.,
                     'scenario_split': 'evaluation', 'export_all_scenario_events': True,
                     'methods': {'am_supervised': {'n_samples': 1, 'permute_samples': False}}}
    provenance = {'training_id': 'manual_training', 'fold_id': fold['fold_id'],
                  'training_seed': 11, 'training_seconds': 8., 'validation_seconds': 2.,
                  'teacher_seconds': 3., 'model_checkpoint_sha256': 'a' * 64,
                  'train_geometry_groups': fold['train_geometry_groups'],
                  'validation_geometry_groups': fold['validation_geometry_groups'],
                  'heldout_geometry_groups': fold['heldout_geometry_groups']}

    def failing_solver(*args):
        raise RuntimeError('Fallo manual de inferencia.')

    root = e.run_methods(bank, runner_config, {'am_supervised': failing_solver},
                         tmp_path / 'work', tmp_path / 'drive', run_id='failure_test',
                         method_provenance={'am_supervised': {f'{x.sha}:11': provenance}})
    rows = e.read_json(root / 'report/summary.json')
    assert len(rows) == 2
    for row in rows:
        assert row['solver_status'] == 'solver_error' and row['evaluation_status'] == 'no_solution'
        assert row['mean_travel_minutes'] is None
        for key in ['training_id', 'fold_id', 'training_seed', 'training_seconds',
                    'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256']:
            assert row[key] == provenance[key]


In [ ]:
# Pruebas aisladas en el runtime de Colab; sus fixtures no se añaden al estudio.
import uuid
def persist_validation(label, files, status):
    check = WORK / 'method_checks' / BANK_ID / (label + '_' + uuid.uuid4().hex[:12])
    check.mkdir(parents=True)
    for file in files:
        shutil.copy2(file, check / Path(file).name)
    exp.write_json(check / 'status.json', {'status': status, 'bank_id': BANK_ID,
                   'evaluator_sha256': p.file_hash(p.__file__),
                   'code_hashes': {file.name: p.file_hash(file) for file in (RUNTIME / 'svrplab').glob('*.py')}})
    p.seal_bundle(check)
    target = DRIVE_ROOT / 'method_checks' / BANK_ID / check.name
    p.sync_verified(check, target)
    print('Pruebas exportadas:', target)
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(RUNTIME)
test_env['SVRPLAB_TEST_GUROBI'] = '0'
TEST_FILES = ['tests/test_baselines.py', 'tests/test_metaheuristics.py', 'tests/test_experiments.py', 'tests/test_attention.py', 'tests/test_teacher_feasibility.py', 'tests/test_supervised_data.py', 'tests/test_supervised.py']
validation = subprocess.run([sys.executable, '-m', 'pytest', *TEST_FILES, '-q',
                             '--junitxml=validation_tests.xml'], env=test_env,
                            capture_output=True, text=True)
(RUNTIME / 'validation_tests.txt').write_text(validation.stdout + validation.stderr)
print(validation.stdout)
VALIDATION_FILES = [RUNTIME / 'validation_tests.txt', RUNTIME / 'validation_tests.xml',
                    *(RUNTIME / name for name in TEST_FILES)]
persist_validation('contract03', VALIDATION_FILES, 'passed' if validation.returncode == 0 else 'failed')
if validation.returncode:
    print(validation.stderr)
    raise RuntimeError('Pruebas fallidas: corregir antes de ejecutar métodos.')


## 5. Corpus de etiquetas heurísticas


In [ ]:
CORPUS = sd.build_teacher_corpus(instances, TRAFFIC, SUPERVISED_CONFIG['teacher'],
                                WORK, DRIVE_ROOT, BANK_ID, BANK_SHA, EVALUATOR_SHA)
corpus_index = exp.read_json(CORPUS / 'index/corpus.json')
print('Corpus:', CORPUS.name, '| estado:', corpus_index['status'],
      '| intentos:', corpus_index['attempts'], '| etiquetas distintas:', len(corpus_index['labels']))
teacher_attempts = pd.read_csv(CORPUS / 'index/teacher_attempts.csv', dtype={'teacher_seed': str})
teacher_columns = ['n_customers', 'teacher_method', 'teacher_seed', 'status', 'reason',
                   'eligible_label', 'search_violation_minutes', 'teacher_search_seconds']
display(teacher_attempts[teacher_columns])
print('No se mostrará ni seleccionará una mejor ruta heldout para entrenar.')


## 6. Entrenamiento supervisado y selección interna


In [ ]:
TRAINING = sup.train_crossfit(instances, TRAFFIC, SUPERVISED_CONFIG, CONFIG['seeds'], CORPUS,
                              WORK, DRIVE_ROOT, BANK_ID, BANK_SHA, EVALUATOR_SHA, device=str(DEVICE))
display(pd.read_csv(TRAINING / 'index/training_summary.csv'))
display(exp.read_json(TRAINING / 'index/offline_costs.json'))
for path in sorted(TRAINING.glob('geometry_*/seed_*/final/learning.png')):
    print(path.relative_to(TRAINING))
    display(Image(filename=str(path)))
print('TODOS LOS MODELOS CONGELADOS; selección realizada sin escenarios de evaluación.')


## 7. Inferencia fuera de geometría y evaluación común


In [ ]:
if p.verify_bundle(BANK) != BANK_SHA:
    raise ValueError('El banco cambió durante el entrenamiento.')
am_solver, METHOD_PROVENANCE = sup.make_solver(TRAINING, instances, TRAFFIC, str(DEVICE),
                                               expected_bank_sha256=BANK_SHA)
RUN = exp.run_methods(BANK, CONFIG, {'am_supervised': am_solver}, WORK, DRIVE_ROOT,
                      run_id=RESUME_RUN_ID or None, validation_artifacts=VALIDATION_FILES,
                      method_provenance=METHOD_PROVENANCE)


## 8. Resultados visibles y archivos en Drive


In [ ]:
summary = pd.read_csv(RUN / 'report/summary.csv', dtype={'solver_seed': str, 'training_seed': str})
columns = ['method','n_customers','solver_seed','fold_id','policy','solver_status','evaluation_status',
           'nominal_feasible','scenario_strict_feasibility','mean_travel_minutes','cvar_travel_minutes',
           'mean_lateness_minutes','mean_depot_overtime_minutes','used_vehicles',
           'training_seconds','teacher_seconds','model_load_seconds','search_total_seconds','evaluation_seconds']
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(summary[columns])
if (RUN / 'report/metrics.png').exists():
    display(Image(filename=str(RUN / 'report/metrics.png')))
for path in sorted(RUN.glob('*/n*/seed_*/solution/routes.png')):
    print(path.relative_to(RUN))
    display(Image(filename=str(path)))
display(exp.read_json(RUN / 'report/validation.json'))
print('Resultados:', DRIVE_ROOT / 'method_runs' / BANK_ID / RUN.name)
print('Etiquetas:', DRIVE_ROOT / 'teacher_corpora' / BANK_ID / CORPUS.name)
print('Entrenamiento:', DRIVE_ROOT / 'supervised_training' / BANK_ID / TRAINING.name)
print('comparison_id:', summary['comparison_id'].iloc[0])
